# Base 1 - LR

In [ ]:
# ============================================================
# SIMPLE EVALUATION — 8 models + Logistic Regression (with JuriBERT)
# + COMPARISON OF 3 INPUT CONFIGS (article/chunk prompt-format)
# + THRESHOLDS: metrics at threshold 0.50 AND at the best threshold (optimized on OOF)
#
# Configs tested:
#   1) [article_text] [SEP] [chunk]
#   2) [ARTICLE] [article_text] [SEP] [CHUNK] [chunk]
#   3) [ARTICLE] Article {pred_art}: [article_text] [SEP] [CHUNK] [chunk]
#
# Objective: general oui/non classification performance
# Metrics: Accuracy, Balanced Acc, Balanced F1, F1-oui, F1-non, MCC
# CV grouped by decision_id (5 folds)
# Reproducible seeds (best-effort) — NO strict determinism
#
# NOTE: the "best threshold" is chosen to maximize MCC on the OOF probabilities.
# ============================================================

import os, re, warnings, random
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
from tqdm import tqdm

from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import (
    accuracy_score, balanced_accuracy_score,
    f1_score, matthews_corrcoef
)

# -------------------------
# CONFIG
# -------------------------
BASE_PATH = "artifacts"  # not shipped — see DATA.md
PATH = "DATA/outputs/benchmark.csv"

OUTPUT = os.path.join(BASE_PATH, "outputs")
os.makedirs(OUTPUT, exist_ok=True)
OUTPUT_PATH = os.path.join(OUTPUT, "outputs_simple")
os.makedirs(OUTPUT_PATH, exist_ok=True)

N_SPLITS = 5
SEED = 42
MAX_LEN = 512
BATCH_SIZE = 16

# HF token (for SAUL/LLaMA if gated)
HF_TOKEN = os.environ["HF_TOKEN"]

# LLM config
LLM_MAX_LEN = 512
SAUL_MODEL = "Equall/Saul-7B-Base"
LLAMA_MODEL = "meta-llama/Llama-3.1-8B"
SAUL_BATCH_SIZE = 16
LLAMA_BATCH_SIZE = 16
USE_BF16 = True

# JuriBERT
JURIBERT_MODEL = "dascim/juribert-base"


# ============================================================
# 0) REPRO (best-effort, without strict determinism)
# ============================================================
def set_all_seeds(seed: int):
    os.environ["PYTHONHASHSEED"] = str(seed)
    random.seed(seed)
    np.random.seed(seed)
    try:
        import torch
        torch.manual_seed(seed)
        if torch.cuda.is_available():
            torch.cuda.manual_seed_all(seed)
        # Keep these "reasonable" settings without enabling strict determinism
        torch.backends.cudnn.benchmark = False
        # torch.backends.cudnn.deterministic = False  # optional (default False)
        try:
            torch.use_deterministic_algorithms(False)
        except Exception:
            pass
    except Exception:
        pass

set_all_seeds(SEED)


# ============================================================
# 1) LOADING + PREPARATION
# ============================================================
print("=" * 70)
print("LOADING DATA")
print("=" * 70)

df0 = pd.read_excel(PATH)
print(f"Available columns: {df0.columns.tolist()}")

cols_needed = ["decision_id", "chunk_id", "pred_art", "text", "article_text",
               "eval_A1", "eval_A2", "eval_A3"]
df0 = df0[[c for c in cols_needed if c in df0.columns]].copy()

def extract_oui_non(x):
    if pd.isna(x):
        return np.nan
    s = str(x).lower()
    has_oui = re.search(r"\boui\b", s) is not None
    has_non = re.search(r"\bnon\b", s) is not None
    if has_oui and not has_non:
        return "oui"
    if has_non and not has_oui:
        return "non"
    return np.nan

df0["a"] = df0["eval_A1"].apply(extract_oui_non)
df0["t"] = df0["eval_A2"].apply(extract_oui_non)
df0["s"] = df0["eval_A3"].apply(extract_oui_non)

def resolve_label(r):
    """Resolve the label: A1-A2 agreement or A3 arbitration"""
    a, t, s = r["a"], r["t"], r["s"]
    if pd.notna(a) and pd.notna(t) and a == t:
        return a
    if pd.notna(a) and pd.notna(t) and a != t and pd.notna(s):
        return s
    return np.nan

df0["label_str"] = df0.apply(resolve_label, axis=1)
df0 = df0[df0["label_str"].isin(["oui", "non"])].copy()
df0["label"] = df0["label_str"].map({"oui": 1, "non": 0}).astype(int)

for c in ["text", "article_text", "pred_art"]:
    if c not in df0.columns:
        df0[c] = ""
    df0[c] = df0[c].fillna("").astype(str)

print(f"\nTotal examples: {len(df0)}")
print(f"  - oui: {(df0['label'] == 1).sum()} ({100*(df0['label'] == 1).mean():.1f}%)")
print(f"  - non: {(df0['label'] == 0).sum()} ({100*(df0['label'] == 0).mean():.1f}%)")
print(f"Unique decisions: {df0['decision_id'].nunique()}")


# ============================================================
# 1bis) BUILD INPUTS — 3 CONFIGS
# ============================================================
CONFIGS = {
    "cfg1": "1) article_text [SEP] chunk",
    "cfg2": "2) [ARTICLE] article_text [SEP] [CHUNK] chunk",
    "cfg3": "3) [ARTICLE] Article {pred_art}: article_text [SEP] [CHUNK] chunk",
}

def build_text_inputs(df: pd.DataFrame, cfg_key: str) -> list:
    art = df["article_text"].astype(str).str.strip()
    chunk = df["text"].astype(str).str.strip()
    pred = df["pred_art"].astype(str).str.strip()

    if cfg_key == "cfg1":
        s = art + " [SEP] " + chunk
    elif cfg_key == "cfg2":
        s = "[ARTICLE] " + art + " [SEP] [CHUNK] " + chunk
    elif cfg_key == "cfg3":
        s = "[ARTICLE] Article " + pred + ": " + art + " [SEP] [CHUNK] " + chunk
    else:
        raise ValueError(f"Unknown cfg_key={cfg_key}")
    return s.tolist()


# ============================================================
# 2) GROUPED CROSS-VALIDATION
# ============================================================
def make_grouped_folds(df, n_splits, seed):
    rng = np.random.default_rng(seed)
    group_sizes = df.groupby("decision_id").size().to_dict()
    uniq_groups = np.array(list(group_sizes.keys()))
    uniq_groups = uniq_groups[rng.permutation(len(uniq_groups))]
    uniq_groups = sorted(uniq_groups, key=lambda g: group_sizes[g], reverse=True)

    fold_loads = np.zeros(n_splits, dtype=int)
    group_to_fold = {}
    for g in uniq_groups:
        f = int(fold_loads.argmin())
        group_to_fold[g] = f
        fold_loads[f] += int(group_sizes[g])

    out = df.copy()
    out["fold"] = out["decision_id"].map(group_to_fold).astype(int)
    return out, fold_loads


# ============================================================
# 3) METRICS + THRESHOLD SELECTION
# ============================================================
def compute_metrics(y_true, y_pred):
    acc = accuracy_score(y_true, y_pred)
    bacc = balanced_accuracy_score(y_true, y_pred)
    f1_oui = f1_score(y_true, y_pred, pos_label=1, zero_division=0)
    f1_non = f1_score(y_true, y_pred, pos_label=0, zero_division=0)
    balanced_f1 = 0.5 * (f1_oui + f1_non)
    mcc = matthews_corrcoef(y_true, y_pred) if len(np.unique(y_true)) > 1 else np.nan
    return {
        "Accuracy": acc,
        "Balanced Acc": bacc,
        "Balanced F1": balanced_f1,
        "F1-oui": f1_oui,
        "F1-non": f1_non,
        "MCC": mcc
    }

def metrics_at_threshold(y_true, p_oui, thr: float):
    y_pred = (p_oui >= thr).astype(int)
    return compute_metrics(y_true, y_pred)

def find_best_threshold(y_true, p_oui, metric="MCC", grid=None):
    if grid is None:
        grid = np.round(np.arange(0.05, 0.951, 0.01), 2)

    best_thr = 0.50
    best = None
    best_val = -1e18

    for thr in grid:
        m = metrics_at_threshold(y_true, p_oui, float(thr))
        val = m.get(metric, np.nan)
        if np.isnan(val):
            continue
        if (val > best_val) or (val == best_val and abs(thr - 0.50) < abs(best_thr - 0.50)):
            best_val = val
            best_thr = float(thr)
            best = m

    if best is None:
        best = metrics_at_threshold(y_true, p_oui, 0.50)

    return best_thr, best


# ============================================================
# 4) EVALUATION (OOF on grouped folds) — returns OOF probabilities
# ============================================================
def evaluate_tfidf_oof_proba(texts, df, seed):
    y_true_oof = np.zeros(len(df), dtype=int)
    p_oui_oof = np.zeros(len(df), dtype=float)

    for fold_id in range(N_SPLITS):
        train_idx = (df["fold"] != fold_id).values
        test_idx = (df["fold"] == fold_id).values

        y_train = df.loc[train_idx, "label"].values
        y_test = df.loc[test_idx, "label"].values
        y_true_oof[test_idx] = y_test

        train_ids = np.where(train_idx)[0]
        test_ids = np.where(test_idx)[0]

        vec = TfidfVectorizer(
            ngram_range=(1, 2),
            min_df=2,
            max_features=50000,
            sublinear_tf=True
        )
        X_train = vec.fit_transform([texts[i] for i in train_ids])
        X_test = vec.transform([texts[i] for i in test_ids])

        clf = LogisticRegression(
            solver="saga",
            max_iter=5000,
            C=1.0,
            random_state=seed,
            n_jobs=-1
        )
        clf.fit(X_train, y_train)

        proba = clf.predict_proba(X_test)[:, 1]
        p_oui_oof[test_idx] = proba

    return y_true_oof, p_oui_oof


def evaluate_embeddings_oof_proba(X_dense, df, seed):
    y_true_oof = np.zeros(len(df), dtype=int)
    p_oui_oof = np.zeros(len(df), dtype=float)

    scaler = StandardScaler()

    for fold_id in range(N_SPLITS):
        train_idx = (df["fold"] != fold_id).values
        test_idx = (df["fold"] == fold_id).values

        X_train = scaler.fit_transform(X_dense[train_idx])
        X_test = scaler.transform(X_dense[test_idx])

        y_train = df.loc[train_idx, "label"].values
        y_test = df.loc[test_idx, "label"].values
        y_true_oof[test_idx] = y_test

        clf = LogisticRegression(
            solver="lbfgs",
            max_iter=2000,
            C=1.0,
            random_state=seed
        )
        clf.fit(X_train, y_train)
        proba = clf.predict_proba(X_test)[:, 1]
        p_oui_oof[test_idx] = proba

    return y_true_oof, p_oui_oof


# ============================================================
# 5) ENCODERS
# ============================================================
import torch
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"\nDevice: {device}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

def encode_sentence_transformer(texts, model_name, batch_size=BATCH_SIZE, max_seq_length=256):
    from sentence_transformers import SentenceTransformer
    texts = ["" if t is None else str(t) for t in texts]
    st_device = "cuda" if torch.cuda.is_available() else "cpu"

    model = SentenceTransformer(model_name, device=st_device)
    model.max_seq_length = int(max_seq_length)
    embs = model.encode(
        texts,
        show_progress_bar=True,
        batch_size=int(batch_size),
        convert_to_numpy=True
    )
    del model
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    return embs

def encode_transformer_mean(texts, model_name, batch_size=BATCH_SIZE, max_len=MAX_LEN):
    from transformers import AutoTokenizer, AutoModel
    texts = ["" if t is None else str(t) for t in texts]

    tokenizer = AutoTokenizer.from_pretrained(model_name)
    model = AutoModel.from_pretrained(model_name).to(device)
    model.eval()

    all_embs = []
    with torch.no_grad():
        for i in tqdm(range(0, len(texts), batch_size), desc=model_name.split("/")[-1]):
            batch = texts[i:i + batch_size]
            inputs = tokenizer(
                batch, padding=True, truncation=True,
                max_length=int(max_len), return_tensors="pt"
            ).to(device)

            out = model(**inputs)
            mask = inputs["attention_mask"].unsqueeze(-1).float()
            emb = (out.last_hidden_state * mask).sum(1) / mask.sum(1).clamp_min(1.0)
            all_embs.append(emb.detach().cpu().numpy())

    del model, tokenizer
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    return np.vstack(all_embs)

def encode_llm_mean_pool(texts, model_name, batch_size, max_len, hf_token=None, use_bf16=True, desc="LLM"):
    from transformers import AutoTokenizer, AutoModel

    texts = ["" if t is None else str(t) for t in texts]
    tok = AutoTokenizer.from_pretrained(model_name, token=hf_token, use_fast=True)

    if tok.pad_token is None:
        tok.pad_token = tok.eos_token
    tok.padding_side = "right"

    model = AutoModel.from_pretrained(
        model_name,
        token=hf_token,
        torch_dtype=(torch.bfloat16 if (use_bf16 and torch.cuda.is_available()) else None),
        low_cpu_mem_usage=True,
    ).to(device)
    model.eval()

    if getattr(model.config, "pad_token_id", None) is None or model.config.pad_token_id < 0:
        model.config.pad_token_id = tok.pad_token_id

    all_embs = []
    with torch.no_grad():
        for i in tqdm(range(0, len(texts), batch_size), desc=desc):
            batch = texts[i:i + batch_size]
            enc = tok(
                batch, padding=True, truncation=True,
                max_length=int(max_len), return_tensors="pt"
            ).to(device)
            out = model(**enc)
            last = out.last_hidden_state
            mask = enc["attention_mask"].unsqueeze(-1).to(last.dtype)
            emb = (last * mask).sum(1) / mask.sum(1).clamp_min(1.0)
            all_embs.append(emb.float().cpu().numpy())

    del model, tok
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    return np.vstack(all_embs)


# ============================================================
# 6) MAIN — EXECUTION (for each config)
# ============================================================
if __name__ == "__main__":

    df_cv, fold_loads = make_grouped_folds(df0, N_SPLITS, seed=SEED)
    print(f"\nFold sizes: {fold_loads.tolist()}")

    all_rows = []
    THR_OPT_METRIC = "MCC"  # <-- change to optimize on "Balanced F1", etc.

    for cfg_key, cfg_name in CONFIGS.items():
        print("\n" + "=" * 90)
        print(f"CONFIG INPUT: {cfg_name}")
        print("=" * 90)

        texts = build_text_inputs(df_cv, cfg_key)

        def add_two_threshold_rows(model_name, y_true_oof, p_oui_oof):
            # (A) threshold 0.50
            m_05 = metrics_at_threshold(y_true_oof, p_oui_oof, 0.50)
            row_05 = {"Config": cfg_key, "Model": model_name, "ThresholdType": "fixed_0.50", "Threshold": 0.50}
            row_05.update(m_05)
            all_rows.append(row_05)

            # (B) best threshold (OOF optimization)
            best_thr, m_best = find_best_threshold(y_true_oof, p_oui_oof, metric=THR_OPT_METRIC)
            row_best = {"Config": cfg_key, "Model": model_name, "ThresholdType": f"best_{THR_OPT_METRIC}", "Threshold": best_thr}
            row_best.update(m_best)
            all_rows.append(row_best)

            print(f"  -> {model_name:12s} | thr=0.50 MCC={m_05['MCC']:.4f} | best thr={best_thr:.2f} MCC={m_best['MCC']:.4f}")

        # ----- TF-IDF -----
        print("\n" + "=" * 50)
        print("TF-IDF + LR (probas OOF)")
        print("=" * 50)
        y_true_oof, p_oui_oof = evaluate_tfidf_oof_proba(texts, df_cv, seed=SEED)
        add_two_threshold_rows("TF-IDF", y_true_oof, p_oui_oof)

        # ----- ST-MiniLM -----
        print("\n" + "=" * 50)
        print("ST-MiniLM-multilingual + LR")
        print("=" * 50)
        X_minilm = encode_sentence_transformer(texts, "paraphrase-multilingual-MiniLM-L12-v2")
        y_true_oof, p_oui_oof = evaluate_embeddings_oof_proba(X_minilm, df_cv, seed=SEED)
        add_two_threshold_rows("ST-MiniLM", y_true_oof, p_oui_oof)

        # ----- ST-MPNet -----
        print("\n" + "=" * 50)
        print("ST-MPNet-multilingual + LR")
        print("=" * 50)
        X_mpnet = encode_sentence_transformer(texts, "paraphrase-multilingual-mpnet-base-v2")
        y_true_oof, p_oui_oof = evaluate_embeddings_oof_proba(X_mpnet, df_cv, seed=SEED)
        add_two_threshold_rows("ST-MPNet", y_true_oof, p_oui_oof)

        # ----- CamemBERT -----
        print("\n" + "=" * 50)
        print("CamemBERT + LR")
        print("=" * 50)
        X_camem = encode_transformer_mean(texts, "camembert-base")
        y_true_oof, p_oui_oof = evaluate_embeddings_oof_proba(X_camem, df_cv, seed=SEED)
        add_two_threshold_rows("CamemBERT", y_true_oof, p_oui_oof)

        # ----- CamemBERTav2 -----
        print("\n" + "=" * 50)
        print("CamemBERTav2 + LR")
        print("=" * 50)
        X_camv2 = encode_transformer_mean(texts, "almanach/camembertav2-base")
        y_true_oof, p_oui_oof = evaluate_embeddings_oof_proba(X_camv2, df_cv, seed=SEED)
        add_two_threshold_rows("CamemBERTav2", y_true_oof, p_oui_oof)

        # ----- JuriBERT-base -----
        print("\n" + "=" * 50)
        print("JuriBERT-base + LR")
        print("=" * 50)
        X_juribert = encode_transformer_mean(texts, JURIBERT_MODEL)
        y_true_oof, p_oui_oof = evaluate_embeddings_oof_proba(X_juribert, df_cv, seed=SEED)
        add_two_threshold_rows("JuriBERT-base", y_true_oof, p_oui_oof)

        # ----- SAUL-7B -----
        print("\n" + "=" * 50)
        print("SAUL-7B-Base + LR")
        print("=" * 50)
        X_saul = encode_llm_mean_pool(texts, SAUL_MODEL, SAUL_BATCH_SIZE, LLM_MAX_LEN, HF_TOKEN, USE_BF16, "SAUL")
        y_true_oof, p_oui_oof = evaluate_embeddings_oof_proba(X_saul, df_cv, seed=SEED)
        add_two_threshold_rows("SAUL-7B", y_true_oof, p_oui_oof)

        # ----- LLaMA-3.1-8B -----
        print("\n" + "=" * 50)
        print("LLaMA-3.1-8B + LR")
        print("=" * 50)
        X_llama = encode_llm_mean_pool(texts, LLAMA_MODEL, LLAMA_BATCH_SIZE, LLM_MAX_LEN, HF_TOKEN, USE_BF16, "LLaMA")
        y_true_oof, p_oui_oof = evaluate_embeddings_oof_proba(X_llama, df_cv, seed=SEED)
        add_two_threshold_rows("LLaMA-3.1-8B", y_true_oof, p_oui_oof)

    # ============================================================
    # 7) SUMMARY TABLE
    # ============================================================
    print("\n" + "=" * 100)
    print(f"RESULTS — 0.50 vs best threshold (optimized on {THR_OPT_METRIC}) — grouped 5-fold CV")
    print("=" * 100)

    df_results = pd.DataFrame(all_rows)
    df_results = df_results[[
        "Config", "Model", "ThresholdType", "Threshold",
        "Accuracy", "Balanced Acc", "Balanced F1", "F1-oui", "F1-non", "MCC"
    ]].copy()

    # Sort: config, model, then (best first) + descending MCC
    df_results["is_best"] = (df_results["ThresholdType"].str.startswith("best_")).astype(int)
    df_results = df_results.sort_values(
        ["Config", "Model", "is_best", "MCC"],
        ascending=[True, True, False, False]
    ).drop(columns=["is_best"]).reset_index(drop=True)

    print(df_results.to_string(index=False, float_format="%.4f"))

    output_file = os.path.join(OUTPUT_PATH, "results_8_models_with_juribert__3_configs__thresholds.xlsx")
    df_results.to_excel(output_file, index=False)
    print(f"\n✅ Results saved: {output_file}")


# Base 2 - MLP 1 layer

In [ ]:
# ============================================================
# SIMPLE EVALUATION — 8 models + MLP (small) (with JuriBERT)
# + COMPARISON OF 3 INPUT CONFIGS (article/chunk prompt-format)
# + THRESHOLDS: metrics at threshold 0.50 AND at the best threshold (optimized on OOF)
#
# Configs tested:
#   1) [article_text] [SEP] [chunk]
#   2) [ARTICLE] [article_text] [SEP] [CHUNK] [chunk]
#   3) [ARTICLE] Article {pred_art}: [article_text] [SEP] [CHUNK] [chunk]
#
# Objective: general oui/non classification performance
# Metrics: Accuracy, Balanced Acc, Balanced F1, F1-oui, F1-non, MCC
# CV grouped by decision_id (5 folds)
# Reproducible seeds (best-effort) — NO strict determinism
#
# NOTE: the "best threshold" is chosen to maximize MCC on the OOF probabilities.
# NOTE: For TF-IDF we use an MLP on dense features (via TruncatedSVD).
# ============================================================

import os, re, warnings, random
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
from tqdm import tqdm

from sklearn.preprocessing import StandardScaler
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.decomposition import TruncatedSVD
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import (
    accuracy_score, balanced_accuracy_score,
    f1_score, matthews_corrcoef
)

# -------------------------
# CONFIG
# -------------------------
BASE_PATH = "artifacts"  # not shipped — see DATA.md
PATH = "DATA/outputs/benchmark.csv"

OUTPUT = os.path.join(BASE_PATH, "outputs")
os.makedirs(OUTPUT, exist_ok=True)
OUTPUT_PATH = os.path.join(OUTPUT, "outputs_simple")
os.makedirs(OUTPUT_PATH, exist_ok=True)

N_SPLITS = 5
SEED = 42
MAX_LEN = 512
BATCH_SIZE = 16

# HF token (for SAUL/LLaMA if gated)
HF_TOKEN = os.environ["HF_TOKEN"]

# LLM config
LLM_MAX_LEN = 512
SAUL_MODEL = "Equall/Saul-7B-Base"
LLAMA_MODEL = "meta-llama/Llama-3.1-8B"
SAUL_BATCH_SIZE = 16
LLAMA_BATCH_SIZE = 16
USE_BF16 = True

# JuriBERT
JURIBERT_MODEL = "dascim/juribert-base"

# ============================================================
# MLP (small & stable) — recommended for ~1k examples
# ============================================================
MLP_HIDDEN = (256,)      # 1 hidden layer
MLP_ALPHA = 1e-3         # stronger L2 regularization
MLP_LR_INIT = 5e-4       # lower LR
MLP_MAX_ITER = 300
MLP_EARLY_STOPPING = True
MLP_N_ITER_NO_CHANGE = 15

# TF-IDF -> SVD dim to give the MLP a stable dense input
TFIDF_MAX_FEATS = 50000
TFIDF_SVD_DIM = 512


# ============================================================
# 0) REPRO (best-effort, without strict determinism)
# ============================================================
def set_all_seeds(seed: int):
    os.environ["PYTHONHASHSEED"] = str(seed)
    random.seed(seed)
    np.random.seed(seed)
    try:
        import torch
        torch.manual_seed(seed)
        if torch.cuda.is_available():
            torch.cuda.manual_seed_all(seed)
        torch.backends.cudnn.benchmark = False
        try:
            torch.use_deterministic_algorithms(False)
        except Exception:
            pass
    except Exception:
        pass

set_all_seeds(SEED)


# ============================================================
# 1) LOADING + PREPARATION
# ============================================================
print("=" * 70)
print("LOADING DATA")
print("=" * 70)

df0 = pd.read_excel(PATH)
print(f"Available columns: {df0.columns.tolist()}")

cols_needed = ["decision_id", "chunk_id", "pred_art", "text", "article_text",
               "eval_A1", "eval_A2", "eval_A3"]
df0 = df0[[c for c in cols_needed if c in df0.columns]].copy()

def extract_oui_non(x):
    if pd.isna(x):
        return np.nan
    s = str(x).lower()
    has_oui = re.search(r"\boui\b", s) is not None
    has_non = re.search(r"\bnon\b", s) is not None
    if has_oui and not has_non:
        return "oui"
    if has_non and not has_oui:
        return "non"
    return np.nan

df0["a"] = df0["eval_A1"].apply(extract_oui_non)
df0["t"] = df0["eval_A2"].apply(extract_oui_non)
df0["s"] = df0["eval_A3"].apply(extract_oui_non)

def resolve_label(r):
    a, t, s = r["a"], r["t"], r["s"]
    if pd.notna(a) and pd.notna(t) and a == t:
        return a
    if pd.notna(a) and pd.notna(t) and a != t and pd.notna(s):
        return s
    return np.nan

df0["label_str"] = df0.apply(resolve_label, axis=1)
df0 = df0[df0["label_str"].isin(["oui", "non"])].copy()
df0["label"] = df0["label_str"].map({"oui": 1, "non": 0}).astype(int)

for c in ["text", "article_text", "pred_art"]:
    if c not in df0.columns:
        df0[c] = ""
    df0[c] = df0[c].fillna("").astype(str)

print(f"\nTotal examples: {len(df0)}")
print(f"  - oui: {(df0['label'] == 1).sum()} ({100*(df0['label'] == 1).mean():.1f}%)")
print(f"  - non: {(df0['label'] == 0).sum()} ({100*(df0['label'] == 0).mean():.1f}%)")
print(f"Unique decisions: {df0['decision_id'].nunique()}")


# ============================================================
# 1bis) BUILD INPUTS — 3 CONFIGS
# ============================================================
CONFIGS = {
    "cfg1": "1) article_text [SEP] chunk",
    "cfg2": "2) [ARTICLE] article_text [SEP] [CHUNK] chunk",
    "cfg3": "3) [ARTICLE] Article {pred_art}: article_text [SEP] [CHUNK] chunk",
}

def build_text_inputs(df: pd.DataFrame, cfg_key: str) -> list:
    art = df["article_text"].astype(str).str.strip()
    chunk = df["text"].astype(str).str.strip()
    pred = df["pred_art"].astype(str).str.strip()

    if cfg_key == "cfg1":
        s = art + " [SEP] " + chunk
    elif cfg_key == "cfg2":
        s = "[ARTICLE] " + art + " [SEP] [CHUNK] " + chunk
    elif cfg_key == "cfg3":
        s = "[ARTICLE] Article " + pred + ": " + art + " [SEP] [CHUNK] " + chunk
    else:
        raise ValueError(f"Unknown cfg_key={cfg_key}")
    return s.tolist()


# ============================================================
# 2) GROUPED CROSS-VALIDATION
# ============================================================
def make_grouped_folds(df, n_splits, seed):
    rng = np.random.default_rng(seed)
    group_sizes = df.groupby("decision_id").size().to_dict()
    uniq_groups = np.array(list(group_sizes.keys()))
    uniq_groups = uniq_groups[rng.permutation(len(uniq_groups))]
    uniq_groups = sorted(uniq_groups, key=lambda g: group_sizes[g], reverse=True)

    fold_loads = np.zeros(n_splits, dtype=int)
    group_to_fold = {}
    for g in uniq_groups:
        f = int(fold_loads.argmin())
        group_to_fold[g] = f
        fold_loads[f] += int(group_sizes[g])

    out = df.copy()
    out["fold"] = out["decision_id"].map(group_to_fold).astype(int)
    return out, fold_loads


# ============================================================
# 3) METRICS + THRESHOLD SELECTION
# ============================================================
def compute_metrics(y_true, y_pred):
    acc = accuracy_score(y_true, y_pred)
    bacc = balanced_accuracy_score(y_true, y_pred)
    f1_oui = f1_score(y_true, y_pred, pos_label=1, zero_division=0)
    f1_non = f1_score(y_true, y_pred, pos_label=0, zero_division=0)
    balanced_f1 = 0.5 * (f1_oui + f1_non)
    mcc = matthews_corrcoef(y_true, y_pred) if len(np.unique(y_true)) > 1 else np.nan
    return {
        "Accuracy": acc,
        "Balanced Acc": bacc,
        "Balanced F1": balanced_f1,
        "F1-oui": f1_oui,
        "F1-non": f1_non,
        "MCC": mcc
    }

def metrics_at_threshold(y_true, p_oui, thr: float):
    y_pred = (p_oui >= thr).astype(int)
    return compute_metrics(y_true, y_pred)

def find_best_threshold(y_true, p_oui, metric="MCC", grid=None):
    if grid is None:
        grid = np.round(np.arange(0.05, 0.951, 0.01), 2)

    best_thr = 0.50
    best = None
    best_val = -1e18

    for thr in grid:
        m = metrics_at_threshold(y_true, p_oui, float(thr))
        val = m.get(metric, np.nan)
        if np.isnan(val):
            continue
        if (val > best_val) or (val == best_val and abs(thr - 0.50) < abs(best_thr - 0.50)):
            best_val = val
            best_thr = float(thr)
            best = m

    if best is None:
        best = metrics_at_threshold(y_true, p_oui, 0.50)

    return best_thr, best


# ============================================================
# 4) EVALUATION (OOF on grouped folds) — returns OOF probabilities (MLP)
# ============================================================
def make_mlp(seed: int):
    return MLPClassifier(
        hidden_layer_sizes=MLP_HIDDEN,
        activation="relu",
        solver="adam",
        alpha=float(MLP_ALPHA),
        learning_rate_init=float(MLP_LR_INIT),
        max_iter=int(MLP_MAX_ITER),
        early_stopping=bool(MLP_EARLY_STOPPING),
        n_iter_no_change=int(MLP_N_ITER_NO_CHANGE),
        random_state=int(seed),
        verbose=False
    )

def evaluate_tfidf_oof_proba(texts, df, seed):
    """
    TF-IDF -> TruncatedSVD -> StandardScaler -> MLP
    (SVD/scaler fit on train fold only => no leakage)
    """
    y_true_oof = np.zeros(len(df), dtype=int)
    p_oui_oof = np.zeros(len(df), dtype=float)

    for fold_id in range(N_SPLITS):
        train_idx = (df["fold"] != fold_id).values
        test_idx = (df["fold"] == fold_id).values

        y_train = df.loc[train_idx, "label"].values
        y_test = df.loc[test_idx, "label"].values
        y_true_oof[test_idx] = y_test

        train_ids = np.where(train_idx)[0]
        test_ids = np.where(test_idx)[0]

        vec = TfidfVectorizer(
            ngram_range=(1, 2),
            min_df=2,
            max_features=int(TFIDF_MAX_FEATS),
            sublinear_tf=True
        )
        X_train = vec.fit_transform([texts[i] for i in train_ids])
        X_test = vec.transform([texts[i] for i in test_ids])

        # safeguard if vocabulary is tiny
        if X_train.shape[1] <= 1:
            Z_train = X_train.toarray()
            Z_test = X_test.toarray()
        else:
            svd_dim = min(int(TFIDF_SVD_DIM), X_train.shape[1] - 1)
            svd = TruncatedSVD(n_components=int(max(1, svd_dim)), random_state=int(seed))
            Z_train = svd.fit_transform(X_train)
            Z_test = svd.transform(X_test)

        scaler = StandardScaler()
        Z_train = scaler.fit_transform(Z_train)
        Z_test = scaler.transform(Z_test)

        clf = make_mlp(seed)
        clf.fit(Z_train, y_train)

        proba = clf.predict_proba(Z_test)[:, 1]
        p_oui_oof[test_idx] = proba

    return y_true_oof, p_oui_oof


def evaluate_embeddings_oof_proba(X_dense, df, seed):
    """
    Embeddings -> StandardScaler -> MLP
    """
    y_true_oof = np.zeros(len(df), dtype=int)
    p_oui_oof = np.zeros(len(df), dtype=float)

    for fold_id in range(N_SPLITS):
        train_idx = (df["fold"] != fold_id).values
        test_idx = (df["fold"] == fold_id).values

        y_train = df.loc[train_idx, "label"].values
        y_test = df.loc[test_idx, "label"].values
        y_true_oof[test_idx] = y_test

        scaler = StandardScaler()
        X_train = scaler.fit_transform(X_dense[train_idx])
        X_test = scaler.transform(X_dense[test_idx])

        clf = make_mlp(seed)
        clf.fit(X_train, y_train)
        proba = clf.predict_proba(X_test)[:, 1]
        p_oui_oof[test_idx] = proba

    return y_true_oof, p_oui_oof


# ============================================================
# 5) ENCODERS
# ============================================================
import torch
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"\nDevice: {device}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

def encode_sentence_transformer(texts, model_name, batch_size=BATCH_SIZE, max_seq_length=256):
    from sentence_transformers import SentenceTransformer
    texts = ["" if t is None else str(t) for t in texts]
    st_device = "cuda" if torch.cuda.is_available() else "cpu"

    model = SentenceTransformer(model_name, device=st_device)
    model.max_seq_length = int(max_seq_length)
    embs = model.encode(
        texts,
        show_progress_bar=True,
        batch_size=int(batch_size),
        convert_to_numpy=True
    )
    del model
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    return embs

def encode_transformer_mean(texts, model_name, batch_size=BATCH_SIZE, max_len=MAX_LEN):
    from transformers import AutoTokenizer, AutoModel
    texts = ["" if t is None else str(t) for t in texts]

    tokenizer = AutoTokenizer.from_pretrained(model_name)
    model = AutoModel.from_pretrained(model_name).to(device)
    model.eval()

    all_embs = []
    with torch.no_grad():
        for i in tqdm(range(0, len(texts), batch_size), desc=model_name.split("/")[-1]):
            batch = texts[i:i + batch_size]
            inputs = tokenizer(
                batch, padding=True, truncation=True,
                max_length=int(max_len), return_tensors="pt"
            ).to(device)

            out = model(**inputs)
            mask = inputs["attention_mask"].unsqueeze(-1).float()
            emb = (out.last_hidden_state * mask).sum(1) / mask.sum(1).clamp_min(1.0)
            all_embs.append(emb.detach().cpu().numpy())

    del model, tokenizer
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    return np.vstack(all_embs)

def encode_llm_mean_pool(texts, model_name, batch_size, max_len, hf_token=None, use_bf16=True, desc="LLM"):
    from transformers import AutoTokenizer, AutoModel

    texts = ["" if t is None else str(t) for t in texts]
    tok = AutoTokenizer.from_pretrained(model_name, token=hf_token, use_fast=True)

    if tok.pad_token is None:
        tok.pad_token = tok.eos_token
    tok.padding_side = "right"

    model = AutoModel.from_pretrained(
        model_name,
        token=hf_token,
        torch_dtype=(torch.bfloat16 if (use_bf16 and torch.cuda.is_available()) else None),
        low_cpu_mem_usage=True,
    ).to(device)
    model.eval()

    if getattr(model.config, "pad_token_id", None) is None or model.config.pad_token_id < 0:
        model.config.pad_token_id = tok.pad_token_id

    all_embs = []
    with torch.no_grad():
        for i in tqdm(range(0, len(texts), batch_size), desc=desc):
            batch = texts[i:i + batch_size]
            enc = tok(
                batch, padding=True, truncation=True,
                max_length=int(max_len), return_tensors="pt"
            ).to(device)
            out = model(**enc)
            last = out.last_hidden_state
            mask = enc["attention_mask"].unsqueeze(-1).to(last.dtype)
            emb = (last * mask).sum(1) / mask.sum(1).clamp_min(1.0)
            all_embs.append(emb.float().cpu().numpy())

    del model, tok
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    return np.vstack(all_embs)


# ============================================================
# 6) MAIN — EXECUTION (for each config)
# ============================================================
if __name__ == "__main__":

    df_cv, fold_loads = make_grouped_folds(df0, N_SPLITS, seed=SEED)
    print(f"\nFold sizes: {fold_loads.tolist()}")

    all_rows = []
    THR_OPT_METRIC = "MCC"  # <-- change to optimize on "Balanced F1", etc.

    for cfg_key, cfg_name in CONFIGS.items():
        print("\n" + "=" * 90)
        print(f"CONFIG INPUT: {cfg_name}")
        print("=" * 90)

        texts = build_text_inputs(df_cv, cfg_key)

        def add_two_threshold_rows(model_name, y_true_oof, p_oui_oof):
            # (A) threshold 0.50
            m_05 = metrics_at_threshold(y_true_oof, p_oui_oof, 0.50)
            row_05 = {"Config": cfg_key, "Model": model_name, "ThresholdType": "fixed_0.50", "Threshold": 0.50}
            row_05.update(m_05)
            all_rows.append(row_05)

            # (B) best threshold (OOF optimization)
            best_thr, m_best = find_best_threshold(y_true_oof, p_oui_oof, metric=THR_OPT_METRIC)
            row_best = {"Config": cfg_key, "Model": model_name, "ThresholdType": f"best_{THR_OPT_METRIC}", "Threshold": best_thr}
            row_best.update(m_best)
            all_rows.append(row_best)

            print(f"  -> {model_name:12s} | thr=0.50 MCC={m_05['MCC']:.4f} | best thr={best_thr:.2f} MCC={m_best['MCC']:.4f}")

        # ----- TF-IDF -----
        print("\n" + "=" * 50)
        print("TF-IDF + MLP (probas OOF) — via SVD")
        print("=" * 50)
        y_true_oof, p_oui_oof = evaluate_tfidf_oof_proba(texts, df_cv, seed=SEED)
        add_two_threshold_rows("TF-IDF", y_true_oof, p_oui_oof)

        # ----- ST-MiniLM -----
        print("\n" + "=" * 50)
        print("ST-MiniLM-multilingual + MLP")
        print("=" * 50)
        X_minilm = encode_sentence_transformer(texts, "paraphrase-multilingual-MiniLM-L12-v2")
        y_true_oof, p_oui_oof = evaluate_embeddings_oof_proba(X_minilm, df_cv, seed=SEED)
        add_two_threshold_rows("ST-MiniLM", y_true_oof, p_oui_oof)

        # ----- ST-MPNet -----
        print("\n" + "=" * 50)
        print("ST-MPNet-multilingual + MLP")
        print("=" * 50)
        X_mpnet = encode_sentence_transformer(texts, "paraphrase-multilingual-mpnet-base-v2")
        y_true_oof, p_oui_oof = evaluate_embeddings_oof_proba(X_mpnet, df_cv, seed=SEED)
        add_two_threshold_rows("ST-MPNet", y_true_oof, p_oui_oof)

        # ----- CamemBERT -----
        print("\n" + "=" * 50)
        print("CamemBERT + MLP")
        print("=" * 50)
        X_camem = encode_transformer_mean(texts, "camembert-base")
        y_true_oof, p_oui_oof = evaluate_embeddings_oof_proba(X_camem, df_cv, seed=SEED)
        add_two_threshold_rows("CamemBERT", y_true_oof, p_oui_oof)

        # ----- CamemBERTav2 -----
        print("\n" + "=" * 50)
        print("CamemBERTav2 + MLP")
        print("=" * 50)
        X_camv2 = encode_transformer_mean(texts, "almanach/camembertav2-base")
        y_true_oof, p_oui_oof = evaluate_embeddings_oof_proba(X_camv2, df_cv, seed=SEED)
        add_two_threshold_rows("CamemBERTav2", y_true_oof, p_oui_oof)

        # ----- JuriBERT-base -----
        print("\n" + "=" * 50)
        print("JuriBERT-base + MLP")
        print("=" * 50)
        X_juribert = encode_transformer_mean(texts, JURIBERT_MODEL)
        y_true_oof, p_oui_oof = evaluate_embeddings_oof_proba(X_juribert, df_cv, seed=SEED)
        add_two_threshold_rows("JuriBERT-base", y_true_oof, p_oui_oof)

        # ----- SAUL-7B -----
        print("\n" + "=" * 50)
        print("SAUL-7B-Base + MLP")
        print("=" * 50)
        X_saul = encode_llm_mean_pool(texts, SAUL_MODEL, SAUL_BATCH_SIZE, LLM_MAX_LEN, HF_TOKEN, USE_BF16, "SAUL")
        y_true_oof, p_oui_oof = evaluate_embeddings_oof_proba(X_saul, df_cv, seed=SEED)
        add_two_threshold_rows("SAUL-7B", y_true_oof, p_oui_oof)

        # ----- LLaMA-3.1-8B -----
        print("\n" + "=" * 50)
        print("LLaMA-3.1-8B + MLP")
        print("=" * 50)
        X_llama = encode_llm_mean_pool(texts, LLAMA_MODEL, LLAMA_BATCH_SIZE, LLM_MAX_LEN, HF_TOKEN, USE_BF16, "LLaMA")
        y_true_oof, p_oui_oof = evaluate_embeddings_oof_proba(X_llama, df_cv, seed=SEED)
        add_two_threshold_rows("LLaMA-3.1-8B", y_true_oof, p_oui_oof)

    # ============================================================
    # 7) SUMMARY TABLE
    # ============================================================
    print("\n" + "=" * 100)
    print(f"RESULTS — 0.50 vs best threshold (optimized on {THR_OPT_METRIC}) — grouped 5-fold CV")
    print("=" * 100)

    df_results = pd.DataFrame(all_rows)
    df_results = df_results[[
        "Config", "Model", "ThresholdType", "Threshold",
        "Accuracy", "Balanced Acc", "Balanced F1", "F1-oui", "F1-non", "MCC"
    ]].copy()

    # Sort: config, model, then (best first) + descending MCC
    df_results["is_best"] = (df_results["ThresholdType"].str.startswith("best_")).astype(int)
    df_results = df_results.sort_values(
        ["Config", "Model", "is_best", "MCC"],
        ascending=[True, True, False, False]
    ).drop(columns=["is_best"]).reset_index(drop=True)

    print(df_results.to_string(index=False, float_format="%.4f"))

    output_file = os.path.join(
        OUTPUT_PATH,
        "results_8_models_with_juribert__3_configs__thresholds__MLP_small.xlsx"
    )
    df_results.to_excel(output_file, index=False)
    print(f"\n✅ Results saved: {output_file}")


In [ ]:
# ============================================================
# 7bis) KEEP BEST CONFIG PER MODEL (by MCC on best_MCC), keep fixed_0.50 + best_MCC
# + format 2 decimals
# ============================================================

df_results = pd.DataFrame(all_rows)

# Columns in order
cols = [
    "Config", "Model", "ThresholdType", "Threshold",
    "Accuracy", "Balanced Acc", "Balanced F1", "F1-oui", "F1-non", "MCC"
]
df_results = df_results[cols].copy()

# 1) Identify the "best_MCC" row for each (Config, Model)
best_rows = df_results[df_results["ThresholdType"].str.startswith("best_")].copy()

# 2) For each Model, pick the Config with the maximal MCC(best_MCC)
#    Tie-break: (i) MCC, then (ii) Balanced F1, then (iii) Balanced Acc
best_choice_per_model = (
    best_rows.sort_values(
        ["Model", "MCC", "Balanced F1", "Balanced Acc"],
        ascending=[True, False, False, False]
    )
    .groupby("Model", as_index=False)
    .head(1)[["Model", "Config"]]
)

# 3) Filter df_results to keep only these (Model, Config)
df_best = df_results.merge(best_choice_per_model, on=["Model", "Config"], how="inner")

# 4) And keep only the two rows: fixed_0.50 and best_MCC
#    (in case there are other best_* in the future)
mask_keep = df_best["ThresholdType"].isin(["fixed_0.50", "best_MCC"]) | df_best["ThresholdType"].str.startswith("best_MCC")
df_best = df_best[mask_keep].copy()

# 5) Clean sort: by Model, then best before fixed (or the reverse if preferred)
df_best["is_best"] = (df_best["ThresholdType"].str.startswith("best_")).astype(int)
df_best = df_best.sort_values(
    ["Model", "is_best"],
    ascending=[True, False]
).drop(columns=["is_best"]).reset_index(drop=True)

# 6) Round to 2 decimals for display + export
num_cols = ["Threshold", "Accuracy", "Balanced Acc", "Balanced F1", "F1-oui", "F1-non", "MCC"]
df_best[num_cols] = df_best[num_cols].astype(float).round(2)

print("\n" + "=" * 100)
print("RESULTS — BEST CONFIG PER MODEL (by MCC on best_MCC) — show fixed_0.50 + best_MCC (2 decimals)")
print("=" * 100)
print(df_best.to_string(index=False))

output_file = os.path.join(OUTPUT_PATH, "results_best_config_per_model__fixed_vs_bestMCC.xlsx")
df_best.to_excel(output_file, index=False)
print(f"\n✅ Results saved: {output_file}")


# Base 3 - MLP 2 layers

In [ ]:
# ============================================================
# SIMPLE EVALUATION — 8 models + MLP (2 layers) (with JuriBERT)
# + COMPARISON OF 3 INPUT CONFIGS (article/chunk prompt-format)
# + THRESHOLDS: metrics at threshold 0.50 AND at the best threshold (optimized on OOF)
#
# Configs tested:
#   1) [article_text] [SEP] [chunk]
#   2) [ARTICLE] [article_text] [SEP] [CHUNK] [chunk]
#   3) [ARTICLE] Article {pred_art}: [article_text] [SEP] [CHUNK] [chunk]
#
# Objective: general oui/non classification performance
# Metrics: Accuracy, Balanced Acc, Balanced F1, F1-oui, F1-non, MCC
# CV grouped by decision_id (5 folds)
# Reproducible seeds (best-effort) — NO strict determinism
#
# NOTE: the "best threshold" is chosen to maximize MCC on the OOF probabilities.
# NOTE: For TF-IDF we use an MLP on dense features (via TruncatedSVD).
# ============================================================

import os, re, warnings, random
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
from tqdm import tqdm

from sklearn.preprocessing import StandardScaler
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.decomposition import TruncatedSVD
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import (
    accuracy_score, balanced_accuracy_score,
    f1_score, matthews_corrcoef
)

# -------------------------
# CONFIG
# -------------------------
BASE_PATH = "artifacts"  # not shipped — see DATA.md
PATH = "DATA/outputs/benchmark.csv"

OUTPUT = os.path.join(BASE_PATH, "outputs")
os.makedirs(OUTPUT, exist_ok=True)
OUTPUT_PATH = os.path.join(OUTPUT, "outputs_simple")
os.makedirs(OUTPUT_PATH, exist_ok=True)

N_SPLITS = 5
SEED = 42
MAX_LEN = 512
BATCH_SIZE = 16

# HF token (for SAUL/LLaMA if gated)
HF_TOKEN = os.environ["HF_TOKEN"]

# LLM config
LLM_MAX_LEN = 512
SAUL_MODEL = "Equall/Saul-7B-Base"
LLAMA_MODEL = "meta-llama/Llama-3.1-8B"
SAUL_BATCH_SIZE = 16
LLAMA_BATCH_SIZE = 16
USE_BF16 = True

# JuriBERT
JURIBERT_MODEL = "dascim/juribert-base"

# ============================================================
# MLP (2 layers "safe") — recommended to test on ~1k examples
# ============================================================
# 2 hidden layers: 256 -> 64 (the small 2nd one limits overfitting)
MLP_HIDDEN = (256, 64)
MLP_ALPHA = 2e-3         # stronger regularization (2 layers => higher overfit risk)
MLP_LR_INIT = 3e-4       # slightly lower LR for stability
MLP_MAX_ITER = 400       # a bit more, with early stopping anyway
MLP_EARLY_STOPPING = True
MLP_N_ITER_NO_CHANGE = 20

# TF-IDF -> SVD dim to give the MLP a stable dense input
TFIDF_MAX_FEATS = 50000
TFIDF_SVD_DIM = 512


# ============================================================
# 0) REPRO (best-effort, without strict determinism)
# ============================================================
def set_all_seeds(seed: int):
    os.environ["PYTHONHASHSEED"] = str(seed)
    random.seed(seed)
    np.random.seed(seed)
    try:
        import torch
        torch.manual_seed(seed)
        if torch.cuda.is_available():
            torch.cuda.manual_seed_all(seed)
        torch.backends.cudnn.benchmark = False
        try:
            torch.use_deterministic_algorithms(False)
        except Exception:
            pass
    except Exception:
        pass

set_all_seeds(SEED)


# ============================================================
# 1) LOADING + PREPARATION
# ============================================================
print("=" * 70)
print("LOADING DATA")
print("=" * 70)

df0 = pd.read_excel(PATH)
print(f"Available columns: {df0.columns.tolist()}")

cols_needed = ["decision_id", "chunk_id", "pred_art", "text", "article_text",
               "eval_A1", "eval_A2", "eval_A3"]
df0 = df0[[c for c in cols_needed if c in df0.columns]].copy()

def extract_oui_non(x):
    if pd.isna(x):
        return np.nan
    s = str(x).lower()
    has_oui = re.search(r"\boui\b", s) is not None
    has_non = re.search(r"\bnon\b", s) is not None
    if has_oui and not has_non:
        return "oui"
    if has_non and not has_oui:
        return "non"
    return np.nan

df0["a"] = df0["eval_A1"].apply(extract_oui_non)
df0["t"] = df0["eval_A2"].apply(extract_oui_non)
df0["s"] = df0["eval_A3"].apply(extract_oui_non)

def resolve_label(r):
    a, t, s = r["a"], r["t"], r["s"]
    if pd.notna(a) and pd.notna(t) and a == t:
        return a
    if pd.notna(a) and pd.notna(t) and a != t and pd.notna(s):
        return s
    return np.nan

df0["label_str"] = df0.apply(resolve_label, axis=1)
df0 = df0[df0["label_str"].isin(["oui", "non"])].copy()
df0["label"] = df0["label_str"].map({"oui": 1, "non": 0}).astype(int)

for c in ["text", "article_text", "pred_art"]:
    if c not in df0.columns:
        df0[c] = ""
    df0[c] = df0[c].fillna("").astype(str)

print(f"\nTotal examples: {len(df0)}")
print(f"  - oui: {(df0['label'] == 1).sum()} ({100*(df0['label'] == 1).mean():.1f}%)")
print(f"  - non: {(df0['label'] == 0).sum()} ({100*(df0['label'] == 0).mean():.1f}%)")
print(f"Unique decisions: {df0['decision_id'].nunique()}")


# ============================================================
# 1bis) BUILD INPUTS — 3 CONFIGS
# ============================================================
CONFIGS = {
    "cfg1": "1) article_text [SEP] chunk",
    "cfg2": "2) [ARTICLE] article_text [SEP] [CHUNK] chunk",
    "cfg3": "3) [ARTICLE] Article {pred_art}: article_text [SEP] [CHUNK] chunk",
}

def build_text_inputs(df: pd.DataFrame, cfg_key: str) -> list:
    art = df["article_text"].astype(str).str.strip()
    chunk = df["text"].astype(str).str.strip()
    pred = df["pred_art"].astype(str).str.strip()

    if cfg_key == "cfg1":
        s = art + " [SEP] " + chunk
    elif cfg_key == "cfg2":
        s = "[ARTICLE] " + art + " [SEP] [CHUNK] " + chunk
    elif cfg_key == "cfg3":
        s = "[ARTICLE] Article " + pred + ": " + art + " [SEP] [CHUNK] " + chunk
    else:
        raise ValueError(f"Unknown cfg_key={cfg_key}")
    return s.tolist()


# ============================================================
# 2) GROUPED CROSS-VALIDATION
# ============================================================
def make_grouped_folds(df, n_splits, seed):
    rng = np.random.default_rng(seed)
    group_sizes = df.groupby("decision_id").size().to_dict()
    uniq_groups = np.array(list(group_sizes.keys()))
    uniq_groups = uniq_groups[rng.permutation(len(uniq_groups))]
    uniq_groups = sorted(uniq_groups, key=lambda g: group_sizes[g], reverse=True)

    fold_loads = np.zeros(n_splits, dtype=int)
    group_to_fold = {}
    for g in uniq_groups:
        f = int(fold_loads.argmin())
        group_to_fold[g] = f
        fold_loads[f] += int(group_sizes[g])

    out = df.copy()
    out["fold"] = out["decision_id"].map(group_to_fold).astype(int)
    return out, fold_loads


# ============================================================
# 3) METRICS + THRESHOLD SELECTION
# ============================================================
def compute_metrics(y_true, y_pred):
    acc = accuracy_score(y_true, y_pred)
    bacc = balanced_accuracy_score(y_true, y_pred)
    f1_oui = f1_score(y_true, y_pred, pos_label=1, zero_division=0)
    f1_non = f1_score(y_true, y_pred, pos_label=0, zero_division=0)
    balanced_f1 = 0.5 * (f1_oui + f1_non)
    mcc = matthews_corrcoef(y_true, y_pred) if len(np.unique(y_true)) > 1 else np.nan
    return {
        "Accuracy": acc,
        "Balanced Acc": bacc,
        "Balanced F1": balanced_f1,
        "F1-oui": f1_oui,
        "F1-non": f1_non,
        "MCC": mcc
    }

def metrics_at_threshold(y_true, p_oui, thr: float):
    y_pred = (p_oui >= thr).astype(int)
    return compute_metrics(y_true, y_pred)

def find_best_threshold(y_true, p_oui, metric="MCC", grid=None):
    if grid is None:
        grid = np.round(np.arange(0.05, 0.951, 0.01), 2)

    best_thr = 0.50
    best = None
    best_val = -1e18

    for thr in grid:
        m = metrics_at_threshold(y_true, p_oui, float(thr))
        val = m.get(metric, np.nan)
        if np.isnan(val):
            continue
        if (val > best_val) or (val == best_val and abs(thr - 0.50) < abs(best_thr - 0.50)):
            best_val = val
            best_thr = float(thr)
            best = m

    if best is None:
        best = metrics_at_threshold(y_true, p_oui, 0.50)

    return best_thr, best


# ============================================================
# 4) EVALUATION (OOF on grouped folds) — returns OOF probabilities (MLP)
# ============================================================
def make_mlp(seed: int):
    return MLPClassifier(
        hidden_layer_sizes=MLP_HIDDEN,
        activation="relu",
        solver="adam",
        alpha=float(MLP_ALPHA),
        learning_rate_init=float(MLP_LR_INIT),
        max_iter=int(MLP_MAX_ITER),
        early_stopping=bool(MLP_EARLY_STOPPING),
        n_iter_no_change=int(MLP_N_ITER_NO_CHANGE),
        random_state=int(seed),
        verbose=False
    )

def evaluate_tfidf_oof_proba(texts, df, seed):
    """
    TF-IDF -> TruncatedSVD -> StandardScaler -> MLP
    (SVD/scaler fit on train fold only => no leakage)
    """
    y_true_oof = np.zeros(len(df), dtype=int)
    p_oui_oof = np.zeros(len(df), dtype=float)

    for fold_id in range(N_SPLITS):
        train_idx = (df["fold"] != fold_id).values
        test_idx = (df["fold"] == fold_id).values

        y_train = df.loc[train_idx, "label"].values
        y_test = df.loc[test_idx, "label"].values
        y_true_oof[test_idx] = y_test

        train_ids = np.where(train_idx)[0]
        test_ids = np.where(test_idx)[0]

        vec = TfidfVectorizer(
            ngram_range=(1, 2),
            min_df=2,
            max_features=int(TFIDF_MAX_FEATS),
            sublinear_tf=True
        )
        X_train = vec.fit_transform([texts[i] for i in train_ids])
        X_test = vec.transform([texts[i] for i in test_ids])

        if X_train.shape[1] <= 1:
            Z_train = X_train.toarray()
            Z_test = X_test.toarray()
        else:
            svd_dim = min(int(TFIDF_SVD_DIM), X_train.shape[1] - 1)
            svd = TruncatedSVD(n_components=int(max(1, svd_dim)), random_state=int(seed))
            Z_train = svd.fit_transform(X_train)
            Z_test = svd.transform(X_test)

        scaler = StandardScaler()
        Z_train = scaler.fit_transform(Z_train)
        Z_test = scaler.transform(Z_test)

        clf = make_mlp(seed)
        clf.fit(Z_train, y_train)

        proba = clf.predict_proba(Z_test)[:, 1]
        p_oui_oof[test_idx] = proba

    return y_true_oof, p_oui_oof


def evaluate_embeddings_oof_proba(X_dense, df, seed):
    """
    Embeddings -> StandardScaler -> MLP
    """
    y_true_oof = np.zeros(len(df), dtype=int)
    p_oui_oof = np.zeros(len(df), dtype=float)

    for fold_id in range(N_SPLITS):
        train_idx = (df["fold"] != fold_id).values
        test_idx = (df["fold"] == fold_id).values

        y_train = df.loc[train_idx, "label"].values
        y_test = df.loc[test_idx, "label"].values
        y_true_oof[test_idx] = y_test

        scaler = StandardScaler()
        X_train = scaler.fit_transform(X_dense[train_idx])
        X_test = scaler.transform(X_dense[test_idx])

        clf = make_mlp(seed)
        clf.fit(X_train, y_train)
        proba = clf.predict_proba(X_test)[:, 1]
        p_oui_oof[test_idx] = proba

    return y_true_oof, p_oui_oof


# ============================================================
# 5) ENCODERS
# ============================================================
import torch
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"\nDevice: {device}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

def encode_sentence_transformer(texts, model_name, batch_size=BATCH_SIZE, max_seq_length=256):
    from sentence_transformers import SentenceTransformer
    texts = ["" if t is None else str(t) for t in texts]
    st_device = "cuda" if torch.cuda.is_available() else "cpu"

    model = SentenceTransformer(model_name, device=st_device)
    model.max_seq_length = int(max_seq_length)
    embs = model.encode(
        texts,
        show_progress_bar=True,
        batch_size=int(batch_size),
        convert_to_numpy=True
    )
    del model
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    return embs

def encode_transformer_mean(texts, model_name, batch_size=BATCH_SIZE, max_len=MAX_LEN):
    from transformers import AutoTokenizer, AutoModel
    texts = ["" if t is None else str(t) for t in texts]

    tokenizer = AutoTokenizer.from_pretrained(model_name)
    model = AutoModel.from_pretrained(model_name).to(device)
    model.eval()

    all_embs = []
    with torch.no_grad():
        for i in tqdm(range(0, len(texts), batch_size), desc=model_name.split("/")[-1]):
            batch = texts[i:i + batch_size]
            inputs = tokenizer(
                batch, padding=True, truncation=True,
                max_length=int(max_len), return_tensors="pt"
            ).to(device)

            out = model(**inputs)
            mask = inputs["attention_mask"].unsqueeze(-1).float()
            emb = (out.last_hidden_state * mask).sum(1) / mask.sum(1).clamp_min(1.0)
            all_embs.append(emb.detach().cpu().numpy())

    del model, tokenizer
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    return np.vstack(all_embs)

def encode_llm_mean_pool(texts, model_name, batch_size, max_len, hf_token=None, use_bf16=True, desc="LLM"):
    from transformers import AutoTokenizer, AutoModel

    texts = ["" if t is None else str(t) for t in texts]
    tok = AutoTokenizer.from_pretrained(model_name, token=hf_token, use_fast=True)

    if tok.pad_token is None:
        tok.pad_token = tok.eos_token
    tok.padding_side = "right"

    model = AutoModel.from_pretrained(
        model_name,
        token=hf_token,
        torch_dtype=(torch.bfloat16 if (use_bf16 and torch.cuda.is_available()) else None),
        low_cpu_mem_usage=True,
    ).to(device)
    model.eval()

    if getattr(model.config, "pad_token_id", None) is None or model.config.pad_token_id < 0:
        model.config.pad_token_id = tok.pad_token_id

    all_embs = []
    with torch.no_grad():
        for i in tqdm(range(0, len(texts), batch_size), desc=desc):
            batch = texts[i:i + batch_size]
            enc = tok(
                batch, padding=True, truncation=True,
                max_length=int(max_len), return_tensors="pt"
            ).to(device)
            out = model(**enc)
            last = out.last_hidden_state
            mask = enc["attention_mask"].unsqueeze(-1).to(last.dtype)
            emb = (last * mask).sum(1) / mask.sum(1).clamp_min(1.0)
            all_embs.append(emb.float().cpu().numpy())

    del model, tok
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    return np.vstack(all_embs)


# ============================================================
# 6) MAIN — EXECUTION (for each config)
# ============================================================
if __name__ == "__main__":

    df_cv, fold_loads = make_grouped_folds(df0, N_SPLITS, seed=SEED)
    print(f"\nFold sizes: {fold_loads.tolist()}")

    all_rows = []
    THR_OPT_METRIC = "MCC"  # <-- change to optimize on "Balanced F1", etc.

    for cfg_key, cfg_name in CONFIGS.items():
        print("\n" + "=" * 90)
        print(f"CONFIG INPUT: {cfg_name}")
        print("=" * 90)

        texts = build_text_inputs(df_cv, cfg_key)

        def add_two_threshold_rows(model_name, y_true_oof, p_oui_oof):
            m_05 = metrics_at_threshold(y_true_oof, p_oui_oof, 0.50)
            row_05 = {"Config": cfg_key, "Model": model_name, "ThresholdType": "fixed_0.50", "Threshold": 0.50}
            row_05.update(m_05)
            all_rows.append(row_05)

            best_thr, m_best = find_best_threshold(y_true_oof, p_oui_oof, metric=THR_OPT_METRIC)
            row_best = {"Config": cfg_key, "Model": model_name, "ThresholdType": f"best_{THR_OPT_METRIC}", "Threshold": best_thr}
            row_best.update(m_best)
            all_rows.append(row_best)

            print(f"  -> {model_name:12s} | thr=0.50 MCC={m_05['MCC']:.4f} | best thr={best_thr:.2f} MCC={m_best['MCC']:.4f}")

        print("\n" + "=" * 50)
        print("TF-IDF + MLP (2 layers) — via SVD")
        print("=" * 50)
        y_true_oof, p_oui_oof = evaluate_tfidf_oof_proba(texts, df_cv, seed=SEED)
        add_two_threshold_rows("TF-IDF", y_true_oof, p_oui_oof)

        print("\n" + "=" * 50)
        print("ST-MiniLM-multilingual + MLP (2 layers)")
        print("=" * 50)
        X_minilm = encode_sentence_transformer(texts, "paraphrase-multilingual-MiniLM-L12-v2")
        y_true_oof, p_oui_oof = evaluate_embeddings_oof_proba(X_minilm, df_cv, seed=SEED)
        add_two_threshold_rows("ST-MiniLM", y_true_oof, p_oui_oof)

        print("\n" + "=" * 50)
        print("ST-MPNet-multilingual + MLP (2 layers)")
        print("=" * 50)
        X_mpnet = encode_sentence_transformer(texts, "paraphrase-multilingual-mpnet-base-v2")
        y_true_oof, p_oui_oof = evaluate_embeddings_oof_proba(X_mpnet, df_cv, seed=SEED)
        add_two_threshold_rows("ST-MPNet", y_true_oof, p_oui_oof)

        print("\n" + "=" * 50)
        print("CamemBERT + MLP (2 layers)")
        print("=" * 50)
        X_camem = encode_transformer_mean(texts, "camembert-base")
        y_true_oof, p_oui_oof = evaluate_embeddings_oof_proba(X_camem, df_cv, seed=SEED)
        add_two_threshold_rows("CamemBERT", y_true_oof, p_oui_oof)

        print("\n" + "=" * 50)
        print("CamemBERTav2 + MLP (2 layers)")
        print("=" * 50)
        X_camv2 = encode_transformer_mean(texts, "almanach/camembertav2-base")
        y_true_oof, p_oui_oof = evaluate_embeddings_oof_proba(X_camv2, df_cv, seed=SEED)
        add_two_threshold_rows("CamemBERTav2", y_true_oof, p_oui_oof)

        print("\n" + "=" * 50)
        print("JuriBERT-base + MLP (2 layers)")
        print("=" * 50)
        X_juribert = encode_transformer_mean(texts, JURIBERT_MODEL)
        y_true_oof, p_oui_oof = evaluate_embeddings_oof_proba(X_juribert, df_cv, seed=SEED)
        add_two_threshold_rows("JuriBERT-base", y_true_oof, p_oui_oof)

        print("\n" + "=" * 50)
        print("SAUL-7B-Base + MLP (2 layers)")
        print("=" * 50)
        X_saul = encode_llm_mean_pool(texts, SAUL_MODEL, SAUL_BATCH_SIZE, LLM_MAX_LEN, HF_TOKEN, USE_BF16, "SAUL")
        y_true_oof, p_oui_oof = evaluate_embeddings_oof_proba(X_saul, df_cv, seed=SEED)
        add_two_threshold_rows("SAUL-7B", y_true_oof, p_oui_oof)

        print("\n" + "=" * 50)
        print("LLaMA-3.1-8B + MLP (2 layers)")
        print("=" * 50)
        X_llama = encode_llm_mean_pool(texts, LLAMA_MODEL, LLAMA_BATCH_SIZE, LLM_MAX_LEN, HF_TOKEN, USE_BF16, "LLaMA")
        y_true_oof, p_oui_oof = evaluate_embeddings_oof_proba(X_llama, df_cv, seed=SEED)
        add_two_threshold_rows("LLaMA-3.1-8B", y_true_oof, p_oui_oof)

    print("\n" + "=" * 100)
    print(f"RESULTS — 0.50 vs best threshold (optimized on {THR_OPT_METRIC}) — grouped 5-fold CV")
    print("=" * 100)

    df_results = pd.DataFrame(all_rows)
    df_results = df_results[[
        "Config", "Model", "ThresholdType", "Threshold",
        "Accuracy", "Balanced Acc", "Balanced F1", "F1-oui", "F1-non", "MCC"
    ]].copy()

    df_results["is_best"] = (df_results["ThresholdType"].str.startswith("best_")).astype(int)
    df_results = df_results.sort_values(
        ["Config", "Model", "is_best", "MCC"],
        ascending=[True, True, False, False]
    ).drop(columns=["is_best"]).reset_index(drop=True)

    print(df_results.to_string(index=False, float_format="%.4f"))

    output_file = os.path.join(
        OUTPUT_PATH,
        "results_8_models_with_juribert__3_configs__thresholds__MLP_2layers.xlsx"
    )
    df_results.to_excel(output_file, index=False)
    print(f"\n✅ Results saved: {output_file}")


# LAWMA

In [ ]:
# ============================================================
# 2-STEP PIPELINE (LAWMA=LR, LLaMA=MLP1)
# A) Separately:
#    - LAWMA: embeddings -> LR OOF -> thr=0.5 + best thr (MCC)
#    - LLaMA: embeddings -> MLP1 OOF -> thr=0.5 + best thr (MCC)
#    - 3 text configs x pooling/layer variants
#    -> keep BEST variant per model
# ============================================================

import os, re, warnings, gc, hashlib
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
from tqdm import tqdm
import json

from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import (
    accuracy_score, balanced_accuracy_score,
    f1_score, matthews_corrcoef
)

import torch
from transformers import AutoTokenizer, AutoModel

# -------------------------
# CONFIG
# -------------------------
BASE_PATH = "artifacts"  # not shipped — see DATA.md
PATH = "DATA/outputs/benchmark.csv"

OUTPUT = os.path.join(BASE_PATH, "outputs")
os.makedirs(OUTPUT, exist_ok=True)
OUTPUT_PATH = os.path.join(OUTPUT, "outputs_lawma_lr_llama_mlp1_3cfg")
os.makedirs(OUTPUT_PATH, exist_ok=True)

CACHE_DIR = os.path.join(OUTPUT_PATH, "_cache_embs")
os.makedirs(CACHE_DIR, exist_ok=True)

N_SPLITS = 5
SEED = 42

HF_TOKEN = os.environ["HF_TOKEN"]

LLM_MAX_LEN = 512
USE_BF16 = True

# ---- MAIN CHANGE: Lawma-8B replaces SAUL-7B ----
LAWMA_MODEL = "ricdomolm/lawma-8b"
LLAMA_MODEL = "meta-llama/Llama-3.1-8B"

LAWMA_BATCH_SIZE = 16
LLAMA_BATCH_SIZE = 16

# MLP1 (1 hidden layer)
MLP_HIDDEN = (256,)
MLP_ALPHA = 1e-3
MLP_LR_INIT = 5e-4
MLP_MAX_ITER = 300
MLP_EARLY_STOPPING = True
MLP_N_ITER_NO_CHANGE = 15

POOLING_CONFIGS = [
    {"pooling": "mean", "layer_strategy": "last"},
    {"pooling": "mean", "layer_strategy": "layer:-2"},
    {"pooling": "mean", "layer_strategy": "avg_last_k:4"},
]

THR_GRID = np.linspace(0.05, 0.95, 181)  # step ~0.005

# Ensemble (after best-per-model selection)
W_GRID = np.linspace(0.0, 1.0, 41)       # step 0.025

# -------------------------
# Helpers: grouped folds
# -------------------------
def make_grouped_folds(df, n_splits, seed):
    rng = np.random.default_rng(seed)
    group_sizes = df.groupby("decision_id").size().to_dict()
    uniq_groups = np.array(list(group_sizes.keys()))
    uniq_groups = uniq_groups[rng.permutation(len(uniq_groups))]
    uniq_groups = sorted(uniq_groups, key=lambda g: group_sizes[g], reverse=True)

    fold_loads = np.zeros(n_splits, dtype=int)
    group_to_fold = {}
    for g in uniq_groups:
        f = int(fold_loads.argmin())
        group_to_fold[g] = f
        fold_loads[f] += int(group_sizes[g])

    out = df.copy()
    out["fold"] = out["decision_id"].map(group_to_fold).astype(int)
    return out, fold_loads

# -------------------------
# Metrics
# -------------------------
def compute_metrics_from_pred(y_true, y_pred):
    acc = accuracy_score(y_true, y_pred)
    bacc = balanced_accuracy_score(y_true, y_pred)
    f1_oui = f1_score(y_true, y_pred, pos_label=1, zero_division=0)
    f1_non = f1_score(y_true, y_pred, pos_label=0, zero_division=0)
    f1_macro = f1_score(y_true, y_pred, average="macro", zero_division=0)
    mcc = matthews_corrcoef(y_true, y_pred) if len(np.unique(y_true)) > 1 else np.nan
    return {
        "Accuracy": acc,
        "Balanced Acc": bacc,
        "Balanced F1": f1_macro,
        "F1-oui": f1_oui,
        "F1-non": f1_non,
        "MCC": mcc
    }

def best_threshold_on_oof(y_true, proba, grid=THR_GRID):
    best = {"thr": 0.5, "mcc": -1e9}
    for t in grid:
        y_pred = (proba >= t).astype(int)
        mcc = matthews_corrcoef(y_true, y_pred) if len(np.unique(y_true)) > 1 else np.nan
        if np.isfinite(mcc) and mcc > best["mcc"]:
            best = {"thr": float(t), "mcc": float(mcc)}
    return best["thr"], best["mcc"]

# -------------------------
# OOF predict_proba — LR / MLP1
# -------------------------
def oof_proba_with_model(X, df_cv, n_splits, seed, model_kind="lr"):
    y = df_cv["label"].values
    proba_oof = np.full(len(df_cv), np.nan, dtype=float)

    for fold in range(n_splits):
        tr = (df_cv["fold"] != fold).values
        te = (df_cv["fold"] == fold).values

        scaler = StandardScaler()
        Xtr = scaler.fit_transform(X[tr])
        Xte = scaler.transform(X[te])
        ytr = y[tr]

        if model_kind == "lr":
            clf = LogisticRegression(
                solver="lbfgs",
                max_iter=2000,
                C=1.0,
                random_state=seed
            )
        elif model_kind == "mlp1":
            clf = MLPClassifier(
                hidden_layer_sizes=MLP_HIDDEN,
                alpha=MLP_ALPHA,
                learning_rate_init=MLP_LR_INIT,
                max_iter=MLP_MAX_ITER,
                early_stopping=MLP_EARLY_STOPPING,
                n_iter_no_change=MLP_N_ITER_NO_CHANGE,
                random_state=seed
            )
        else:
            raise ValueError("model_kind must be 'lr' or 'mlp1'")

        clf.fit(Xtr, ytr)
        proba_oof[te] = clf.predict_proba(Xte)[:, 1]

    assert np.isfinite(proba_oof).all()
    return proba_oof

# -------------------------
# Embeddings encoder (pooling/layers) + cache
# -------------------------
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

def _clear_cuda():
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        torch.cuda.ipc_collect()

def _cache_key(model_name, pooling, layer_strategy, max_len, texts_hash):
    s = json.dumps(
        {"m": model_name, "p": pooling, "l": layer_strategy, "L": int(max_len), "h": texts_hash},
        sort_keys=True
    )
    return hashlib.md5(s.encode("utf-8")).hexdigest()

def _hash_texts(texts, n=200):
    sample = texts[:n] + texts[-n:] if len(texts) > 2*n else texts
    blob = "\n".join(map(str, sample)) + f"\n__N__{len(texts)}__"
    return hashlib.md5(blob.encode("utf-8")).hexdigest()

def encode_transformer_pool(texts, model_name, batch_size, max_len, hf_token, use_bf16, pooling, layer_strategy, desc):
    assert pooling in {"mean", "cls"}
    texts = ["" if t is None else str(t) for t in texts]
    texts_hash = _hash_texts(texts)

    key = _cache_key(model_name, pooling, layer_strategy, max_len, texts_hash)
    cache_path = os.path.join(CACHE_DIR, f"emb_{key}.npy")
    if os.path.exists(cache_path):
        return np.load(cache_path)

    tok = AutoTokenizer.from_pretrained(model_name, token=hf_token, use_fast=True)
    if tok.pad_token is None:
        tok.pad_token = tok.eos_token
    tok.padding_side = "right"

    dtype = torch.bfloat16 if (torch.cuda.is_available() and use_bf16) else None

    model = AutoModel.from_pretrained(
        model_name,
        token=hf_token,
        torch_dtype=dtype,
        low_cpu_mem_usage=True
    ).to(device)
    model.eval()

    if getattr(model.config, "pad_token_id", None) is None or model.config.pad_token_id < 0:
        model.config.pad_token_id = tok.pad_token_id

    def select_layers(hidden_states, strategy):
        if strategy == "last":
            return hidden_states[-1]
        if strategy.startswith("layer:"):
            idx = int(strategy.split(":")[1])
            return hidden_states[idx]
        if strategy.startswith("avg_last_k:"):
            k = int(strategy.split(":")[1])
            acc = None
            for h in hidden_states[-k:]:
                acc = h if acc is None else (acc + h)
            return acc / float(k)
        if strategy.startswith("concat_last_k:"):
            k = int(strategy.split(":")[1])
            return torch.cat(hidden_states[-k:], dim=-1)
        raise ValueError(f"Unknown layer_strategy: {strategy}")

    all_embs = []
    use_amp = torch.cuda.is_available() and use_bf16

    with torch.no_grad():
        for i in tqdm(range(0, len(texts), batch_size), desc=desc):
            batch = texts[i:i+batch_size]
            enc = tok(batch, padding=True, truncation=True, max_length=int(max_len), return_tensors="pt")
            enc = {k: v.to(device) for k, v in enc.items()}

            if use_amp:
                with torch.autocast("cuda", dtype=torch.bfloat16):
                    out = model(**enc, output_hidden_states=True, use_cache=False)
            else:
                out = model(**enc, output_hidden_states=True, use_cache=False)

            hs = out.hidden_states
            attn = enc["attention_mask"]
            x = select_layers(hs, layer_strategy)

            if pooling == "cls":
                emb = x[:, 0, :]
            else:
                mask = attn.unsqueeze(-1).to(x.dtype)
                emb = (x * mask).sum(dim=1) / mask.sum(dim=1).clamp_min(1.0)

            all_embs.append(emb.float().cpu().numpy())

            del enc, out, hs, attn, x, emb
            _clear_cuda()

    X = np.vstack(all_embs)
    np.save(cache_path, X)

    del model, tok
    _clear_cuda()
    return X

# -------------------------
# 3 text configs
# -------------------------
def make_text_inputs(df, cfg_id):
    art = df["article_text"].astype(str).str.strip()
    chunk = df["text"].astype(str).str.strip()
    pred_art = df["pred_art"].astype(str).str.strip()

    if cfg_id == "cfg1":
        return (art + " [SEP] " + chunk).tolist()
    if cfg_id == "cfg2":
        return ("[ARTICLE] " + art + " [SEP] [CHUNK] " + chunk).tolist()
    if cfg_id == "cfg3":
        return ("[ARTICLE] Article " + pred_art + ": " + art + " [SEP] [CHUNK] " + chunk).tolist()

    raise ValueError("cfg_id must be cfg1/cfg2/cfg3")

# ============================================================
# LOAD + PREP DATA (labels)
# ============================================================
print("="*80)
print("LOADING DATA")
print("="*80)

df0 = pd.read_excel(PATH)
cols_needed = ["decision_id", "chunk_id", "pred_art", "text", "article_text",
               "eval_A1", "eval_A2", "eval_A3"]
df0 = df0[[c for c in cols_needed if c in df0.columns]].copy()

def extract_oui_non(x):
    if pd.isna(x): return np.nan
    s = str(x).lower()
    has_oui = re.search(r"\boui\b", s) is not None
    has_non = re.search(r"\bnon\b", s) is not None
    if has_oui and not has_non: return "oui"
    if has_non and not has_oui: return "non"
    return np.nan

df0["a"] = df0["eval_A1"].apply(extract_oui_non)
df0["t"] = df0["eval_A2"].apply(extract_oui_non)
df0["s"] = df0["eval_A3"].apply(extract_oui_non)

def resolve_label(r):
    a, t, s = r["a"], r["t"], r["s"]
    if pd.notna(a) and pd.notna(t) and a == t:
        return a
    if pd.notna(a) and pd.notna(t) and a != t and pd.notna(s):
        return s
    return np.nan

df0["label_str"] = df0.apply(resolve_label, axis=1)
df0 = df0[df0["label_str"].isin(["oui", "non"])].copy()
df0["label"] = df0["label_str"].map({"oui": 1, "non": 0}).astype(int)

print(f"Total examples: {len(df0)} | oui={(df0.label==1).sum()} | non={(df0.label==0).sum()} | decisions={df0.decision_id.nunique()}")

df_cv, fold_loads = make_grouped_folds(df0, N_SPLITS, SEED)
print(f"Fold sizes: {fold_loads.tolist()}")

y_true = df_cv["label"].values

# ============================================================
# STEP A — separate evaluation per model
# ============================================================
def run_model_search(short_name, hf_model, batch_size, classifier_kind):
    rows = []
    best = None

    for cfg_id in ["cfg1", "cfg2", "cfg3"]:
        texts = make_text_inputs(df_cv, cfg_id)

        for pcfg in POOLING_CONFIGS:
            pooling = pcfg["pooling"]
            layer_strategy = pcfg["layer_strategy"]

            run_id = f"{short_name}|{cfg_id}|{pooling}|{layer_strategy}"
            print("\n" + "="*90)
            print(f"[{short_name}] {run_id}  (clf={classifier_kind})")
            print("="*90)

            X = encode_transformer_pool(
                texts=texts,
                model_name=hf_model,
                batch_size=batch_size,
                max_len=LLM_MAX_LEN,
                hf_token=HF_TOKEN,
                use_bf16=USE_BF16,
                pooling=pooling,
                layer_strategy=layer_strategy,
                desc=run_id
            )
            print(f"✓ Embeddings: {X.shape}")

            proba_oof = oof_proba_with_model(X, df_cv, N_SPLITS, SEED, model_kind=classifier_kind)

            # threshold 0.5
            pred_05 = (proba_oof >= 0.5).astype(int)
            met_05 = compute_metrics_from_pred(y_true, pred_05)

            # best MCC threshold
            best_thr, best_mcc = best_threshold_on_oof(y_true, proba_oof, THR_GRID)
            pred_best = (proba_oof >= best_thr).astype(int)
            met_best = compute_metrics_from_pred(y_true, pred_best)

            row = {
                "Model": short_name,
                "HF_model": hf_model,
                "Classifier": classifier_kind,
                "TextCfg": cfg_id,
                "pooling": pooling,
                "layer_strategy": layer_strategy,

                "Thr(best)": best_thr,
                "MCC(best)": met_best["MCC"],
                "Acc(best)": met_best["Accuracy"],
                "BAcc(best)": met_best["Balanced Acc"],
                "BF1(best)": met_best["Balanced F1"],
                "F1-oui(best)": met_best["F1-oui"],
                "F1-non(best)": met_best["F1-non"],

                "MCC@0.5": met_05["MCC"],
                "Acc@0.5": met_05["Accuracy"],
                "BAcc@0.5": met_05["Balanced Acc"],
                "BF1@0.5": met_05["Balanced F1"],
                "F1-oui@0.5": met_05["F1-oui"],
                "F1-non@0.5": met_05["F1-non"],

                "ΔMCC": met_best["MCC"] - met_05["MCC"],
                "proba_oof_path": None,
            }

            # save proba_oof
            proba_path = os.path.join(OUTPUT_PATH, f"proba_oof_{run_id.replace('|','__')}.npy")
            np.save(proba_path, proba_oof)
            row["proba_oof_path"] = proba_path

            rows.append(row)

            if (best is None) or (row["MCC(best)"] > best["MCC(best)"]):
                best = row

            print(f"@0.5  MCC={row['MCC@0.5']:.4f} | best thr={row['Thr(best)']:.3f} MCC={row['MCC(best)']:.4f} | Δ={row['ΔMCC']:.4f}")

    df_rows = pd.DataFrame(rows).sort_values(["MCC(best)", "BAcc(best)", "BF1(best)"], ascending=False).reset_index(drop=True)
    return df_rows, best

# --- LAWMA-8B (LR) — replaces SAUL-7B
df_lawma, best_lawma = run_model_search(
    short_name="Lawma-8B",
    hf_model=LAWMA_MODEL,
    batch_size=LAWMA_BATCH_SIZE,
    classifier_kind="lr"
)

# --- LLaMA (MLP1) — unchanged
df_llama, best_llama = run_model_search(
    short_name="LLaMA-3.1-8B",
    hf_model=LLAMA_MODEL,
    batch_size=LLAMA_BATCH_SIZE,
    classifier_kind="mlp1"
)

# save recap
out_a = os.path.join(OUTPUT_PATH, "STEP_A__lawma_lr__llama_mlp1__all_variants.xlsx")
with pd.ExcelWriter(out_a) as w:
    df_lawma.to_excel(w, index=False, sheet_name="Lawma_all")
    df_llama.to_excel(w, index=False, sheet_name="LLaMA_all")
print(f"\n✅ STEP A saved: {out_a}")

best_df = pd.DataFrame([best_lawma, best_llama])
out_best = os.path.join(OUTPUT_PATH, "STEP_A__BEST_PER_MODEL.xlsx")
best_df.to_excel(out_best, index=False)
print(f"✅ BEST per model: {out_best}")

print("\n" + "="*100)
print("BEST LAWMA:")
print(pd.Series(best_lawma)[["TextCfg","pooling","layer_strategy","Thr(best)","MCC(best)","MCC@0.5","ΔMCC","proba_oof_path"]])
print("\nBEST LLaMA:")
print(pd.Series(best_llama)[["TextCfg","pooling","layer_strategy","Thr(best)","MCC(best)","MCC@0.5","ΔMCC","proba_oof_path"]])
print("="*100)

# LAWMA MLP 1 2

In [ ]:
# ============================================================
# LAWMA-8B: MLP1 (1 layer) + MLP2 (2 layers)
# 3 text configs x 3 pooling/layer variants x 2 classifiers
# ============================================================

import os, re, warnings, gc, hashlib
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
from tqdm import tqdm
import json

from sklearn.preprocessing import StandardScaler
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import (
    accuracy_score, balanced_accuracy_score,
    f1_score, matthews_corrcoef
)

import torch
from transformers import AutoTokenizer, AutoModel

# -------------------------
# CONFIG
# -------------------------
BASE_PATH = "artifacts"  # not shipped — see DATA.md
PATH = "DATA/outputs/benchmark.csv"

OUTPUT = os.path.join(BASE_PATH, "outputs")
os.makedirs(OUTPUT, exist_ok=True)
OUTPUT_PATH = os.path.join(OUTPUT, "outputs_lawma_mlp1_mlp2_3cfg")
os.makedirs(OUTPUT_PATH, exist_ok=True)

CACHE_DIR = os.path.join(OUTPUT_PATH, "_cache_embs")
os.makedirs(CACHE_DIR, exist_ok=True)

# Also reuse the LR run cache if available
CACHE_DIR_LR = os.path.join(OUTPUT, "outputs_lawma_lr_llama_mlp1_3cfg", "_cache_embs")

N_SPLITS = 5
SEED = 42

HF_TOKEN = os.environ["HF_TOKEN"]

LLM_MAX_LEN = 512
USE_BF16 = True

LAWMA_MODEL = "ricdomolm/lawma-8b"
LAWMA_BATCH_SIZE = 16

# MLP1 (1 hidden layer)
MLP1_HIDDEN = (256,)
MLP1_ALPHA = 1e-3
MLP1_LR_INIT = 5e-4
MLP1_MAX_ITER = 300
MLP1_EARLY_STOPPING = True
MLP1_N_ITER_NO_CHANGE = 15

# MLP2 (2 hidden layers)
MLP2_HIDDEN = (256, 128)
MLP2_ALPHA = 1e-3
MLP2_LR_INIT = 5e-4
MLP2_MAX_ITER = 300
MLP2_EARLY_STOPPING = True
MLP2_N_ITER_NO_CHANGE = 15

POOLING_CONFIGS = [
    {"pooling": "mean", "layer_strategy": "last"},
    {"pooling": "mean", "layer_strategy": "layer:-2"},
    {"pooling": "mean", "layer_strategy": "avg_last_k:4"},
]

THR_GRID = np.linspace(0.05, 0.95, 181)

# -------------------------
# Helpers: grouped folds
# -------------------------
def make_grouped_folds(df, n_splits, seed):
    rng = np.random.default_rng(seed)
    group_sizes = df.groupby("decision_id").size().to_dict()
    uniq_groups = np.array(list(group_sizes.keys()))
    uniq_groups = uniq_groups[rng.permutation(len(uniq_groups))]
    uniq_groups = sorted(uniq_groups, key=lambda g: group_sizes[g], reverse=True)

    fold_loads = np.zeros(n_splits, dtype=int)
    group_to_fold = {}
    for g in uniq_groups:
        f = int(fold_loads.argmin())
        group_to_fold[g] = f
        fold_loads[f] += int(group_sizes[g])

    out = df.copy()
    out["fold"] = out["decision_id"].map(group_to_fold).astype(int)
    return out, fold_loads

# -------------------------
# Metrics
# -------------------------
def compute_metrics_from_pred(y_true, y_pred):
    acc = accuracy_score(y_true, y_pred)
    bacc = balanced_accuracy_score(y_true, y_pred)
    f1_oui = f1_score(y_true, y_pred, pos_label=1, zero_division=0)
    f1_non = f1_score(y_true, y_pred, pos_label=0, zero_division=0)
    f1_macro = f1_score(y_true, y_pred, average="macro", zero_division=0)
    mcc = matthews_corrcoef(y_true, y_pred) if len(np.unique(y_true)) > 1 else np.nan
    return {
        "Accuracy": acc,
        "Balanced Acc": bacc,
        "Balanced F1": f1_macro,
        "F1-oui": f1_oui,
        "F1-non": f1_non,
        "MCC": mcc
    }

def best_threshold_on_oof(y_true, proba, grid=THR_GRID):
    best = {"thr": 0.5, "mcc": -1e9}
    for t in grid:
        y_pred = (proba >= t).astype(int)
        mcc = matthews_corrcoef(y_true, y_pred) if len(np.unique(y_true)) > 1 else np.nan
        if np.isfinite(mcc) and mcc > best["mcc"]:
            best = {"thr": float(t), "mcc": float(mcc)}
    return best["thr"], best["mcc"]

# -------------------------
# OOF predict_proba — MLP1 / MLP2
# -------------------------
def oof_proba_with_model(X, df_cv, n_splits, seed, model_kind="mlp1"):
    y = df_cv["label"].values
    proba_oof = np.full(len(df_cv), np.nan, dtype=float)

    for fold in range(n_splits):
        tr = (df_cv["fold"] != fold).values
        te = (df_cv["fold"] == fold).values

        scaler = StandardScaler()
        Xtr = scaler.fit_transform(X[tr])
        Xte = scaler.transform(X[te])
        ytr = y[tr]

        if model_kind == "mlp1":
            clf = MLPClassifier(
                hidden_layer_sizes=MLP1_HIDDEN,
                alpha=MLP1_ALPHA,
                learning_rate_init=MLP1_LR_INIT,
                max_iter=MLP1_MAX_ITER,
                early_stopping=MLP1_EARLY_STOPPING,
                n_iter_no_change=MLP1_N_ITER_NO_CHANGE,
                random_state=seed
            )
        elif model_kind == "mlp2":
            clf = MLPClassifier(
                hidden_layer_sizes=MLP2_HIDDEN,
                alpha=MLP2_ALPHA,
                learning_rate_init=MLP2_LR_INIT,
                max_iter=MLP2_MAX_ITER,
                early_stopping=MLP2_EARLY_STOPPING,
                n_iter_no_change=MLP2_N_ITER_NO_CHANGE,
                random_state=seed
            )
        else:
            raise ValueError("model_kind must be 'mlp1' or 'mlp2'")

        clf.fit(Xtr, ytr)
        proba_oof[te] = clf.predict_proba(Xte)[:, 1]

    assert np.isfinite(proba_oof).all()
    return proba_oof

# -------------------------
# Embeddings encoder (pooling/layers) + cache
# -------------------------
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

def _clear_cuda():
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        torch.cuda.ipc_collect()

def _cache_key(model_name, pooling, layer_strategy, max_len, texts_hash):
    s = json.dumps(
        {"m": model_name, "p": pooling, "l": layer_strategy, "L": int(max_len), "h": texts_hash},
        sort_keys=True
    )
    return hashlib.md5(s.encode("utf-8")).hexdigest()

def _hash_texts(texts, n=200):
    sample = texts[:n] + texts[-n:] if len(texts) > 2*n else texts
    blob = "\n".join(map(str, sample)) + f"\n__N__{len(texts)}__"
    return hashlib.md5(blob.encode("utf-8")).hexdigest()

def encode_transformer_pool(texts, model_name, batch_size, max_len, hf_token, use_bf16, pooling, layer_strategy, desc):
    assert pooling in {"mean", "cls"}
    texts = ["" if t is None else str(t) for t in texts]
    texts_hash = _hash_texts(texts)

    key = _cache_key(model_name, pooling, layer_strategy, max_len, texts_hash)

    # Look in the current cache AND in the previous LR run cache
    cache_path = os.path.join(CACHE_DIR, f"emb_{key}.npy")
    cache_path_lr = os.path.join(CACHE_DIR_LR, f"emb_{key}.npy") if os.path.isdir(CACHE_DIR_LR) else None

    if os.path.exists(cache_path):
        print(f"  ⚡ Cache hit (local): {cache_path}")
        return np.load(cache_path)
    if cache_path_lr and os.path.exists(cache_path_lr):
        print(f"  ⚡ Cache hit (run LR): {cache_path_lr}")
        X = np.load(cache_path_lr)
        np.save(cache_path, X)  # local copy
        return X

    tok = AutoTokenizer.from_pretrained(model_name, token=hf_token, use_fast=True)
    if tok.pad_token is None:
        tok.pad_token = tok.eos_token
    tok.padding_side = "right"

    dtype = torch.bfloat16 if (torch.cuda.is_available() and use_bf16) else None

    model = AutoModel.from_pretrained(
        model_name,
        token=hf_token,
        torch_dtype=dtype,
        low_cpu_mem_usage=True
    ).to(device)
    model.eval()

    if getattr(model.config, "pad_token_id", None) is None or model.config.pad_token_id < 0:
        model.config.pad_token_id = tok.pad_token_id

    def select_layers(hidden_states, strategy):
        if strategy == "last":
            return hidden_states[-1]
        if strategy.startswith("layer:"):
            idx = int(strategy.split(":")[1])
            return hidden_states[idx]
        if strategy.startswith("avg_last_k:"):
            k = int(strategy.split(":")[1])
            acc = None
            for h in hidden_states[-k:]:
                acc = h if acc is None else (acc + h)
            return acc / float(k)
        if strategy.startswith("concat_last_k:"):
            k = int(strategy.split(":")[1])
            return torch.cat(hidden_states[-k:], dim=-1)
        raise ValueError(f"Unknown layer_strategy: {strategy}")

    all_embs = []
    use_amp = torch.cuda.is_available() and use_bf16

    with torch.no_grad():
        for i in tqdm(range(0, len(texts), batch_size), desc=desc):
            batch = texts[i:i+batch_size]
            enc = tok(batch, padding=True, truncation=True, max_length=int(max_len), return_tensors="pt")
            enc = {k: v.to(device) for k, v in enc.items()}

            if use_amp:
                with torch.autocast("cuda", dtype=torch.bfloat16):
                    out = model(**enc, output_hidden_states=True, use_cache=False)
            else:
                out = model(**enc, output_hidden_states=True, use_cache=False)

            hs = out.hidden_states
            attn = enc["attention_mask"]
            x = select_layers(hs, layer_strategy)

            if pooling == "cls":
                emb = x[:, 0, :]
            else:
                mask = attn.unsqueeze(-1).to(x.dtype)
                emb = (x * mask).sum(dim=1) / mask.sum(dim=1).clamp_min(1.0)

            all_embs.append(emb.float().cpu().numpy())

            del enc, out, hs, attn, x, emb
            _clear_cuda()

    X = np.vstack(all_embs)
    np.save(cache_path, X)

    del model, tok
    _clear_cuda()
    return X

# -------------------------
# 3 text configs
# -------------------------
def make_text_inputs(df, cfg_id):
    art = df["article_text"].astype(str).str.strip()
    chunk = df["text"].astype(str).str.strip()
    pred_art = df["pred_art"].astype(str).str.strip()

    if cfg_id == "cfg1":
        return (art + " [SEP] " + chunk).tolist()
    if cfg_id == "cfg2":
        return ("[ARTICLE] " + art + " [SEP] [CHUNK] " + chunk).tolist()
    if cfg_id == "cfg3":
        return ("[ARTICLE] Article " + pred_art + ": " + art + " [SEP] [CHUNK] " + chunk).tolist()

    raise ValueError("cfg_id must be cfg1/cfg2/cfg3")

# ============================================================
# LOAD + PREP DATA
# ============================================================
print("="*80)
print("LOADING DATA")
print("="*80)

df0 = pd.read_excel(PATH)
cols_needed = ["decision_id", "chunk_id", "pred_art", "text", "article_text",
               "eval_A1", "eval_A2", "eval_A3"]
df0 = df0[[c for c in cols_needed if c in df0.columns]].copy()

def extract_oui_non(x):
    if pd.isna(x): return np.nan
    s = str(x).lower()
    has_oui = re.search(r"\boui\b", s) is not None
    has_non = re.search(r"\bnon\b", s) is not None
    if has_oui and not has_non: return "oui"
    if has_non and not has_oui: return "non"
    return np.nan

df0["a"] = df0["eval_A1"].apply(extract_oui_non)
df0["t"] = df0["eval_A2"].apply(extract_oui_non)
df0["s"] = df0["eval_A3"].apply(extract_oui_non)

def resolve_label(r):
    a, t, s = r["a"], r["t"], r["s"]
    if pd.notna(a) and pd.notna(t) and a == t:
        return a
    if pd.notna(a) and pd.notna(t) and a != t and pd.notna(s):
        return s
    return np.nan

df0["label_str"] = df0.apply(resolve_label, axis=1)
df0 = df0[df0["label_str"].isin(["oui", "non"])].copy()
df0["label"] = df0["label_str"].map({"oui": 1, "non": 0}).astype(int)

print(f"Total examples: {len(df0)} | oui={(df0.label==1).sum()} | non={(df0.label==0).sum()} | decisions={df0.decision_id.nunique()}")

df_cv, fold_loads = make_grouped_folds(df0, N_SPLITS, SEED)
print(f"Fold sizes: {fold_loads.tolist()}")

y_true = df_cv["label"].values

# ============================================================
# RUN — Lawma-8B with MLP1 and MLP2
# ============================================================
all_rows = []

for classifier_kind in ["mlp1", "mlp2"]:
    for cfg_id in ["cfg1", "cfg2", "cfg3"]:
        texts = make_text_inputs(df_cv, cfg_id)

        for pcfg in POOLING_CONFIGS:
            pooling = pcfg["pooling"]
            layer_strategy = pcfg["layer_strategy"]

            run_id = f"Lawma-8B|{cfg_id}|{pooling}|{layer_strategy}|{classifier_kind}"
            print("\n" + "="*90)
            print(f"  {run_id}")
            print("="*90)

            X = encode_transformer_pool(
                texts=texts,
                model_name=LAWMA_MODEL,
                batch_size=LAWMA_BATCH_SIZE,
                max_len=LLM_MAX_LEN,
                hf_token=HF_TOKEN,
                use_bf16=USE_BF16,
                pooling=pooling,
                layer_strategy=layer_strategy,
                desc=run_id
            )
            print(f"✓ Embeddings: {X.shape}")

            proba_oof = oof_proba_with_model(X, df_cv, N_SPLITS, SEED, model_kind=classifier_kind)

            # threshold 0.5
            pred_05 = (proba_oof >= 0.5).astype(int)
            met_05 = compute_metrics_from_pred(y_true, pred_05)

            # best MCC threshold
            best_thr, best_mcc = best_threshold_on_oof(y_true, proba_oof, THR_GRID)
            pred_best = (proba_oof >= best_thr).astype(int)
            met_best = compute_metrics_from_pred(y_true, pred_best)

            row = {
                "Model": "Lawma-8B",
                "HF_model": LAWMA_MODEL,
                "Classifier": classifier_kind,
                "TextCfg": cfg_id,
                "pooling": pooling,
                "layer_strategy": layer_strategy,

                "Thr(best)": best_thr,
                "MCC(best)": met_best["MCC"],
                "Acc(best)": met_best["Accuracy"],
                "BAcc(best)": met_best["Balanced Acc"],
                "BF1(best)": met_best["Balanced F1"],
                "F1-oui(best)": met_best["F1-oui"],
                "F1-non(best)": met_best["F1-non"],

                "MCC@0.5": met_05["MCC"],
                "Acc@0.5": met_05["Accuracy"],
                "BAcc@0.5": met_05["Balanced Acc"],
                "BF1@0.5": met_05["Balanced F1"],
                "F1-oui@0.5": met_05["F1-oui"],
                "F1-non@0.5": met_05["F1-non"],

                "ΔMCC": met_best["MCC"] - met_05["MCC"],
                "proba_oof_path": None,
            }

            proba_path = os.path.join(OUTPUT_PATH, f"proba_oof_{run_id.replace('|','__')}.npy")
            np.save(proba_path, proba_oof)
            row["proba_oof_path"] = proba_path

            all_rows.append(row)

            print(f"@0.5  MCC={row['MCC@0.5']:.4f} | best thr={row['Thr(best)']:.3f} MCC={row['MCC(best)']:.4f} | Δ={row['ΔMCC']:.4f}")

# ============================================================
# SAVE
# ============================================================
df_all = pd.DataFrame(all_rows).sort_values(["MCC(best)", "BAcc(best)", "BF1(best)"], ascending=False).reset_index(drop=True)

out_path = os.path.join(OUTPUT_PATH, "LAWMA_8B__mlp1_mlp2__all_variants.xlsx")
with pd.ExcelWriter(out_path) as w:
    df_all.to_excel(w, index=False, sheet_name="All")
    df_all[df_all.Classifier == "mlp1"].to_excel(w, index=False, sheet_name="MLP1")
    df_all[df_all.Classifier == "mlp2"].to_excel(w, index=False, sheet_name="MLP2")

print(f"\n✅ Results saved: {out_path}")

# Best per classifier
for clf_name in ["mlp1", "mlp2"]:
    sub = df_all[df_all.Classifier == clf_name]
    if len(sub) > 0:
        best = sub.iloc[0]
        print(f"\nBEST Lawma-8B ({clf_name.upper()}):")
        print(f"  Config: {best['TextCfg']} | {best['pooling']} | {best['layer_strategy']}")
        print(f"  MCC@0.5={best['MCC@0.5']:.4f} | best thr={best['Thr(best)']:.3f} MCC(best)={best['MCC(best)']:.4f}")

# COMPARISON LAWMA LLAMA

In [ ]:
# ============================================================
# FULL COMPARISON: LAWMA-8B vs LLaMA-3.1-8B
# 3 classifiers (LR, MLP1, MLP2) x 3 text configs x 3 pooling
# Reuse cached embeddings from previous runs
# ============================================================

import os, re, warnings, gc, hashlib
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
from tqdm import tqdm
import json

from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import (
    accuracy_score, balanced_accuracy_score,
    f1_score, matthews_corrcoef
)

import torch
from transformers import AutoTokenizer, AutoModel

# -------------------------
# CONFIG
# -------------------------
BASE_PATH = "artifacts"  # not shipped — see DATA.md
PATH = "DATA/outputs/benchmark.csv"

OUTPUT = os.path.join(BASE_PATH, "outputs")
os.makedirs(OUTPUT, exist_ok=True)
OUTPUT_PATH = os.path.join(OUTPUT, "outputs_lawma_vs_llama_full_comparison")
os.makedirs(OUTPUT_PATH, exist_ok=True)

CACHE_DIR = os.path.join(OUTPUT_PATH, "_cache_embs")
os.makedirs(CACHE_DIR, exist_ok=True)

# All previous caches to scan
PREV_CACHE_DIRS = [
    os.path.join(OUTPUT, "outputs_lawma_lr_llama_mlp1_3cfg", "_cache_embs"),
    os.path.join(OUTPUT, "outputs_lawma_mlp1_mlp2_3cfg", "_cache_embs"),
    os.path.join(OUTPUT, "outputs_saul_lr_llama_mlp1_3cfg", "_cache_embs"),
]

N_SPLITS = 5
SEED = 42

HF_TOKEN = os.environ["HF_TOKEN"]

LLM_MAX_LEN = 512
USE_BF16 = True

LAWMA_MODEL = "ricdomolm/lawma-8b"
LLAMA_MODEL = "meta-llama/Llama-3.1-8B"
BATCH_SIZE = 16

# Classifiers
MLP1_HIDDEN = (256,)
MLP2_HIDDEN = (256, 128)
MLP_ALPHA = 1e-3
MLP_LR_INIT = 5e-4
MLP_MAX_ITER = 300
MLP_EARLY_STOPPING = True
MLP_N_ITER_NO_CHANGE = 15

POOLING_CONFIGS = [
    {"pooling": "mean", "layer_strategy": "last"},
    {"pooling": "mean", "layer_strategy": "layer:-2"},
    {"pooling": "mean", "layer_strategy": "avg_last_k:4"},
]

THR_GRID = np.linspace(0.05, 0.95, 181)

# -------------------------
# Helpers
# -------------------------
def make_grouped_folds(df, n_splits, seed):
    rng = np.random.default_rng(seed)
    group_sizes = df.groupby("decision_id").size().to_dict()
    uniq_groups = np.array(list(group_sizes.keys()))
    uniq_groups = uniq_groups[rng.permutation(len(uniq_groups))]
    uniq_groups = sorted(uniq_groups, key=lambda g: group_sizes[g], reverse=True)
    fold_loads = np.zeros(n_splits, dtype=int)
    group_to_fold = {}
    for g in uniq_groups:
        f = int(fold_loads.argmin())
        group_to_fold[g] = f
        fold_loads[f] += int(group_sizes[g])
    out = df.copy()
    out["fold"] = out["decision_id"].map(group_to_fold).astype(int)
    return out, fold_loads

def compute_metrics_from_pred(y_true, y_pred):
    return {
        "Accuracy": accuracy_score(y_true, y_pred),
        "Balanced Acc": balanced_accuracy_score(y_true, y_pred),
        "Balanced F1": f1_score(y_true, y_pred, average="macro", zero_division=0),
        "F1-oui": f1_score(y_true, y_pred, pos_label=1, zero_division=0),
        "F1-non": f1_score(y_true, y_pred, pos_label=0, zero_division=0),
        "MCC": matthews_corrcoef(y_true, y_pred) if len(np.unique(y_true)) > 1 else np.nan
    }

def best_threshold_on_oof(y_true, proba, grid=THR_GRID):
    best = {"thr": 0.5, "mcc": -1e9}
    for t in grid:
        y_pred = (proba >= t).astype(int)
        mcc = matthews_corrcoef(y_true, y_pred) if len(np.unique(y_true)) > 1 else np.nan
        if np.isfinite(mcc) and mcc > best["mcc"]:
            best = {"thr": float(t), "mcc": float(mcc)}
    return best["thr"], best["mcc"]

# -------------------------
# OOF predict_proba
# -------------------------
def oof_proba_with_model(X, df_cv, n_splits, seed, model_kind="lr"):
    y = df_cv["label"].values
    proba_oof = np.full(len(df_cv), np.nan, dtype=float)
    for fold in range(n_splits):
        tr = (df_cv["fold"] != fold).values
        te = (df_cv["fold"] == fold).values
        scaler = StandardScaler()
        Xtr = scaler.fit_transform(X[tr])
        Xte = scaler.transform(X[te])
        ytr = y[tr]
        if model_kind == "lr":
            clf = LogisticRegression(solver="lbfgs", max_iter=2000, C=1.0, random_state=seed)
        elif model_kind == "mlp1":
            clf = MLPClassifier(hidden_layer_sizes=MLP1_HIDDEN, alpha=MLP_ALPHA,
                                learning_rate_init=MLP_LR_INIT, max_iter=MLP_MAX_ITER,
                                early_stopping=MLP_EARLY_STOPPING, n_iter_no_change=MLP_N_ITER_NO_CHANGE,
                                random_state=seed)
        elif model_kind == "mlp2":
            clf = MLPClassifier(hidden_layer_sizes=MLP2_HIDDEN, alpha=MLP_ALPHA,
                                learning_rate_init=MLP_LR_INIT, max_iter=MLP_MAX_ITER,
                                early_stopping=MLP_EARLY_STOPPING, n_iter_no_change=MLP_N_ITER_NO_CHANGE,
                                random_state=seed)
        else:
            raise ValueError(f"Unknown model_kind: {model_kind}")
        clf.fit(Xtr, ytr)
        proba_oof[te] = clf.predict_proba(Xte)[:, 1]
    assert np.isfinite(proba_oof).all()
    return proba_oof

# -------------------------
# Embeddings + cache multi-sources
# -------------------------
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

def _clear_cuda():
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        torch.cuda.ipc_collect()

def _cache_key(model_name, pooling, layer_strategy, max_len, texts_hash):
    s = json.dumps({"m": model_name, "p": pooling, "l": layer_strategy, "L": int(max_len), "h": texts_hash}, sort_keys=True)
    return hashlib.md5(s.encode("utf-8")).hexdigest()

def _hash_texts(texts, n=200):
    sample = texts[:n] + texts[-n:] if len(texts) > 2*n else texts
    blob = "\n".join(map(str, sample)) + f"\n__N__{len(texts)}__"
    return hashlib.md5(blob.encode("utf-8")).hexdigest()

def encode_transformer_pool(texts, model_name, batch_size, max_len, hf_token, use_bf16, pooling, layer_strategy, desc):
    assert pooling in {"mean", "cls"}
    texts = ["" if t is None else str(t) for t in texts]
    texts_hash = _hash_texts(texts)
    key = _cache_key(model_name, pooling, layer_strategy, max_len, texts_hash)
    fname = f"emb_{key}.npy"

    # Look in local cache
    cache_path = os.path.join(CACHE_DIR, fname)
    if os.path.exists(cache_path):
        print(f"  ⚡ Cache hit (local)")
        return np.load(cache_path)

    # Look in all previous caches
    for prev_dir in PREV_CACHE_DIRS:
        prev_path = os.path.join(prev_dir, fname)
        if os.path.exists(prev_path):
            print(f"  ⚡ Cache hit (prev): {prev_dir}")
            X = np.load(prev_path)
            np.save(cache_path, X)
            return X

    # No cache -> compute
    print(f"  🔄 Computing embeddings...")
    tok = AutoTokenizer.from_pretrained(model_name, token=hf_token, use_fast=True)
    if tok.pad_token is None:
        tok.pad_token = tok.eos_token
    tok.padding_side = "right"
    dtype = torch.bfloat16 if (torch.cuda.is_available() and use_bf16) else None
    model = AutoModel.from_pretrained(model_name, token=hf_token, torch_dtype=dtype, low_cpu_mem_usage=True).to(device)
    model.eval()
    if getattr(model.config, "pad_token_id", None) is None or model.config.pad_token_id < 0:
        model.config.pad_token_id = tok.pad_token_id

    def select_layers(hidden_states, strategy):
        if strategy == "last": return hidden_states[-1]
        if strategy.startswith("layer:"):
            return hidden_states[int(strategy.split(":")[1])]
        if strategy.startswith("avg_last_k:"):
            k = int(strategy.split(":")[1])
            acc = None
            for h in hidden_states[-k:]:
                acc = h if acc is None else (acc + h)
            return acc / float(k)
        raise ValueError(f"Unknown: {strategy}")

    all_embs = []
    use_amp = torch.cuda.is_available() and use_bf16
    with torch.no_grad():
        for i in tqdm(range(0, len(texts), batch_size), desc=desc):
            batch = texts[i:i+batch_size]
            enc = tok(batch, padding=True, truncation=True, max_length=int(max_len), return_tensors="pt")
            enc = {k: v.to(device) for k, v in enc.items()}
            if use_amp:
                with torch.autocast("cuda", dtype=torch.bfloat16):
                    out = model(**enc, output_hidden_states=True, use_cache=False)
            else:
                out = model(**enc, output_hidden_states=True, use_cache=False)
            hs = out.hidden_states
            attn = enc["attention_mask"]
            x = select_layers(hs, layer_strategy)
            mask = attn.unsqueeze(-1).to(x.dtype)
            emb = (x * mask).sum(dim=1) / mask.sum(dim=1).clamp_min(1.0)
            all_embs.append(emb.float().cpu().numpy())
            del enc, out, hs, attn, x, emb
            _clear_cuda()

    X = np.vstack(all_embs)
    np.save(cache_path, X)
    del model, tok
    _clear_cuda()
    return X

# -------------------------
# Text configs
# -------------------------
def make_text_inputs(df, cfg_id):
    art = df["article_text"].astype(str).str.strip()
    chunk = df["text"].astype(str).str.strip()
    pred_art = df["pred_art"].astype(str).str.strip()
    if cfg_id == "cfg1": return (art + " [SEP] " + chunk).tolist()
    if cfg_id == "cfg2": return ("[ARTICLE] " + art + " [SEP] [CHUNK] " + chunk).tolist()
    if cfg_id == "cfg3": return ("[ARTICLE] Article " + pred_art + ": " + art + " [SEP] [CHUNK] " + chunk).tolist()
    raise ValueError(cfg_id)

# ============================================================
# LOAD DATA
# ============================================================
print("="*80)
print("LOADING DATA")
print("="*80)

df0 = pd.read_excel(PATH)
cols_needed = ["decision_id", "chunk_id", "pred_art", "text", "article_text",
               "eval_A1", "eval_A2", "eval_A3"]
df0 = df0[[c for c in cols_needed if c in df0.columns]].copy()

def extract_oui_non(x):
    if pd.isna(x): return np.nan
    s = str(x).lower()
    has_oui = re.search(r"\boui\b", s) is not None
    has_non = re.search(r"\bnon\b", s) is not None
    if has_oui and not has_non: return "oui"
    if has_non and not has_oui: return "non"
    return np.nan

df0["a"] = df0["eval_A1"].apply(extract_oui_non)
df0["t"] = df0["eval_A2"].apply(extract_oui_non)
df0["s"] = df0["eval_A3"].apply(extract_oui_non)

def resolve_label(r):
    a, t, s = r["a"], r["t"], r["s"]
    if pd.notna(a) and pd.notna(t) and a == t: return a
    if pd.notna(a) and pd.notna(t) and a != t and pd.notna(s): return s
    return np.nan

df0["label_str"] = df0.apply(resolve_label, axis=1)
df0 = df0[df0["label_str"].isin(["oui", "non"])].copy()
df0["label"] = df0["label_str"].map({"oui": 1, "non": 0}).astype(int)

print(f"Total examples: {len(df0)} | oui={(df0.label==1).sum()} | non={(df0.label==0).sum()} | decisions={df0.decision_id.nunique()}")

df_cv, fold_loads = make_grouped_folds(df0, N_SPLITS, SEED)
print(f"Fold sizes: {fold_loads.tolist()}")
y_true = df_cv["label"].values

# ============================================================
# RUN ALL COMBINATIONS
# ============================================================
MODELS = [
    ("Lawma-8B", LAWMA_MODEL),
    ("LLaMA-3.1-8B", LLAMA_MODEL),
]
CLASSIFIERS = ["lr", "mlp1", "mlp2"]

all_rows = []

for model_name, hf_model in MODELS:
    for cfg_id in ["cfg1", "cfg2", "cfg3"]:
        texts = make_text_inputs(df_cv, cfg_id)

        for pcfg in POOLING_CONFIGS:
            pooling = pcfg["pooling"]
            layer_strategy = pcfg["layer_strategy"]

            emb_id = f"{model_name}|{cfg_id}|{pooling}|{layer_strategy}"
            print("\n" + "-"*80)
            print(f"  EMB: {emb_id}")
            print("-"*80)

            X = encode_transformer_pool(
                texts=texts, model_name=hf_model, batch_size=BATCH_SIZE,
                max_len=LLM_MAX_LEN, hf_token=HF_TOKEN, use_bf16=USE_BF16,
                pooling=pooling, layer_strategy=layer_strategy, desc=emb_id
            )
            print(f"  ✓ Embeddings: {X.shape}")

            for clf_kind in CLASSIFIERS:
                run_id = f"{emb_id}|{clf_kind}"

                proba_oof = oof_proba_with_model(X, df_cv, N_SPLITS, SEED, model_kind=clf_kind)

                pred_05 = (proba_oof >= 0.5).astype(int)
                met_05 = compute_metrics_from_pred(y_true, pred_05)

                best_thr, best_mcc = best_threshold_on_oof(y_true, proba_oof, THR_GRID)
                pred_best = (proba_oof >= best_thr).astype(int)
                met_best = compute_metrics_from_pred(y_true, pred_best)

                row = {
                    "Model": model_name,
                    "Classifier": clf_kind,
                    "TextCfg": cfg_id,
                    "pooling": pooling,
                    "layer_strategy": layer_strategy,
                    "Thr(best)": best_thr,
                    "MCC(best)": met_best["MCC"],
                    "Acc(best)": met_best["Accuracy"],
                    "BAcc(best)": met_best["Balanced Acc"],
                    "BF1(best)": met_best["Balanced F1"],
                    "F1-oui(best)": met_best["F1-oui"],
                    "F1-non(best)": met_best["F1-non"],
                    "MCC@0.5": met_05["MCC"],
                    "Acc@0.5": met_05["Accuracy"],
                    "BAcc@0.5": met_05["Balanced Acc"],
                    "BF1@0.5": met_05["Balanced F1"],
                    "F1-oui@0.5": met_05["F1-oui"],
                    "F1-non@0.5": met_05["F1-non"],
                    "ΔMCC": met_best["MCC"] - met_05["MCC"],
                }
                all_rows.append(row)

                print(f"    [{clf_kind:4s}] @0.5 MCC={met_05['MCC']:.4f} | best thr={best_thr:.3f} MCC={met_best['MCC']:.4f}")

# ============================================================
# SAVE ALL RESULTS
# ============================================================
df_all = pd.DataFrame(all_rows)

out_path = os.path.join(OUTPUT_PATH, "FULL_COMPARISON__lawma_vs_llama__lr_mlp1_mlp2.xlsx")
with pd.ExcelWriter(out_path) as w:
    df_all.to_excel(w, index=False, sheet_name="All")
    for clf in CLASSIFIERS:
        df_all[df_all.Classifier == clf].sort_values("MCC(best)", ascending=False).to_excel(
            w, index=False, sheet_name=f"All_{clf.upper()}")

print(f"\n✅ Results saved: {out_path}")

# ============================================================
# COMPARISON TABLE: Lawma vs LLaMA side by side
# ============================================================
print("\n" + "="*120)
print("TABLEAU COMPARATIF — MCC(best): Lawma-8B vs LLaMA-3.1-8B")
print("="*120)

for clf_kind in CLASSIFIERS:
    print(f"\n{'─'*100}")
    print(f"  Classifier: {clf_kind.upper()}")
    print(f"{'─'*100}")
    print(f"  {'Config':<6} {'Layer':<15} │ {'Lawma-8B':>10} │ {'LLaMA-3.1':>10} │ {'Δ(L-Ll)':>10} │ {'Winner':<12}")
    print(f"  {'─'*6} {'─'*15} ┼ {'─'*10} ┼ {'─'*10} ┼ {'─'*10} ┼ {'─'*12}")

    sub = df_all[df_all.Classifier == clf_kind]

    for cfg_id in ["cfg1", "cfg2", "cfg3"]:
        for pcfg in POOLING_CONFIGS:
            ls = pcfg["layer_strategy"]

            lawma_row = sub[(sub.Model == "Lawma-8B") & (sub.TextCfg == cfg_id) & (sub.layer_strategy == ls)]
            llama_row = sub[(sub.Model == "LLaMA-3.1-8B") & (sub.TextCfg == cfg_id) & (sub.layer_strategy == ls)]

            if len(lawma_row) == 0 or len(llama_row) == 0:
                print(f"  {cfg_id:<6} {ls:<15} │ {'N/A':>10} │ {'N/A':>10} │ {'N/A':>10} │ {'N/A':<12}")
                continue

            lawma_mcc = lawma_row.iloc[0]["MCC(best)"]
            llama_mcc = llama_row.iloc[0]["MCC(best)"]
            delta = lawma_mcc - llama_mcc
            winner = "Lawma" if delta > 0.005 else ("LLaMA" if delta < -0.005 else "≈ equal")

            print(f"  {cfg_id:<6} {ls:<15} │ {lawma_mcc:>10.4f} │ {llama_mcc:>10.4f} │ {delta:>+10.4f} │ {winner:<12}")

    # Best per model for this classifier
    lawma_best = sub[sub.Model == "Lawma-8B"].sort_values("MCC(best)", ascending=False).iloc[0]
    llama_best = sub[sub.Model == "LLaMA-3.1-8B"].sort_values("MCC(best)", ascending=False).iloc[0]
    print(f"\n  BEST Lawma  ({clf_kind.upper()}): {lawma_best['TextCfg']}|{lawma_best['layer_strategy']}  MCC={lawma_best['MCC(best)']:.4f}")
    print(f"  BEST LLaMA  ({clf_kind.upper()}): {llama_best['TextCfg']}|{llama_best['layer_strategy']}  MCC={llama_best['MCC(best)']:.4f}")

# Global summary
print("\n" + "="*120)
print("GLOBAL SUMMARY — BEST MCC(best) per model × classifier")
print("="*120)
print(f"  {'Classifier':<12} │ {'Lawma-8B':>12} │ {'LLaMA-3.1-8B':>14} │ {'Δ':>8} │ {'Winner':<10}")
print(f"  {'─'*12} ┼ {'─'*12} ┼ {'─'*14} ┼ {'─'*8} ┼ {'─'*10}")

for clf_kind in CLASSIFIERS:
    sub = df_all[df_all.Classifier == clf_kind]
    lawma_best = sub[sub.Model == "Lawma-8B"]["MCC(best)"].max()
    llama_best = sub[sub.Model == "LLaMA-3.1-8B"]["MCC(best)"].max()
    delta = lawma_best - llama_best
    winner = "Lawma" if delta > 0.005 else ("LLaMA" if delta < -0.005 else "≈ equal")
    print(f"  {clf_kind.upper():<12} │ {lawma_best:>12.4f} │ {llama_best:>14.4f} │ {delta:>+8.4f} │ {winner:<10}")

# Lawma wins count
print(f"\n  Lawma wins (out of 27 configs): ", end="")
wins = 0
for _, row_l in df_all[df_all.Model == "Lawma-8B"].iterrows():
    match = df_all[(df_all.Model == "LLaMA-3.1-8B") &
                   (df_all.Classifier == row_l["Classifier"]) &
                   (df_all.TextCfg == row_l["TextCfg"]) &
                   (df_all.layer_strategy == row_l["layer_strategy"])]
    if len(match) > 0 and row_l["MCC(best)"] > match.iloc[0]["MCC(best)"] + 0.005:
        wins += 1
print(f"{wins}/27")

# Layers n°1 (Saul & LLaMa)





In [ ]:
# ============================================================
# 2-STEP PIPELINE (SAUL=LR, LLaMA=MLP1)
# A) Separately:
#    - SAUL: embeddings -> LR OOF -> thr=0.5 + best thr (MCC)
#    - LLaMA: embeddings -> MLP1 OOF -> thr=0.5 + best thr (MCC)
#    - 3 text configs x pooling/layer variants
#    -> keep BEST variant per model
# ============================================================

import os, re, warnings, gc, hashlib
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
from tqdm import tqdm
import json

from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import (
    accuracy_score, balanced_accuracy_score,
    f1_score, matthews_corrcoef
)

import torch
from transformers import AutoTokenizer, AutoModel

# -------------------------
# CONFIG
# -------------------------
BASE_PATH = "artifacts"  # not shipped — see DATA.md
PATH = "DATA/outputs/benchmark.csv"

OUTPUT = os.path.join(BASE_PATH, "outputs")
os.makedirs(OUTPUT, exist_ok=True)
OUTPUT_PATH = os.path.join(OUTPUT, "outputs_saul_lr_llama_mlp1_3cfg")
os.makedirs(OUTPUT_PATH, exist_ok=True)

CACHE_DIR = os.path.join(OUTPUT_PATH, "_cache_embs")
os.makedirs(CACHE_DIR, exist_ok=True)

N_SPLITS = 5
SEED = 42

HF_TOKEN = os.environ["HF_TOKEN"]

LLM_MAX_LEN = 512
USE_BF16 = True

SAUL_MODEL = "Equall/Saul-7B-Base"
LLAMA_MODEL = "meta-llama/Llama-3.1-8B"

SAUL_BATCH_SIZE = 16
LLAMA_BATCH_SIZE = 16

# MLP1 (1 hidden layer)
MLP_HIDDEN = (256,)
MLP_ALPHA = 1e-3
MLP_LR_INIT = 5e-4
MLP_MAX_ITER = 300
MLP_EARLY_STOPPING = True
MLP_N_ITER_NO_CHANGE = 15

POOLING_CONFIGS = [
    {"pooling": "mean", "layer_strategy": "last"},
    {"pooling": "mean", "layer_strategy": "layer:-2"},
    {"pooling": "mean", "layer_strategy": "avg_last_k:4"},
]

THR_GRID = np.linspace(0.05, 0.95, 181)  # step ~0.005

# Ensemble (after best-per-model selection)
W_GRID = np.linspace(0.0, 1.0, 41)       # step 0.025

# -------------------------
# Helpers: grouped folds
# -------------------------
def make_grouped_folds(df, n_splits, seed):
    rng = np.random.default_rng(seed)
    group_sizes = df.groupby("decision_id").size().to_dict()
    uniq_groups = np.array(list(group_sizes.keys()))
    uniq_groups = uniq_groups[rng.permutation(len(uniq_groups))]
    uniq_groups = sorted(uniq_groups, key=lambda g: group_sizes[g], reverse=True)

    fold_loads = np.zeros(n_splits, dtype=int)
    group_to_fold = {}
    for g in uniq_groups:
        f = int(fold_loads.argmin())
        group_to_fold[g] = f
        fold_loads[f] += int(group_sizes[g])

    out = df.copy()
    out["fold"] = out["decision_id"].map(group_to_fold).astype(int)
    return out, fold_loads

# -------------------------
# Metrics
# -------------------------
def compute_metrics_from_pred(y_true, y_pred):
    acc = accuracy_score(y_true, y_pred)
    bacc = balanced_accuracy_score(y_true, y_pred)
    f1_oui = f1_score(y_true, y_pred, pos_label=1, zero_division=0)
    f1_non = f1_score(y_true, y_pred, pos_label=0, zero_division=0)
    f1_macro = f1_score(y_true, y_pred, average="macro", zero_division=0)
    mcc = matthews_corrcoef(y_true, y_pred) if len(np.unique(y_true)) > 1 else np.nan
    return {
        "Accuracy": acc,
        "Balanced Acc": bacc,
        "Balanced F1": f1_macro,   # (a.k.a. BF1)
        "F1-oui": f1_oui,
        "F1-non": f1_non,
        "MCC": mcc
    }

def best_threshold_on_oof(y_true, proba, grid=THR_GRID):
    best = {"thr": 0.5, "mcc": -1e9}
    for t in grid:
        y_pred = (proba >= t).astype(int)
        mcc = matthews_corrcoef(y_true, y_pred) if len(np.unique(y_true)) > 1 else np.nan
        if np.isfinite(mcc) and mcc > best["mcc"]:
            best = {"thr": float(t), "mcc": float(mcc)}
    return best["thr"], best["mcc"]

# -------------------------
# OOF predict_proba — LR / MLP1
# -------------------------
def oof_proba_with_model(X, df_cv, n_splits, seed, model_kind="lr"):
    """
    Return OOF probability (p(oui)) for each row of df_cv.
    - model_kind="lr": LogisticRegression
    - model_kind="mlp1": MLPClassifier (1 hidden layer) + early stopping
    """
    y = df_cv["label"].values
    proba_oof = np.full(len(df_cv), np.nan, dtype=float)

    for fold in range(n_splits):
        tr = (df_cv["fold"] != fold).values
        te = (df_cv["fold"] == fold).values

        scaler = StandardScaler()
        Xtr = scaler.fit_transform(X[tr])
        Xte = scaler.transform(X[te])
        ytr = y[tr]

        if model_kind == "lr":
            clf = LogisticRegression(
                solver="lbfgs",
                max_iter=2000,
                C=1.0,
                random_state=seed
            )
        elif model_kind == "mlp1":
            clf = MLPClassifier(
                hidden_layer_sizes=MLP_HIDDEN,
                alpha=MLP_ALPHA,
                learning_rate_init=MLP_LR_INIT,
                max_iter=MLP_MAX_ITER,
                early_stopping=MLP_EARLY_STOPPING,
                n_iter_no_change=MLP_N_ITER_NO_CHANGE,
                random_state=seed
            )
        else:
            raise ValueError("model_kind must be 'lr' or 'mlp1'")

        clf.fit(Xtr, ytr)
        proba_oof[te] = clf.predict_proba(Xte)[:, 1]

    assert np.isfinite(proba_oof).all()
    return proba_oof

# -------------------------
# Embeddings encoder (pooling/layers) + cache
# -------------------------
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

def _clear_cuda():
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        torch.cuda.ipc_collect()

def _cache_key(model_name, pooling, layer_strategy, max_len, texts_hash):
    s = json.dumps(
        {"m": model_name, "p": pooling, "l": layer_strategy, "L": int(max_len), "h": texts_hash},
        sort_keys=True
    )
    return hashlib.md5(s.encode("utf-8")).hexdigest()

def _hash_texts(texts, n=200):
    # "stable-ish" hash without blowing up runtime: takes n examples + total len
    sample = texts[:n] + texts[-n:] if len(texts) > 2*n else texts
    blob = "\n".join(map(str, sample)) + f"\n__N__{len(texts)}__"
    return hashlib.md5(blob.encode("utf-8")).hexdigest()

def encode_transformer_pool(texts, model_name, batch_size, max_len, hf_token, use_bf16, pooling, layer_strategy, desc):
    assert pooling in {"mean", "cls"}
    texts = ["" if t is None else str(t) for t in texts]
    texts_hash = _hash_texts(texts)

    key = _cache_key(model_name, pooling, layer_strategy, max_len, texts_hash)
    cache_path = os.path.join(CACHE_DIR, f"emb_{key}.npy")
    if os.path.exists(cache_path):
        return np.load(cache_path)

    tok = AutoTokenizer.from_pretrained(model_name, token=hf_token, use_fast=True)
    if tok.pad_token is None:
        tok.pad_token = tok.eos_token
    tok.padding_side = "right"

    dtype = torch.bfloat16 if (torch.cuda.is_available() and use_bf16) else None

    model = AutoModel.from_pretrained(
        model_name,
        token=hf_token,
        torch_dtype=dtype,
        low_cpu_mem_usage=True
    ).to(device)
    model.eval()

    if getattr(model.config, "pad_token_id", None) is None or model.config.pad_token_id < 0:
        model.config.pad_token_id = tok.pad_token_id

    def select_layers(hidden_states, strategy):
        if strategy == "last":
            return hidden_states[-1]
        if strategy.startswith("layer:"):
            idx = int(strategy.split(":")[1])
            return hidden_states[idx]
        if strategy.startswith("avg_last_k:"):
            k = int(strategy.split(":")[1])
            acc = None
            for h in hidden_states[-k:]:
                acc = h if acc is None else (acc + h)
            return acc / float(k)
        if strategy.startswith("concat_last_k:"):
            k = int(strategy.split(":")[1])
            return torch.cat(hidden_states[-k:], dim=-1)
        raise ValueError(f"Unknown layer_strategy: {strategy}")

    all_embs = []
    use_amp = torch.cuda.is_available() and use_bf16

    with torch.no_grad():
        for i in tqdm(range(0, len(texts), batch_size), desc=desc):
            batch = texts[i:i+batch_size]
            enc = tok(batch, padding=True, truncation=True, max_length=int(max_len), return_tensors="pt")
            enc = {k: v.to(device) for k, v in enc.items()}

            if use_amp:
                with torch.autocast("cuda", dtype=torch.bfloat16):
                    out = model(**enc, output_hidden_states=True, use_cache=False)
            else:
                out = model(**enc, output_hidden_states=True, use_cache=False)

            hs = out.hidden_states
            attn = enc["attention_mask"]
            x = select_layers(hs, layer_strategy)

            if pooling == "cls":
                emb = x[:, 0, :]
            else:
                mask = attn.unsqueeze(-1).to(x.dtype)
                emb = (x * mask).sum(dim=1) / mask.sum(dim=1).clamp_min(1.0)

            all_embs.append(emb.float().cpu().numpy())

            del enc, out, hs, attn, x, emb
            _clear_cuda()

    X = np.vstack(all_embs)
    np.save(cache_path, X)

    del model, tok
    _clear_cuda()
    return X

# -------------------------
# 3 text configs
# -------------------------
def make_text_inputs(df, cfg_id):
    art = df["article_text"].astype(str).str.strip()
    chunk = df["text"].astype(str).str.strip()
    pred_art = df["pred_art"].astype(str).str.strip()

    if cfg_id == "cfg1":
        return (art + " [SEP] " + chunk).tolist()
    if cfg_id == "cfg2":
        return ("[ARTICLE] " + art + " [SEP] [CHUNK] " + chunk).tolist()
    if cfg_id == "cfg3":
        return ("[ARTICLE] Article " + pred_art + ": " + art + " [SEP] [CHUNK] " + chunk).tolist()

    raise ValueError("cfg_id must be cfg1/cfg2/cfg3")

# ============================================================
# LOAD + PREP DATA (labels)
# ============================================================
print("="*80)
print("LOADING DATA")
print("="*80)

df0 = pd.read_excel(PATH)
cols_needed = ["decision_id", "chunk_id", "pred_art", "text", "article_text",
               "eval_A1", "eval_A2", "eval_A3"]
df0 = df0[[c for c in cols_needed if c in df0.columns]].copy()

def extract_oui_non(x):
    if pd.isna(x): return np.nan
    s = str(x).lower()
    has_oui = re.search(r"\boui\b", s) is not None
    has_non = re.search(r"\bnon\b", s) is not None
    if has_oui and not has_non: return "oui"
    if has_non and not has_oui: return "non"
    return np.nan

df0["a"] = df0["eval_A1"].apply(extract_oui_non)
df0["t"] = df0["eval_A2"].apply(extract_oui_non)
df0["s"] = df0["eval_A3"].apply(extract_oui_non)

def resolve_label(r):
    a, t, s = r["a"], r["t"], r["s"]
    if pd.notna(a) and pd.notna(t) and a == t:
        return a
    if pd.notna(a) and pd.notna(t) and a != t and pd.notna(s):
        return s
    return np.nan

df0["label_str"] = df0.apply(resolve_label, axis=1)
df0 = df0[df0["label_str"].isin(["oui", "non"])].copy()
df0["label"] = df0["label_str"].map({"oui": 1, "non": 0}).astype(int)

print(f"Total examples: {len(df0)} | oui={(df0.label==1).sum()} | non={(df0.label==0).sum()} | decisions={df0.decision_id.nunique()}")

df_cv, fold_loads = make_grouped_folds(df0, N_SPLITS, SEED)
print(f"Fold sizes: {fold_loads.tolist()}")

y_true = df_cv["label"].values

# ============================================================
# STEP A — separate evaluation per model
# ============================================================
def run_model_search(short_name, hf_model, batch_size, classifier_kind):
    """
    classifier_kind:
      - "lr"  for SAUL
      - "mlp1" for LLaMA
    """
    rows = []
    best = None

    for cfg_id in ["cfg1", "cfg2", "cfg3"]:
        texts = make_text_inputs(df_cv, cfg_id)

        for pcfg in POOLING_CONFIGS:
            pooling = pcfg["pooling"]
            layer_strategy = pcfg["layer_strategy"]

            run_id = f"{short_name}|{cfg_id}|{pooling}|{layer_strategy}"
            print("\n" + "="*90)
            print(f"[{short_name}] {run_id}  (clf={classifier_kind})")
            print("="*90)

            X = encode_transformer_pool(
                texts=texts,
                model_name=hf_model,
                batch_size=batch_size,
                max_len=LLM_MAX_LEN,
                hf_token=HF_TOKEN,
                use_bf16=USE_BF16,
                pooling=pooling,
                layer_strategy=layer_strategy,
                desc=run_id
            )
            print(f"✓ Embeddings: {X.shape}")

            proba_oof = oof_proba_with_model(X, df_cv, N_SPLITS, SEED, model_kind=classifier_kind)

            # threshold 0.5
            pred_05 = (proba_oof >= 0.5).astype(int)
            met_05 = compute_metrics_from_pred(y_true, pred_05)

            # best MCC threshold
            best_thr, best_mcc = best_threshold_on_oof(y_true, proba_oof, THR_GRID)
            pred_best = (proba_oof >= best_thr).astype(int)
            met_best = compute_metrics_from_pred(y_true, pred_best)

            row = {
                "Model": short_name,
                "HF_model": hf_model,
                "Classifier": classifier_kind,
                "TextCfg": cfg_id,
                "pooling": pooling,
                "layer_strategy": layer_strategy,

                "Thr(best)": best_thr,
                "MCC(best)": met_best["MCC"],
                "Acc(best)": met_best["Accuracy"],
                "BAcc(best)": met_best["Balanced Acc"],
                "BF1(best)": met_best["Balanced F1"],
                "F1-oui(best)": met_best["F1-oui"],
                "F1-non(best)": met_best["F1-non"],

                "MCC@0.5": met_05["MCC"],
                "Acc@0.5": met_05["Accuracy"],
                "BAcc@0.5": met_05["Balanced Acc"],
                "BF1@0.5": met_05["Balanced F1"],
                "F1-oui@0.5": met_05["F1-oui"],
                "F1-non@0.5": met_05["F1-non"],

                "ΔMCC": met_best["MCC"] - met_05["MCC"],
                "proba_oof_path": None,  # filled in after saving
            }

            # save proba_oof
            proba_path = os.path.join(OUTPUT_PATH, f"proba_oof_{run_id.replace('|','__')}.npy")
            np.save(proba_path, proba_oof)
            row["proba_oof_path"] = proba_path

            rows.append(row)

            if (best is None) or (row["MCC(best)"] > best["MCC(best)"]):
                best = row

            print(f"@0.5  MCC={row['MCC@0.5']:.4f} | best thr={row['Thr(best)']:.3f} MCC={row['MCC(best)']:.4f} | Δ={row['ΔMCC']:.4f}")

    df_rows = pd.DataFrame(rows).sort_values(["MCC(best)", "BAcc(best)", "BF1(best)"], ascending=False).reset_index(drop=True)
    return df_rows, best

# --- SAUL (LR)
df_saul, best_saul = run_model_search(
    short_name="SAUL-7B",
    hf_model=SAUL_MODEL,
    batch_size=SAUL_BATCH_SIZE,
    classifier_kind="lr"
)

# --- LLaMA (MLP1)
df_llama, best_llama = run_model_search(
    short_name="LLaMA-3.1-8B",
    hf_model=LLAMA_MODEL,
    batch_size=LLAMA_BATCH_SIZE,
    classifier_kind="mlp1"
)

# save recap
out_a = os.path.join(OUTPUT_PATH, "STEP_A__saul_lr__llama_mlp1__all_variants.xlsx")
with pd.ExcelWriter(out_a) as w:
    df_saul.to_excel(w, index=False, sheet_name="SAUL_all")
    df_llama.to_excel(w, index=False, sheet_name="LLaMA_all")
print(f"\n✅ STEP A saved: {out_a}")

best_df = pd.DataFrame([best_saul, best_llama])
out_best = os.path.join(OUTPUT_PATH, "STEP_A__BEST_PER_MODEL.xlsx")
best_df.to_excel(out_best, index=False)
print(f"✅ BEST per model: {out_best}")

print("\n" + "="*100)
print("BEST SAUL:")
print(pd.Series(best_saul)[["TextCfg","pooling","layer_strategy","Thr(best)","MCC(best)","MCC@0.5","ΔMCC","proba_oof_path"]])
print("\nBEST LLaMA:")
print(pd.Series(best_llama)[["TextCfg","pooling","layer_strategy","Thr(best)","MCC(best)","MCC@0.5","ΔMCC","proba_oof_path"]])
print("="*100)


# Layers n°2 (camemBert, CamemBERTA, St-LM, St-Net)

In [ ]:
# ============================================================
# GRID SEARCH — TRANSFORMERS & SENTENCE TRANSFORMERS
# PARALLELIZED VERSION + SAVING OF BEST EMBEDDINGS/OOF
#
# Parallelization:
#   - Embeddings: sequential (GPU-bound, 1 GPU)
#   - Classifiers: parallel with joblib (CPU-bound)
#   - Threshold search: vectorized with numpy
#
# Models tested (ALL with pooling/layer configs):
#   - CamemBERT, CamemBERTav2, JuriBERT-base
#   - ST-MiniLM, ST-MPNet (treated as transformers for pooling/layer)
#
# Grid: 3 text configs × 4 pooling/layer × 3 classifiers = 36 per model
# Total: 5 models × 36 = 180 combinations
#
# NEW: Save embeddings + OOF for the best config per model
# ============================================================

import os, re, warnings, gc, hashlib, json
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
from tqdm import tqdm

from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import (
    accuracy_score, balanced_accuracy_score,
    f1_score, matthews_corrcoef
)

from joblib import Parallel, delayed
import torch

# ============================================================
# CONFIG
# ============================================================
BASE_PATH = "artifacts"  # not shipped — see DATA.md
PATH = "DATA/outputs/benchmark.csv"

OUTPUT = os.path.join(BASE_PATH)
os.makedirs(OUTPUT, exist_ok=True)
OUTPUT_PATH = os.path.join(OUTPUT, "oof_proba_final")
os.makedirs(OUTPUT_PATH, exist_ok=True)

# Folder for the best results
BEST_OUTPUT_PATH = os.path.join(OUTPUT_PATH, "best_models")
os.makedirs(BEST_OUTPUT_PATH, exist_ok=True)

CACHE_DIR = os.path.join(OUTPUT_PATH, "_cache_embs")
os.makedirs(CACHE_DIR, exist_ok=True)

N_SPLITS = 5
SEED = 42

# Parallelization settings
N_JOBS = -1  # -1 = all CPU cores

# Transformer settings
MAX_LEN = 512
BATCH_SIZE = 16

# MLP configs
MLP1_HIDDEN = (256,)
MLP1_ALPHA = 1e-3
MLP1_LR_INIT = 5e-4
MLP1_MAX_ITER = 300

MLP2_HIDDEN = (256, 64)
MLP2_ALPHA = 2e-3
MLP2_LR_INIT = 3e-4
MLP2_MAX_ITER = 400

MLP_EARLY_STOPPING = True
MLP_N_ITER_NO_CHANGE = 15

# Threshold grid (vectorized)
THR_GRID = np.linspace(0.05, 0.95, 181)

# ============================================================
# ALL MODELS — Treated uniformly as transformers
# ============================================================
ALL_MODELS = {
    "CamemBERT": {
        "hf_model": "camembert-base",
        "batch_size": BATCH_SIZE,
        "type": "transformer",
    },
    "CamemBERTav2": {
        "hf_model": "almanach/camembertav2-base",
        "batch_size": BATCH_SIZE,
        "type": "transformer",
    },
    "JuriBERT-base": {
        "hf_model": "dascim/juribert-base",
        "batch_size": BATCH_SIZE,
        "type": "transformer",
    },
    # SentenceTransformers — access the underlying model
    "ST-MiniLM": {
        "hf_model": "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2",
        "batch_size": 32,
        "type": "sentence_transformer",
    },
    "ST-MPNet": {
        "hf_model": "sentence-transformers/paraphrase-multilingual-mpnet-base-v2",
        "batch_size": 32,
        "type": "sentence_transformer",
    },
}

# Grid search configs
TEXT_CONFIGS = ["cfg1", "cfg2", "cfg3"]
CLASSIFIERS = ["LR", "MLP1", "MLP2"]

POOLING_LAYER_CONFIGS = [
    {"pooling": "mean", "layer_strategy": "last"},
    {"pooling": "mean", "layer_strategy": "layer:-2"},
    {"pooling": "mean", "layer_strategy": "avg_last_k:4"},
    {"pooling": "cls", "layer_strategy": "last"},
]


# ============================================================
# HELPERS
# ============================================================
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")


def _clear_cuda():
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        torch.cuda.ipc_collect()


def _hash_texts(texts, n=200):
    sample = texts[:n] + texts[-n:] if len(texts) > 2*n else texts
    blob = "\n".join(map(str, sample)) + f"\n__N__{len(texts)}__"
    return hashlib.md5(blob.encode("utf-8")).hexdigest()


def _cache_key(model_name, pooling, layer_strategy, max_len, texts_hash):
    s = json.dumps(
        {"m": model_name, "p": pooling, "l": layer_strategy, "L": int(max_len), "h": texts_hash},
        sort_keys=True
    )
    return hashlib.md5(s.encode("utf-8")).hexdigest()


# ============================================================
# TEXT CONFIG BUILDERS
# ============================================================
def make_text_inputs(df, cfg_id):
    art = df["article_text"].astype(str).str.strip()
    chunk = df["text"].astype(str).str.strip()
    pred_art = df["pred_art"].astype(str).str.strip()

    if cfg_id == "cfg1":
        return (art + " [SEP] " + chunk).tolist()
    if cfg_id == "cfg2":
        return ("[ARTICLE] " + art + " [SEP] [CHUNK] " + chunk).tolist()
    if cfg_id == "cfg3":
        return ("[ARTICLE] Article " + pred_art + ": " + art + " [SEP] [CHUNK] " + chunk).tolist()
    raise ValueError("cfg_id must be cfg1/cfg2/cfg3")


# ============================================================
# CLASSIFIER FACTORIES
# ============================================================
def make_classifier(clf_type, seed):
    if clf_type == "LR":
        return LogisticRegression(solver="lbfgs", max_iter=2000, C=1.0, random_state=seed)
    elif clf_type == "MLP1":
        return MLPClassifier(
            hidden_layer_sizes=MLP1_HIDDEN,
            activation="relu",
            solver="adam",
            alpha=MLP1_ALPHA,
            learning_rate_init=MLP1_LR_INIT,
            max_iter=MLP1_MAX_ITER,
            early_stopping=MLP_EARLY_STOPPING,
            n_iter_no_change=MLP_N_ITER_NO_CHANGE,
            random_state=seed,
            verbose=False
        )
    elif clf_type == "MLP2":
        return MLPClassifier(
            hidden_layer_sizes=MLP2_HIDDEN,
            activation="relu",
            solver="adam",
            alpha=MLP2_ALPHA,
            learning_rate_init=MLP2_LR_INIT,
            max_iter=MLP2_MAX_ITER,
            early_stopping=MLP_EARLY_STOPPING,
            n_iter_no_change=MLP_N_ITER_NO_CHANGE,
            random_state=seed,
            verbose=False
        )
    else:
        raise ValueError(f"Unknown classifier type: {clf_type}")


# ============================================================
# METRICS — VECTORIZED THRESHOLD SEARCH
# ============================================================
def compute_metrics(y_true, y_pred):
    return {
        "Accuracy": accuracy_score(y_true, y_pred),
        "Balanced Acc": balanced_accuracy_score(y_true, y_pred),
        "Balanced F1": f1_score(y_true, y_pred, average="macro", zero_division=0),
        "F1-oui": f1_score(y_true, y_pred, pos_label=1, zero_division=0),
        "F1-non": f1_score(y_true, y_pred, pos_label=0, zero_division=0),
        "MCC": matthews_corrcoef(y_true, y_pred) if len(np.unique(y_true)) > 1 else np.nan
    }


def best_threshold_vectorized(y_true, proba, thresholds=THR_GRID):
    """Vectorized threshold search — much faster than loop."""
    y_true = np.asarray(y_true)
    proba = np.asarray(proba)

    # Broadcast: (n_samples,) vs (n_thresholds,) -> (n_thresholds, n_samples)
    preds = (proba[np.newaxis, :] >= thresholds[:, np.newaxis]).astype(int)

    # Compute MCC for each threshold
    tp = ((preds == 1) & (y_true == 1)).sum(axis=1)
    tn = ((preds == 0) & (y_true == 0)).sum(axis=1)
    fp = ((preds == 1) & (y_true == 0)).sum(axis=1)
    fn = ((preds == 0) & (y_true == 1)).sum(axis=1)

    # MCC formula
    num = (tp * tn - fp * fn).astype(float)
    denom = np.sqrt((tp + fp) * (tp + fn) * (tn + fp) * (tn + fn).astype(float))
    denom = np.where(denom == 0, 1, denom)  # Avoid division by zero
    mcc = num / denom

    best_idx = np.argmax(mcc)
    return float(thresholds[best_idx]), float(mcc[best_idx])


# ============================================================
# GROUPED FOLDS
# ============================================================
def make_grouped_folds(df, n_splits, seed):
    rng = np.random.default_rng(seed)
    group_sizes = df.groupby("decision_id").size().to_dict()
    uniq_groups = np.array(list(group_sizes.keys()))
    uniq_groups = uniq_groups[rng.permutation(len(uniq_groups))]
    uniq_groups = sorted(uniq_groups, key=lambda g: group_sizes[g], reverse=True)

    fold_loads = np.zeros(n_splits, dtype=int)
    group_to_fold = {}
    for g in uniq_groups:
        f = int(fold_loads.argmin())
        group_to_fold[g] = f
        fold_loads[f] += int(group_sizes[g])

    out = df.copy()
    out["fold"] = out["decision_id"].map(group_to_fold).astype(int)
    return out, fold_loads


# ============================================================
# UNIFIED ENCODER — Works for both transformers and SentenceTransformers
# ============================================================
def encode_with_pooling_layer(texts, model_name, model_type, batch_size, max_len, pooling, layer_strategy, desc):
    """
    Unified encoder that handles both regular transformers and SentenceTransformers.
    For SentenceTransformers, we access the underlying transformer model directly.

    Returns: (embeddings, cache_path) - embeddings array and path where they're cached
    """
    from transformers import AutoTokenizer, AutoModel

    texts = ["" if t is None else str(t) for t in texts]
    texts_hash = _hash_texts(texts)

    # Check cache
    key = _cache_key(model_name, pooling, layer_strategy, max_len, texts_hash)
    cache_path = os.path.join(CACHE_DIR, f"emb_{key}.npy")
    if os.path.exists(cache_path):
        print(f"  [CACHE HIT] {os.path.basename(cache_path)}")
        return np.load(cache_path), cache_path

    print(f"  [CACHE MISS] Computing embeddings...")

    # Load model
    tok = AutoTokenizer.from_pretrained(model_name)
    model = AutoModel.from_pretrained(model_name).to(device)
    model.eval()

    def select_layers(hidden_states, strategy):
        if strategy == "last":
            return hidden_states[-1]
        if strategy.startswith("layer:"):
            idx = int(strategy.split(":")[1])
            return hidden_states[idx]
        if strategy.startswith("avg_last_k:"):
            k = int(strategy.split(":")[1])
            k = min(k, len(hidden_states))
            acc = None
            for h in hidden_states[-k:]:
                acc = h if acc is None else (acc + h)
            return acc / float(k)
        raise ValueError(f"Unknown layer_strategy: {strategy}")

    all_embs = []
    with torch.no_grad():
        for i in tqdm(range(0, len(texts), batch_size), desc=desc, leave=False):
            batch = texts[i:i + batch_size]
            enc = tok(batch, padding=True, truncation=True, max_length=int(max_len), return_tensors="pt")
            enc = {k: v.to(device) for k, v in enc.items()}

            out = model(**enc, output_hidden_states=True)
            hs = out.hidden_states
            attn = enc["attention_mask"]
            x = select_layers(hs, layer_strategy)

            if pooling == "cls":
                emb = x[:, 0, :]
            else:  # mean pooling
                mask = attn.unsqueeze(-1).float()
                emb = (x * mask).sum(dim=1) / mask.sum(dim=1).clamp_min(1.0)

            all_embs.append(emb.cpu().numpy())

            del enc, out, hs, attn, x, emb
            if i % (batch_size * 10) == 0:
                _clear_cuda()

    X = np.vstack(all_embs)
    np.save(cache_path, X)
    print(f"  [SAVED] {os.path.basename(cache_path)} — shape: {X.shape}")

    del model, tok
    _clear_cuda()
    return X, cache_path


# ============================================================
# PARALLEL OOF EVALUATION
# ============================================================
def train_fold(fold, X, y, folds, clf_type, seed):
    """Train on one fold and return OOF predictions for test indices."""
    tr = folds != fold
    te = folds == fold

    scaler = StandardScaler()
    Xtr = scaler.fit_transform(X[tr])
    Xte = scaler.transform(X[te])
    ytr = y[tr]

    clf = make_classifier(clf_type, seed)  # constant seed (as in reference script)
    clf.fit(Xtr, ytr)
    proba_te = clf.predict_proba(Xte)[:, 1]

    return np.where(te)[0], proba_te


def oof_proba_parallel(X, y, folds, n_splits, clf_type, seed, n_jobs=N_JOBS):
    """Parallel OOF computation across folds."""
    proba_oof = np.full(len(y), np.nan, dtype=float)

    results = Parallel(n_jobs=n_jobs, backend="loky")(
        delayed(train_fold)(fold, X, y, folds, clf_type, seed)
        for fold in range(n_splits)
    )

    for te_idx, proba_te in results:
        proba_oof[te_idx] = proba_te

    assert np.isfinite(proba_oof).all(), "NaN in OOF predictions!"
    return proba_oof


def evaluate_single_classifier(X, y, folds, n_splits, clf_type, seed, model_name, cfg_id, pooling, layer_strategy):
    """Evaluate one classifier configuration — called in parallel."""
    proba_oof = oof_proba_parallel(X, y, folds, n_splits, clf_type, seed, n_jobs=1)

    # Metrics at 0.5
    pred_05 = (proba_oof >= 0.5).astype(int)
    met_05 = compute_metrics(y, pred_05)

    # Best threshold (vectorized)
    best_thr, best_mcc = best_threshold_vectorized(y, proba_oof)
    pred_best = (proba_oof >= best_thr).astype(int)
    met_best = compute_metrics(y, pred_best)

    return {
        "clf_type": clf_type,
        "proba_oof": proba_oof,
        "best_thr": best_thr,
        "met_05": met_05,
        "met_best": met_best,
    }


def evaluate_all_classifiers_parallel(X, y, folds, n_splits, seed, model_name, cfg_id, pooling, layer_strategy):
    """Evaluate all classifiers in parallel for a given embedding."""
    results = Parallel(n_jobs=len(CLASSIFIERS), backend="loky")(
        delayed(evaluate_single_classifier)(
            X, y, folds, n_splits, clf_type, seed,
            model_name, cfg_id, pooling, layer_strategy
        )
        for clf_type in CLASSIFIERS
    )
    return {r["clf_type"]: r for r in results}


# ============================================================
# LOAD DATA
# ============================================================
print("=" * 80)
print("LOADING DATA")
print("=" * 80)

df0 = pd.read_excel(PATH)
cols_needed = ["decision_id", "chunk_id", "pred_art", "text", "article_text",
               "eval_A1", "eval_A2", "eval_A3"]
df0 = df0[[c for c in cols_needed if c in df0.columns]].copy()


def extract_oui_non(x):
    if pd.isna(x):
        return np.nan
    s = str(x).lower()
    has_oui = re.search(r"\boui\b", s) is not None
    has_non = re.search(r"\bnon\b", s) is not None
    if has_oui and not has_non:
        return "oui"
    if has_non and not has_oui:
        return "non"
    return np.nan


df0["a"] = df0["eval_A1"].apply(extract_oui_non)
df0["t"] = df0["eval_A2"].apply(extract_oui_non)
df0["s"] = df0["eval_A3"].apply(extract_oui_non)


def resolve_label(r):
    a, t, s = r["a"], r["t"], r["s"]
    if pd.notna(a) and pd.notna(t) and a == t:
        return a
    if pd.notna(a) and pd.notna(t) and a != t and pd.notna(s):
        return s
    return np.nan


df0["label_str"] = df0.apply(resolve_label, axis=1)
df0 = df0[df0["label_str"].isin(["oui", "non"])].copy()
df0["label"] = df0["label_str"].map({"oui": 1, "non": 0}).astype(int)

print(f"Total: {len(df0)} examples | oui={(df0.label==1).sum()} | non={(df0.label==0).sum()}")

df_cv, fold_loads = make_grouped_folds(df0, N_SPLITS, SEED)
print(f"Fold sizes: {fold_loads.tolist()}")

y_true = df_cv["label"].values
folds = df_cv["fold"].values


# ============================================================
# PRE-COMPUTE ALL TEXT CONFIGS
# ============================================================
print("\n" + "=" * 80)
print("PRE-COMPUTING TEXT CONFIGS")
print("=" * 80)

texts_cache = {}
for cfg_id in TEXT_CONFIGS:
    texts_cache[cfg_id] = make_text_inputs(df_cv, cfg_id)
    print(f"  {cfg_id}: {len(texts_cache[cfg_id])} texts")


# ============================================================
# MAIN GRID SEARCH — PARALLELIZED + SAVE BEST
# ============================================================
print("\n" + "=" * 80)
print(f"GRID SEARCH — {len(ALL_MODELS)} MODELS × {len(TEXT_CONFIGS)} CONFIGS × {len(POOLING_LAYER_CONFIGS)} POOLING × {len(CLASSIFIERS)} CLASSIFIERS")
print(f"Total combinations: {len(ALL_MODELS) * len(TEXT_CONFIGS) * len(POOLING_LAYER_CONFIGS) * len(CLASSIFIERS)}")
print("=" * 80)

all_results = []

# To track the best per model (with embeddings path)
best_per_model_tracking = {}

for model_name, model_cfg in ALL_MODELS.items():
    hf_model = model_cfg["hf_model"]
    batch_size = model_cfg["batch_size"]
    model_type = model_cfg["type"]

    print(f"\n{'='*80}")
    print(f"MODEL: {model_name} ({hf_model})")
    print(f"{'='*80}")

    best_for_model = {"MCC(best)": -1e9}

    for cfg_id in TEXT_CONFIGS:
        texts = texts_cache[cfg_id]

        for pcfg in POOLING_LAYER_CONFIGS:
            pooling = pcfg["pooling"]
            layer_strategy = pcfg["layer_strategy"]

            run_id = f"{model_name}|{cfg_id}|{pooling}|{layer_strategy}"
            print(f"\n--- {run_id} ---")

            # Get embeddings (GPU, sequential) - now also returns the cache_path
            X, emb_cache_path = encode_with_pooling_layer(
                texts=texts,
                model_name=hf_model,
                model_type=model_type,
                batch_size=batch_size,
                max_len=MAX_LEN,
                pooling=pooling,
                layer_strategy=layer_strategy,
                desc=run_id
            )

            # Evaluate all classifiers in parallel (CPU)
            print(f"  Evaluating {len(CLASSIFIERS)} classifiers in parallel...")
            clf_results = evaluate_all_classifiers_parallel(
                X, y_true, folds, N_SPLITS, SEED,
                model_name, cfg_id, pooling, layer_strategy
            )

            # Collect results
            for clf_type, res in clf_results.items():
                met_05 = res["met_05"]
                met_best = res["met_best"]
                best_thr = res["best_thr"]
                proba_oof = res["proba_oof"]

                row = {
                    "Model": model_name,
                    "HF_model": hf_model,
                    "TextCfg": cfg_id,
                    "pooling": pooling,
                    "layer_strategy": layer_strategy,
                    "Classifier": clf_type,

                    "Thr(best)": best_thr,
                    "MCC(best)": met_best["MCC"],
                    "Acc(best)": met_best["Accuracy"],
                    "BAcc(best)": met_best["Balanced Acc"],
                    "BF1(best)": met_best["Balanced F1"],
                    "F1-oui(best)": met_best["F1-oui"],
                    "F1-non(best)": met_best["F1-non"],

                    "MCC@0.5": met_05["MCC"],
                    "Acc@0.5": met_05["Accuracy"],
                    "BAcc@0.5": met_05["Balanced Acc"],
                    "ΔMCC": met_best["MCC"] - met_05["MCC"],
                }

                # Save ALL proba (as before)
                proba_fname = f"proba_oof_{model_name}_{cfg_id}_{pooling}_{layer_strategy.replace(':', '_')}_{clf_type}.npy"
                proba_path = os.path.join(OUTPUT_PATH, proba_fname)
                np.save(proba_path, proba_oof)
                row["proba_oof_path"] = proba_path
                row["emb_cache_path"] = emb_cache_path  # Keep track of the embeddings path

                all_results.append(row)

                # Track the best for this model
                if row["MCC(best)"] > best_for_model.get("MCC(best)", -1e9):
                    best_for_model = row.copy()
                    best_for_model["_proba_oof"] = proba_oof  # Keep the data in memory
                    best_for_model["_emb_cache_path"] = emb_cache_path

                print(f"    {clf_type}: MCC@0.5={met_05['MCC']:.4f} | thr={best_thr:.3f} MCC={met_best['MCC']:.4f}")

    # Save the best for this model
    best_per_model_tracking[model_name] = best_for_model

    print(f"\n✓ BEST for {model_name}:")
    print(f"  {best_for_model['TextCfg']}, {best_for_model['pooling']}, {best_for_model['layer_strategy']}, {best_for_model['Classifier']}")
    print(f"  MCC(best): {best_for_model['MCC(best)']:.4f} @ thr={best_for_model['Thr(best)']:.3f}")


# ============================================================
# SAVE BEST EMBEDDINGS & OOF PER MODEL
# ============================================================
print("\n" + "=" * 80)
print("SAVING BEST EMBEDDINGS & OOF PER MODEL")
print("=" * 80)

for model_name, best_info in best_per_model_tracking.items():
    print(f"\n{model_name}:")

    # Standardized file name
    safe_layer = best_info['layer_strategy'].replace(':', '_')
    base_name = f"{model_name}_BEST_{best_info['TextCfg']}_{best_info['pooling']}_{safe_layer}_{best_info['Classifier']}"

    # 1. Save OOF proba
    oof_path = os.path.join(BEST_OUTPUT_PATH, f"proba_oof_{base_name}.npy")
    np.save(oof_path, best_info["_proba_oof"])
    print(f"  ✓ OOF proba: {oof_path}")

    # 2. Copy/save embeddings
    emb_src = best_info["_emb_cache_path"]
    emb_dst = os.path.join(BEST_OUTPUT_PATH, f"embeddings_{base_name}.npy")

    # Load and save (or copy)
    if os.path.exists(emb_src):
        import shutil
        shutil.copy(emb_src, emb_dst)
        print(f"  ✓ Embeddings: {emb_dst}")
    else:
        print(f"  ⚠ Embeddings source not found: {emb_src}")

    # 3. Save config JSON
    config = {
        "model_name": model_name,
        "hf_model": best_info["HF_model"],
        "text_cfg": best_info["TextCfg"],
        "pooling": best_info["pooling"],
        "layer_strategy": best_info["layer_strategy"],
        "classifier": best_info["Classifier"],
        "threshold": best_info["Thr(best)"],
        "MCC_best": best_info["MCC(best)"],
        "MCC_05": best_info["MCC@0.5"],
        "oof_path": oof_path,
        "emb_path": emb_dst,
    }
    config_path = os.path.join(BEST_OUTPUT_PATH, f"config_{base_name}.json")
    with open(config_path, "w") as f:
        json.dump(config, f, indent=2)
    print(f"  ✓ Config: {config_path}")


# ============================================================
# SAVE ALL RESULTS
# ============================================================
print("\n" + "=" * 80)
print("SAVING ALL RESULTS")
print("=" * 80)

df_all = pd.DataFrame(all_results)
df_all = df_all.sort_values("MCC(best)", ascending=False).reset_index(drop=True)

# Excel with sheets per model
out_xlsx = os.path.join(OUTPUT_PATH, "GRID_SEARCH_ALL_RESULTS.xlsx")
with pd.ExcelWriter(out_xlsx) as writer:
    df_all.to_excel(writer, index=False, sheet_name="All_Results")
    for model_name in df_all["Model"].unique():
        df_model = df_all[df_all["Model"] == model_name].copy()
        df_model.to_excel(writer, index=False, sheet_name=model_name[:31])
print(f"✅ Saved: {out_xlsx}")

# CSV
out_csv = os.path.join(OUTPUT_PATH, "GRID_SEARCH_ALL_RESULTS.csv")
df_all.to_csv(out_csv, index=False)
print(f"✅ Saved: {out_csv}")

# Best per model summary
best_per_model = []
for model_name in df_all["Model"].unique():
    df_model = df_all[df_all["Model"] == model_name]
    best_row = df_model.iloc[0].to_dict()
    best_per_model.append(best_row)

df_best = pd.DataFrame(best_per_model)
out_best = os.path.join(OUTPUT_PATH, "BEST_PER_MODEL.xlsx")
df_best.to_excel(out_best, index=False)
print(f"✅ Saved: {out_best}")

# JSON config global
best_configs = {}
for row in best_per_model:
    safe_layer = row['layer_strategy'].replace(':', '_')
    base_name = f"{row['Model']}_BEST_{row['TextCfg']}_{row['pooling']}_{safe_layer}_{row['Classifier']}"

    best_configs[row["Model"]] = {
        "cfg": row["TextCfg"],
        "classifier": row["Classifier"],
        "pooling": row["pooling"],
        "layer_strategy": row["layer_strategy"],
        "threshold": row["Thr(best)"],
        "MCC": row["MCC(best)"],
        "hf_model": row["HF_model"],
        "proba_path": os.path.join(BEST_OUTPUT_PATH, f"proba_oof_{base_name}.npy"),
        "emb_path": os.path.join(BEST_OUTPUT_PATH, f"embeddings_{base_name}.npy"),
    }

out_json = os.path.join(OUTPUT_PATH, "BEST_CONFIGS.json")
with open(out_json, "w") as f:
    json.dump(best_configs, f, indent=2)
print(f"✅ Saved: {out_json}")


# ============================================================
# SUMMARY
# ============================================================
print("\n" + "=" * 80)
print("SUMMARY — BEST CONFIG PER MODEL")
print("=" * 80)

summary_cols = ["Model", "TextCfg", "pooling", "layer_strategy", "Classifier", "Thr(best)", "MCC(best)", "MCC@0.5"]
print(df_best[summary_cols].to_string(index=False))

print("\n" + "=" * 80)
print("TOP 15 OVERALL")
print("=" * 80)

print(df_all.head(15)[summary_cols].to_string(index=False))

print(f"\n✅ DONE — All outputs in: {OUTPUT_PATH}")
print(f"✅ Best models saved in: {BEST_OUTPUT_PATH}")
print(f"   Total configurations tested: {len(df_all)}")

# TF-IDF

In [ ]:
# =============================================================================
# DETAILED ANALYSIS OF THE BEST ENSEMBLE
# Run after the ensemble search step
# =============================================================================

import numpy as np
import pandas as pd
from sklearn.metrics import (
    confusion_matrix, accuracy_score, balanced_accuracy_score,
    f1_score, precision_score, recall_score, matthews_corrcoef
)
import matplotlib.pyplot as plt
import seaborn as sns

# Load the best ensemble's data
BASE_PATH = "artifacts/oof_proba_final/best_models"  # not shipped — see DATA.md
oof_df = pd.read_csv(f"{BASE_PATH}/oof_BEST_ENSEMBLE.csv")

y_true = oof_df["label"].values
y_pred = oof_df["pred_best_thr"].values
p_best = oof_df["proba_ensemble"].values

# =============================================================================
# CONFUSION MATRIX
# =============================================================================
cm = confusion_matrix(y_true, y_pred)
tn, fp, fn, tp = cm.ravel()

print("=" * 70)
print("CONFUSION MATRIX - BEST ENSEMBLE")
print("=" * 70)
print(f"\n{'':15} Predicted NON    Predicted OUI")
print(f"{'Actual NON':15} {tn:8}      {fp:8}")
print(f"{'Actual OUI':15} {fn:8}      {tp:8}")

# =============================================================================
# GLOBAL METRICS
# =============================================================================
print("\n" + "=" * 70)
print("GLOBAL METRICS")
print("=" * 70)

acc = accuracy_score(y_true, y_pred)
bacc = balanced_accuracy_score(y_true, y_pred)
mcc = matthews_corrcoef(y_true, y_pred)
f1_macro = f1_score(y_true, y_pred, average='macro')
f1_weighted = f1_score(y_true, y_pred, average='weighted')

print(f"\nAccuracy:              {acc:.4f}  ({acc*100:.2f}%)")
print(f"Balanced Accuracy:     {bacc:.4f}  ({bacc*100:.2f}%)")
print(f"MCC (Matthews):        {mcc:.4f}")
print(f"F1 Macro:              {f1_macro:.4f}")
print(f"F1 Weighted:           {f1_weighted:.4f}")

# =============================================================================
# PER-CLASS METRICS
# =============================================================================
print("\n" + "=" * 70)
print("PER-CLASS METRICS")
print("=" * 70)

print(f"\n{'Class':<12} {'Precision':>10} {'Recall':>10} {'F1-Score':>10} {'Support':>10}")
print("-" * 54)

# Class NON (0)
prec_non = precision_score(y_true, y_pred, pos_label=0)
rec_non = recall_score(y_true, y_pred, pos_label=0)
f1_non = f1_score(y_true, y_pred, pos_label=0)
support_non = np.sum(y_true == 0)
print(f"{'NON (0)':<12} {prec_non:>10.4f} {rec_non:>10.4f} {f1_non:>10.4f} {support_non:>10}")

# Class OUI (1)
prec_oui = precision_score(y_true, y_pred, pos_label=1)
rec_oui = recall_score(y_true, y_pred, pos_label=1)
f1_oui = f1_score(y_true, y_pred, pos_label=1)
support_oui = np.sum(y_true == 1)
print(f"{'OUI (1)':<12} {prec_oui:>10.4f} {rec_oui:>10.4f} {f1_oui:>10.4f} {support_oui:>10}")

print("-" * 54)
print(f"{'Macro avg':<12} {(prec_non+prec_oui)/2:>10.4f} {(rec_non+rec_oui)/2:>10.4f} {f1_macro:>10.4f} {len(y_true):>10}")

# =============================================================================
# DETAILED STATISTICS
# =============================================================================
print("\n" + "=" * 70)
print("DETAILED STATISTICS")
print("=" * 70)

print(f"\n📊 Confusion Matrix Values:")
print(f"   True Positives (TP):   {tp:5d}  (true OUI)")
print(f"   True Negatives (TN):   {tn:5d}  (true NON)")
print(f"   False Positives (FP):  {fp:5d}  (false OUI)")
print(f"   False Negatives (FN):  {fn:5d}  (false NON)")

print(f"\n📈 Rates:")
print(f"   Sensitivity (TPR/Recall OUI): {tp/(tp+fn):.4f}  ({tp/(tp+fn)*100:.1f}%)")
print(f"   Specificity (TNR/Recall NON): {tn/(tn+fp):.4f}  ({tn/(tn+fp)*100:.1f}%)")
print(f"   Precision OUI (PPV):          {tp/(tp+fp):.4f}  ({tp/(tp+fp)*100:.1f}%)")
print(f"   Precision NON (NPV):          {tn/(tn+fn):.4f}  ({tn/(tn+fn)*100:.1f}%)")

print(f"\n📋 Distribution:")
print(f"   Total samples:     {len(y_true)}")
print(f"   Class NON (0):     {support_non} ({support_non/len(y_true)*100:.1f}%)")
print(f"   Class OUI (1):     {support_oui} ({support_oui/len(y_true)*100:.1f}%)")
print(f"   Imbalance ratio:   1:{support_non/support_oui:.2f}" if support_oui < support_non else f"   Imbalance ratio:   {support_oui/support_non:.2f}:1")

# =============================================================================
# VISUALIZATION
# =============================================================================
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Absolute confusion matrix
ax1 = axes[0]
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=ax1,
            xticklabels=['NON', 'OUI'], yticklabels=['NON', 'OUI'],
            annot_kws={'size': 18, 'weight': 'bold'})
ax1.set_xlabel('Predicted', fontsize=12)
ax1.set_ylabel('Actual', fontsize=12)
ax1.set_title(f'Confusion Matrix\nBest Ensemble (MCC={mcc:.4f})', fontsize=14)

# Normalized confusion matrix
ax2 = axes[1]
cm_norm = cm.astype('float') / cm.sum(axis=1)[:, np.newaxis]
sns.heatmap(cm_norm, annot=True, fmt='.1%', cmap='Blues', ax=ax2,
            xticklabels=['NON', 'OUI'], yticklabels=['NON', 'OUI'],
            annot_kws={'size': 16})
ax2.set_xlabel('Predicted', fontsize=12)
ax2.set_ylabel('Actual', fontsize=12)
ax2.set_title('Normalized Matrix (per row)', fontsize=14)

plt.tight_layout()
plt.savefig(f'{BASE_PATH}/confusion_matrix_best_ensemble.png', dpi=150, bbox_inches='tight')
plt.show()

print("\n" + "=" * 70)
print(f"✓ Figure saved: {BASE_PATH}/confusion_matrix_best_ensemble.png")
print("=" * 70)

# =============================================================================
# LATEX SUMMARY (for the paper)
# =============================================================================
print("\n" + "=" * 70)
print("LATEX TABLE")
print("=" * 70)
print("""
\\begin{table}[h]
\\centering
\\begin{tabular}{lcc}
\\toprule
\\textbf{Metric} & \\textbf{Value} \\\\
\\midrule""")
print(f"Accuracy & {acc:.4f} \\\\")
print(f"Balanced Accuracy & {bacc:.4f} \\\\")
print(f"MCC & {mcc:.4f} \\\\")
print(f"F1 Macro & {f1_macro:.4f} \\\\")
print(f"F1 (NON) & {f1_non:.4f} \\\\")
print(f"F1 (OUI) & {f1_oui:.4f} \\\\")
print("""\\bottomrule
\\end{tabular}
\\caption{Performance of the best ensemble (6 models, weighted average)}
\\label{tab:best_ensemble_metrics}
\\end{table}
""")

run analyse predictions FIRST

In [ ]:
# ============================================================
# MCC ANALYSIS BY CASE TYPE (CONSENSUS vs ARBITRATION)
# ============================================================
print("\n" + "=" * 80)
print("MCC ANALYSIS: CONSENSUS (agree) vs ARBITRATION (disagree)")
print("=" * 80)

print("""
Reminder on the composition of the Ground Truth:
- CONSENSUS (agree):  The 2 annotators agree → their answer = ground truth
- ARBITRATION (disagree): The 2 annotators diverge → the arbitrator decides

Key question: Is the model good only on the "easy" cases (consensus),
or does it also perform on the "hard" cases (arbitration)?
""")

from sklearn.metrics import matthews_corrcoef, accuracy_score, balanced_accuracy_score, f1_score

# ============================================================
# Compute metrics per subgroup
# ============================================================

results_by_case = []

for model_name, pred_df in all_predictions.items():

    for case in ["agree", "disagree"]:
        subset = pred_df[pred_df["case"] == case]

        if len(subset) < 10:
            continue

        y_true_sub = subset["y_true"].values
        y_pred_sub = subset["y_pred"].values
        p_oui_sub = subset["p_oui"].values

        # Metrics with the model's default threshold
        mcc = matthews_corrcoef(y_true_sub, y_pred_sub)
        acc = accuracy_score(y_true_sub, y_pred_sub)
        bacc = balanced_accuracy_score(y_true_sub, y_pred_sub)
        f1 = f1_score(y_true_sub, y_pred_sub, average="macro", zero_division=0)

        # Label distribution in this subgroup
        n_oui = (y_true_sub == 1).sum()
        n_non = (y_true_sub == 0).sum()
        pct_oui = 100 * n_oui / len(y_true_sub)

        results_by_case.append({
            "model": model_name,
            "case": case,
            "n_total": len(subset),
            "n_oui": n_oui,
            "n_non": n_non,
            "pct_oui": pct_oui,
            "mcc": mcc,
            "acc": acc,
            "bacc": bacc,
            "f1": f1,
        })

results_case_df = pd.DataFrame(results_by_case)

# ============================================================
# Main display: MCC per case
# ============================================================
print("\n" + "=" * 80)
print("MCC PER SUBGROUP")
print("=" * 80)

print(f"\n{'Model':20s} | {'CONSENSUS (agree)':^25s} | {'ARBITRATION (disagree)':^25s} | {'Δ':^8s}")
print(f"{'':20s} | {'n':^6s} {'%oui':^6s} {'MCC':^10s} | {'n':^6s} {'%oui':^6s} {'MCC':^10s} |")
print("-" * 95)

for model_name in all_predictions.keys():
    agree_row = results_case_df[(results_case_df["model"] == model_name) &
                                 (results_case_df["case"] == "agree")]
    disagree_row = results_case_df[(results_case_df["model"] == model_name) &
                                    (results_case_df["case"] == "disagree")]

    if len(agree_row) == 0 or len(disagree_row) == 0:
        continue

    agree_row = agree_row.iloc[0]
    disagree_row = disagree_row.iloc[0]

    delta = disagree_row["mcc"] - agree_row["mcc"]

    print(f"{model_name:20s} | {agree_row['n_total']:5.0f} {agree_row['pct_oui']:5.1f}% {agree_row['mcc']:+10.4f} | "
          f"{disagree_row['n_total']:5.0f} {disagree_row['pct_oui']:5.1f}% {disagree_row['mcc']:+10.4f} | {delta:+8.4f}")

# ============================================================
# Statistical summary
# ============================================================
print("\n" + "=" * 80)
print("STATISTICAL SUMMARY")
print("=" * 80)

agree_mccs = results_case_df[results_case_df["case"] == "agree"]["mcc"]
disagree_mccs = results_case_df[results_case_df["case"] == "disagree"]["mcc"]

print(f"""
Mean MCC on CONSENSUS (agree):     {agree_mccs.mean():+.4f}  (min: {agree_mccs.min():+.4f}, max: {agree_mccs.max():+.4f})
Mean MCC on ARBITRATION (disagree):  {disagree_mccs.mean():+.4f}  (min: {disagree_mccs.min():+.4f}, max: {disagree_mccs.max():+.4f})

Mean difference (arbitration - consensus): {(disagree_mccs.mean() - agree_mccs.mean()):+.4f}
""")

# ============================================================
# Interpretation
# ============================================================
print("=" * 80)
print("INTERPRETATION")
print("=" * 80)

mean_agree = agree_mccs.mean()
mean_disagree = disagree_mccs.mean()

if mean_disagree > 0.2:
    interpretation_disagree = "✓ GOOD - The model learned something from the arbitrator's logic"
elif mean_disagree > 0.1:
    interpretation_disagree = "~ MEDIUM - The model captures some signal, but remains fragile"
elif mean_disagree > 0:
    interpretation_disagree = "⚠ WEAK - The model does barely better than chance on hard cases"
else:
    interpretation_disagree = "✗ FAILURE - The model does not understand arbitrated cases"

if mean_agree > 0.6:
    interpretation_agree = "✓ EXCELLENT - Perfect mastery of consensus cases"
elif mean_agree > 0.4:
    interpretation_agree = "✓ GOOD - Good performance on clear cases"
else:
    interpretation_agree = "⚠ Problem - Insufficient performance even on easy cases"

print(f"""
On CONSENSUS (agree) cases:
  Mean MCC = {mean_agree:.4f}
  → {interpretation_agree}

On ARBITRATION (disagree) cases:
  Mean MCC = {mean_disagree:.4f}
  → {interpretation_disagree}
""")

# Performance ratio
ratio = mean_disagree / mean_agree if mean_agree > 0 else 0
print(f"Retention ratio (MCC_arbitration / MCC_consensus): {ratio:.2%}")

if ratio > 0.7:
    print("→ The model maintains its performance well on hard cases")
elif ratio > 0.4:
    print("→ Moderate degradation on hard cases")
else:
    print("→ Strong degradation: the model is mostly good on easy cases")

# ============================================================
# Visualization
# ============================================================
print("\n" + "-" * 60)
print("Generating the plot...")

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Plot 1: MCC per model and case
ax = axes[0]
models = list(all_predictions.keys())
x = np.arange(len(models))
width = 0.35

agree_vals = [results_case_df[(results_case_df["model"] == m) & (results_case_df["case"] == "agree")]["mcc"].values[0]
              for m in models]
disagree_vals = [results_case_df[(results_case_df["model"] == m) & (results_case_df["case"] == "disagree")]["mcc"].values[0]
                 for m in models]

bars1 = ax.bar(x - width/2, agree_vals, width, label='Consensus (agree)', color='#3498db', alpha=0.8)
bars2 = ax.bar(x + width/2, disagree_vals, width, label='Arbitration (disagree)', color='#e74c3c', alpha=0.8)

# Reference line
ax.axhline(y=0, color='black', linestyle='-', linewidth=0.5)
ax.axhline(y=0.2, color='green', linestyle='--', linewidth=1, alpha=0.5, label='Acceptable threshold (0.2)')

ax.set_ylabel("MCC")
ax.set_xlabel("Model")
ax.set_title("MCC by case type\n(Consensus = easy cases, Arbitration = hard cases)")
ax.set_xticks(x)
ax.set_xticklabels(models, rotation=45, ha='right', fontsize=9)
ax.legend(loc='upper right', fontsize=8)
ax.grid(True, alpha=0.3, axis='y')

# Add the values on the bars
for bar, val in zip(bars1, agree_vals):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.02, f'{val:.2f}',
            ha='center', va='bottom', fontsize=8)
for bar, val in zip(bars2, disagree_vals):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.02, f'{val:.2f}',
            ha='center', va='bottom', fontsize=8)

# Plot 2: Scatter MCC agree vs disagree
ax = axes[1]

for i, model_name in enumerate(models):
    agree_mcc = results_case_df[(results_case_df["model"] == model_name) &
                                 (results_case_df["case"] == "agree")]["mcc"].values[0]
    disagree_mcc = results_case_df[(results_case_df["model"] == model_name) &
                                    (results_case_df["case"] == "disagree")]["mcc"].values[0]

    color = '#2ecc71' if model_name == "BEST_ENSEMBLE" else '#3498db'
    size = 150 if model_name == "BEST_ENSEMBLE" else 80

    ax.scatter(agree_mcc, disagree_mcc, s=size, c=color, alpha=0.7, edgecolors='black', linewidths=1)
    ax.annotate(model_name, (agree_mcc, disagree_mcc), textcoords="offset points",
                xytext=(5, 5), ha='left', fontsize=8)

# Parity line (same perf on both)
ax.plot([0, 0.8], [0, 0.8], 'k--', alpha=0.3, label='Parity')

# Zones
ax.axhline(y=0.2, color='green', linestyle=':', alpha=0.5)
ax.axvline(x=0.4, color='blue', linestyle=':', alpha=0.5)

ax.set_xlabel("MCC on Consensus (agree)")
ax.set_ylabel("MCC on Arbitration (disagree)")
ax.set_title("Consensus vs Arbitration Performance\n(Ideal = close to the diagonal)")
ax.set_xlim(-0.1, 0.8)
ax.set_ylim(-0.1, 0.8)
ax.grid(True, alpha=0.3)
ax.set_aspect('equal')

plt.tight_layout()
plot_and_save(fig, os.path.join(ANALYSIS_OUTPUT, "mcc_consensus_vs_arbitrage.png"))

# ============================================================
# Detailed table with all metrics
# ============================================================
print("\n" + "=" * 80)
print("DETAILED TABLE (ALL METRICS)")
print("=" * 80)

print(f"\n{'Model':18s} | {'Case':^10s} | {'n':^5s} | {'%oui':^6s} | {'MCC':^8s} | {'Acc':^7s} | {'BAcc':^7s} | {'F1':^7s}")
print("-" * 90)

for _, row in results_case_df.sort_values(["model", "case"]).iterrows():
    print(f"{row['model']:18s} | {row['case']:^10s} | {row['n_total']:5.0f} | {row['pct_oui']:5.1f}% | "
          f"{row['mcc']:+8.4f} | {row['acc']:7.4f} | {row['bacc']:7.4f} | {row['f1']:7.4f}")

# ============================================================
# Export
# ============================================================
output_csv = os.path.join(ANALYSIS_OUTPUT, "mcc_by_case_type.csv")
results_case_df.to_csv(output_csv, index=False)
print(f"\n✓ Results exported: {output_csv}")

# Best Model with Lawma

In [ ]:
# ============================================================
# NESTED CV ENSEMBLE SEARCH - WITH LAWMA
# ============================================================

import os
import json
import shutil
import numpy as np
import pandas as pd
from itertools import combinations
import warnings
warnings.filterwarnings("ignore")

from sklearn.metrics import (
    accuracy_score, balanced_accuracy_score, f1_score,
    precision_score, recall_score, confusion_matrix,
    classification_report, matthews_corrcoef
)
from sklearn.linear_model import LogisticRegression
from sklearn.neural_network import MLPClassifier
from sklearn.preprocessing import StandardScaler
from joblib import Parallel, delayed
import re

# ============================================================
# CONFIG
# ============================================================
BASE_PATH = "artifacts"  # not shipped — see DATA.md
OUTPUT_PATH = os.path.join(BASE_PATH, "oof_proba_final")
BEST_OUTPUT_PATH = os.path.join(OUTPUT_PATH, "best_models")
ENSEMBLE_OUTPUT = os.path.join(OUTPUT_PATH, "ensemble_search_nested_cv_with_lawma")

os.makedirs(OUTPUT_PATH, exist_ok=True)
os.makedirs(BEST_OUTPUT_PATH, exist_ok=True)
os.makedirs(ENSEMBLE_OUTPUT, exist_ok=True)

SEED = 42
N_SPLITS = 5
N_JOBS = -1
THR_GRID = np.linspace(0.15, 0.85, 500)

np.random.seed(SEED)

# ============================================================
# MODEL DEFINITIONS — 9 models (8 original + Lawma)
# ============================================================
MODELS = {
    "CamemBERT": "proba_oof_CamemBERT_BEST_cfg3_mean_avg_last_k_4_MLP2.npy",
    "CamemBERTav2": "proba_oof_CamemBERTav2_BEST_cfg1_mean_avg_last_k_4_MLP1.npy",
    "JuriBERT": "proba_oof_JuriBERT-base_BEST_cfg1_cls_last_LR.npy",
    "LLaMA": "proba_oof_LLaMA_BEST_cfg2_mean_layer_-2_MLP1.npy",
    "SAUL": "proba_oof_SAUL_BEST_cfg3_mean_last_k_4_LR.npy",
    "ST-MiniLM": "proba_oof_ST-MiniLM_BEST_cfg3_mean_layer_-2_MLP2.npy",
    "ST-MPNet": "proba_oof_ST-MPNet_BEST_cfg1_mean_avg_last_k_4_MLP2.npy",
}
TFIDF_CSV = "tfidf_oof_probas.csv"

# Lawma: best config = MLP1, cfg1, mean, last (MCC=0.4576)
# File saved in the lawma_mlp1_mlp2 run
LAWMA_PROBA_PATH = os.path.join(
    BASE_PATH, "outputs", "outputs_lawma_mlp1_mlp2_3cfg",
    "proba_oof_Lawma-8B__cfg1__mean__last__mlp1.npy"
)
# Fallback: also try the lawma_vs_llama run
LAWMA_PROBA_PATH_ALT = os.path.join(
    BASE_PATH, "outputs", "outputs_lawma_vs_llama_full_comparison",
    "proba_oof_Lawma-8B__cfg1__mean__last__mlp1.npy"
)

# ============================================================
# LOAD DATA
# ============================================================
print("=" * 80)
print("NESTED CV ENSEMBLE SEARCH - WITH LAWMA (9 MODELS)")
print("=" * 80)

def load_labels():
    EXCEL_PATH = "DATA/outputs/benchmark.csv"
    df0 = pd.read_excel(EXCEL_PATH)
    cols = ["decision_id", "eval_A1", "eval_A2", "eval_A3"]
    df0 = df0[cols].copy()

    def extract(x):
        if pd.isna(x): return np.nan
        s = str(x).lower()
        if re.search(r"\boui\b", s) and not re.search(r"\bnon\b", s): return "oui"
        if re.search(r"\bnon\b", s) and not re.search(r"\boui\b", s): return "non"
        return np.nan

    df0["a"] = df0["eval_A1"].apply(extract)
    df0["t"] = df0["eval_A2"].apply(extract)
    df0["s"] = df0["eval_A3"].apply(extract)

    def resolve(row):
        a, t, s = row["a"], row["t"], row["s"]
        if pd.notna(a) and pd.notna(t) and a == t: return a
        if pd.notna(a) and pd.notna(t) and a != t and pd.notna(s): return s
        return np.nan

    df0["label_str"] = df0.apply(resolve, axis=1)
    df0 = df0[df0["label_str"].isin(["oui", "non"])].copy()
    df0["label"] = df0["label_str"].map({"oui": 1, "non": 0}).astype(int)

    rng = np.random.default_rng(SEED)
    groups = df0.groupby("decision_id").size().to_dict()
    uniq = list(groups.keys())
    rng.shuffle(uniq)
    uniq = sorted(uniq, key=lambda g: groups[g], reverse=True)
    loads = np.zeros(N_SPLITS, dtype=int)
    g2f = {}
    for g in uniq:
        f = int(loads.argmin())
        g2f[g] = f
        loads[f] += groups[g]
    df0["fold"] = df0["decision_id"].map(g2f).astype(int)

    return df0["label"].values.astype(int), df0["fold"].values

y_true, folds = load_labels()
print(f"  {len(y_true)} samples, {N_SPLITS} folds")

# Load probas — 8 original models
probas = {}
for name, fname in MODELS.items():
    path = os.path.join(OUTPUT_PATH, fname)
    if os.path.exists(path):
        probas[name] = np.load(path)
        print(f"  ✓ {name}: loaded")
    else:
        print(f"  ✗ {name}: NOT FOUND at {path}")

tfidf_path = os.path.join(OUTPUT_PATH, TFIDF_CSV)
if os.path.exists(tfidf_path):
    probas["TF-IDF"] = pd.read_csv(tfidf_path)["proba_oui"].values
    print(f"  ✓ TF-IDF: loaded")

# Load Lawma
lawma_loaded = False
for lpath in [LAWMA_PROBA_PATH, LAWMA_PROBA_PATH_ALT]:
    if os.path.exists(lpath):
        probas["Lawma"] = np.load(lpath)
        print(f"  ✓ Lawma: loaded from {lpath}")
        lawma_loaded = True
        break

if not lawma_loaded:
    # Search in all outputs subfolders
    print("  ⚠ Lawma proba not found at expected paths, searching...")
    for root, dirs, files in os.walk(os.path.join(BASE_PATH, "outputs")):
        for f in files:
            if "Lawma" in f and "cfg1" in f and "last" in f and "mlp1" in f and f.endswith(".npy"):
                full = os.path.join(root, f)
                probas["Lawma"] = np.load(full)
                print(f"  ✓ Lawma: found at {full}")
                lawma_loaded = True
                break
        if lawma_loaded:
            break

if not lawma_loaded:
    print("  ✗✗ LAWMA NOT FOUND — will run without it")

model_names = list(probas.keys())
n_models = len(model_names)
print(f"\n{n_models} models loaded: {model_names}")

# Size check
for name in model_names:
    assert len(probas[name]) == len(y_true), f"{name}: {len(probas[name])} != {len(y_true)}"

PROBA_MATRIX = np.column_stack([probas[m] for m in model_names])
MODEL_IDX = {m: i for i, m in enumerate(model_names)}

# ============================================================
# METRICS FUNCTIONS
# ============================================================
def fast_mcc(y_true, y_pred):
    tp = np.sum((y_true == 1) & (y_pred == 1))
    tn = np.sum((y_true == 0) & (y_pred == 0))
    fp = np.sum((y_true == 0) & (y_pred == 1))
    fn = np.sum((y_true == 1) & (y_pred == 0))
    num = float(tp * tn - fp * fn)
    den = np.sqrt(float((tp + fp) * (tp + fn) * (tn + fp) * (tn + fn)))
    return num / den if den > 0 else 0.0

def best_threshold_search(y_true, p_proba):
    best_thr, best_mcc = 0.5, -1
    for thr in THR_GRID:
        y_pred = (p_proba >= thr).astype(int)
        mcc = fast_mcc(y_true, y_pred)
        if mcc > best_mcc:
            best_mcc = mcc
            best_thr = thr
    return best_thr, best_mcc

def combo_to_seed(combo):
    s = "+".join(sorted(combo))
    return sum(ord(c) * (i + 1) for i, c in enumerate(s)) % (2**31)

# ============================================================
# NESTED CV FUNCTIONS
# ============================================================
def optimize_weights_on_train(combo, y_train, P_train, idxs, fold_seed):
    n = len(combo)
    best = {"mcc_train": -1, "weights": None, "thr": 0.5}
    rng = np.random.default_rng(fold_seed)

    if n == 2:
        for w1 in np.linspace(0, 1, 21):
            weights = (w1, 1 - w1)
            p_mix = sum(w * P_train[:, idx] for w, idx in zip(weights, idxs))
            thr, mcc = best_threshold_search(y_train, p_mix)
            if mcc > best["mcc_train"]:
                best = {"weights": weights, "thr": thr, "mcc_train": mcc}

    elif n == 3:
        for w1 in np.linspace(0, 1, 11):
            for w2 in np.linspace(0, 1 - w1, 11):
                w3 = 1 - w1 - w2
                if w3 < -1e-6: continue
                weights = (w1, w2, w3)
                p_mix = sum(w * P_train[:, idx] for w, idx in zip(weights, idxs))
                thr, mcc = best_threshold_search(y_train, p_mix)
                if mcc > best["mcc_train"]:
                    best = {"weights": weights, "thr": thr, "mcc_train": mcc}
    else:
        for _ in range(1000):
            weights = tuple(rng.dirichlet(np.ones(n)))
            p_mix = sum(w * P_train[:, idx] for w, idx in zip(weights, idxs))
            thr, mcc = best_threshold_search(y_train, p_mix)
            if mcc > best["mcc_train"]:
                best = {"weights": weights, "thr": thr, "mcc_train": mcc}
        weights = tuple(np.ones(n) / n)
        p_mix = sum(w * P_train[:, idx] for w, idx in zip(weights, idxs))
        thr, mcc = best_threshold_search(y_train, p_mix)
        if mcc > best["mcc_train"]:
            best = {"weights": weights, "thr": thr, "mcc_train": mcc}

    return best["weights"], best["thr"]

def nested_cv_weighted_avg(combo):
    idxs = [MODEL_IDX[m] for m in combo]
    n = len(y_true)
    combo_seed = combo_to_seed(combo)
    p_oof = np.zeros(n)
    fold_info = []
    for fold_id in range(N_SPLITS):
        tr_mask = folds != fold_id
        te_mask = folds == fold_id
        fold_seed = combo_seed + fold_id
        weights, thr = optimize_weights_on_train(combo, y_true[tr_mask], PROBA_MATRIX[tr_mask], idxs, fold_seed)
        p_test = sum(w * PROBA_MATRIX[te_mask, idx] for w, idx in zip(weights, idxs))
        p_oof[te_mask] = p_test
        fold_info.append({"fold": fold_id, "weights": weights, "thr": thr})
    final_thr, final_mcc = best_threshold_search(y_true, p_oof)
    y_pred = (p_oof >= final_thr).astype(int)
    return {
        "method": "WeightedAvg",
        "models": combo,
        "thr": final_thr,
        "mcc": final_mcc,
        "acc": float(accuracy_score(y_true, y_pred)),
        "bacc": float(balanced_accuracy_score(y_true, y_pred)),
        "f1": float(f1_score(y_true, y_pred, average="macro")),
        "p_oof": p_oof,
        "fold_info": fold_info,
    }

def nested_cv_stacking(combo, meta_type="LR"):
    idxs = [MODEL_IDX[m] for m in combo]
    X_meta = PROBA_MATRIX[:, idxs]
    n = len(y_true)
    p_oof = np.zeros(n)
    for fold_id in range(N_SPLITS):
        tr_mask = folds != fold_id
        te_mask = folds == fold_id
        scaler = StandardScaler()
        X_train = scaler.fit_transform(X_meta[tr_mask])
        X_test = scaler.transform(X_meta[te_mask])
        if meta_type == "LR":
            clf = LogisticRegression(solver="lbfgs", max_iter=500, C=1.0, random_state=SEED)
        else:
            clf = MLPClassifier(hidden_layer_sizes=(32,), max_iter=300, random_state=SEED, early_stopping=True)
        clf.fit(X_train, y_true[tr_mask])
        p_oof[te_mask] = clf.predict_proba(X_test)[:, 1]
    final_thr, final_mcc = best_threshold_search(y_true, p_oof)
    y_pred = (p_oof >= final_thr).astype(int)
    return {
        "method": f"Stacking_{meta_type}",
        "models": combo,
        "thr": final_thr,
        "mcc": final_mcc,
        "acc": float(accuracy_score(y_true, y_pred)),
        "bacc": float(balanced_accuracy_score(y_true, y_pred)),
        "f1": float(f1_score(y_true, y_pred, average="macro")),
        "p_oof": p_oof,
    }

def nested_cv_rank_fusion(combo):
    idxs = [MODEL_IDX[m] for m in combo]
    n = len(y_true)
    p_oof = np.zeros(n)
    for fold_id in range(N_SPLITS):
        tr_mask = folds != fold_id
        te_mask = folds == fold_id
        ranks_test = []
        for idx in idxs:
            p_train = PROBA_MATRIX[tr_mask, idx]
            p_test = PROBA_MATRIX[te_mask, idx]
            r_test = np.array([np.mean(p_train <= p) for p in p_test])
            ranks_test.append(r_test)
        p_oof[te_mask] = np.mean(ranks_test, axis=0)
    final_thr, final_mcc = best_threshold_search(y_true, p_oof)
    y_pred = (p_oof >= final_thr).astype(int)
    return {
        "method": "RankFusion",
        "models": combo,
        "thr": final_thr,
        "mcc": final_mcc,
        "acc": float(accuracy_score(y_true, y_pred)),
        "bacc": float(balanced_accuracy_score(y_true, y_pred)),
        "f1": float(f1_score(y_true, y_pred, average="macro")),
        "p_oof": p_oof,
    }

# ============================================================
# MAIN EXECUTION
# ============================================================
print("\n" + "=" * 80)
print("RUNNING NESTED CV ENSEMBLE SEARCH")
print("=" * 80)

all_results = []

# --- 1. Weighted Average ---
print("\n[1/3] Weighted Average (Nested CV)...")
all_combos = []
for n in range(2, n_models + 1):
    all_combos.extend(list(combinations(model_names, n)))
print(f"  {len(all_combos)} combinations")

results_wa = Parallel(n_jobs=N_JOBS, backend="loky", verbose=10)(
    delayed(nested_cv_weighted_avg)(combo) for combo in all_combos
)
all_results.extend(results_wa)
print(f"  {len(results_wa)} done")

# --- 2. Stacking ---
print("\n[2/3] Stacking (Nested CV)...")
wa_df = pd.DataFrame([{k: v for k, v in r.items() if k != "p_oof" and k != "fold_info"} for r in results_wa])
wa_df = wa_df.sort_values("mcc", ascending=False)
top_combos = [tuple(row["models"]) for _, row in wa_df.head(30).iterrows()]

specific_combos = [
    ("SAUL", "LLaMA"), ("SAUL", "TF-IDF"), ("SAUL", "LLaMA", "TF-IDF"),
    ("SAUL", "LLaMA", "TF-IDF", "JuriBERT"), tuple(model_names),
]
for c in specific_combos:
    valid = all(m in model_names for m in c)
    if valid and c not in top_combos:
        top_combos.append(c)

stacking_tasks = [(c, "LR") for c in top_combos] + [(c, "MLP") for c in top_combos]
results_stack = Parallel(n_jobs=N_JOBS, backend="loky", verbose=5)(
    delayed(nested_cv_stacking)(combo, meta) for combo, meta in stacking_tasks
)
all_results.extend(results_stack)
print(f"  {len(results_stack)} done")

# --- 3. Rank Fusion ---
print("\n[3/3] Rank Fusion (Nested CV)...")
rank_combos = list(combinations(model_names, 2)) + list(combinations(model_names, 3))
rank_combos.extend([
    tuple(model_names),
    ("SAUL", "LLaMA", "TF-IDF", "JuriBERT"),
])
# Deduplicate
rank_combos = list(set(rank_combos))

results_rank = Parallel(n_jobs=N_JOBS, backend="loky", verbose=5)(
    delayed(nested_cv_rank_fusion)(combo) for combo in rank_combos
)
all_results.extend(results_rank)
print(f"  {len(results_rank)} done")

# ============================================================
# FIND BEST & KEEP P_OOF
# ============================================================
all_results_sorted = sorted(all_results, key=lambda x: x["mcc"], reverse=True)
best_result = all_results_sorted[0]

best_p_oof = best_result["p_oof"].copy()
best_thr = best_result["thr"]
best_models = best_result["models"]
best_method = best_result["method"]
best_mcc = best_result["mcc"]

# ============================================================
# BASELINES
# ============================================================
print("\n[4/4] Single model baselines...")
baseline_results = {}
for name in model_names:
    p = probas[name]
    thrs_per_fold = []
    for fold_id in range(N_SPLITS):
        tr_mask = folds != fold_id
        thr, _ = best_threshold_search(y_true[tr_mask], p[tr_mask])
        thrs_per_fold.append(thr)
    avg_thr = np.mean(thrs_per_fold)
    y_pred = (p >= avg_thr).astype(int)
    mcc = fast_mcc(y_true, y_pred)
    y_pred_05 = (p >= 0.5).astype(int)
    mcc_05 = fast_mcc(y_true, y_pred_05)
    baseline_results[name] = {"thr": float(avg_thr), "mcc": float(mcc), "mcc_05": float(mcc_05)}
    print(f"  {name:15s}: MCC={mcc:.4f} (thr={avg_thr:.4f}) | MCC@0.5={mcc_05:.4f}")

# ============================================================
# RESULTS SUMMARY
# ============================================================
print("\n" + "=" * 80)
print("RESULTS (NESTED CV - NO LEAKAGE)")
print("=" * 80)

results_df = pd.DataFrame([
    {k: v for k, v in r.items() if k not in ["p_oof", "fold_info"]}
    for r in all_results_sorted
])

print("\n=== TOP 25 OVERALL ===")
for i, row in results_df.head(25).iterrows():
    models_str = '+'.join(row['models'])
    has_lawma = "⭐" if "Lawma" in row["models"] else "  "
    print(f"{i+1:2d}. {has_lawma} {row['method']:15s} MCC={row['mcc']:.4f} | {models_str}")

print("\n=== BEST BY METHOD ===")
for method in results_df["method"].unique():
    best = results_df[results_df["method"] == method].iloc[0]
    print(f"{method:15s}: MCC={best['mcc']:.4f} | {'+'.join(best['models'])}")

print("\n=== BEST WITH LAWMA (by method) ===")
for method in results_df["method"].unique():
    sub = results_df[results_df["method"] == method]
    with_lawma = sub[sub["models"].apply(lambda m: "Lawma" in m)]
    if len(with_lawma) > 0:
        best = with_lawma.iloc[0]
        print(f"{method:15s}: MCC={best['mcc']:.4f} | {'+'.join(best['models'])}")

print("\n=== BEST WITHOUT LAWMA (by method) ===")
for method in results_df["method"].unique():
    sub = results_df[results_df["method"] == method]
    without_lawma = sub[sub["models"].apply(lambda m: "Lawma" not in m)]
    if len(without_lawma) > 0:
        best = without_lawma.iloc[0]
        print(f"{method:15s}: MCC={best['mcc']:.4f} | {'+'.join(best['models'])}")

print("\n=== LAWMA IMPACT: BEST WITH vs WITHOUT ===")
overall_with = results_df[results_df["models"].apply(lambda m: "Lawma" in m)]
overall_without = results_df[results_df["models"].apply(lambda m: "Lawma" not in m)]
best_with = overall_with.iloc[0]["mcc"] if len(overall_with) > 0 else 0
best_without = overall_without.iloc[0]["mcc"] if len(overall_without) > 0 else 0
print(f"  Best WITH Lawma:    MCC={best_with:.4f}")
print(f"  Best WITHOUT Lawma: MCC={best_without:.4f}")
print(f"  Delta:              {best_with - best_without:+.4f}")

print("\n=== SINGLE MODEL BASELINES ===")
for name, res in sorted(baseline_results.items(), key=lambda x: x[1]["mcc"], reverse=True):
    print(f"  {name:15s}: MCC={res['mcc']:.4f}")

# ============================================================
# SAVE RESULTS
# ============================================================
print("\n" + "=" * 80)
print("SAVING")
print("=" * 80)

def serialize(row):
    return {
        "method": row["method"],
        "models": list(row["models"]),
        "threshold": float(row["thr"]),
        "mcc": float(row["mcc"]),
        "bacc": float(row.get("bacc", 0)),
        "acc": float(row.get("acc", 0)),
        "f1": float(row.get("f1", 0)),
    }

with open(os.path.join(ENSEMBLE_OUTPUT, "all_results_nested_cv_with_lawma.json"), "w") as f:
    json.dump([serialize(row) for _, row in results_df.iterrows()], f, indent=2)

csv_df = results_df.copy()
csv_df["models"] = csv_df["models"].apply(lambda x: "+".join(x))
csv_df.to_csv(os.path.join(ENSEMBLE_OUTPUT, "all_results_nested_cv_with_lawma.csv"), index=False)

np.save(os.path.join(BEST_OUTPUT_PATH, "proba_oof_BEST_ENSEMBLE_nested_cv_with_lawma.npy"), best_p_oof)

y_pred_best = (best_p_oof >= best_thr).astype(int)
df_oof = pd.DataFrame({
    "label": y_true,
    "fold": folds,
    "proba_ensemble": best_p_oof,
    "pred": y_pred_best,
})
df_oof.to_csv(os.path.join(BEST_OUTPUT_PATH, "oof_BEST_ENSEMBLE_nested_cv_with_lawma.csv"), index=False)

print(f"Saved to: {ENSEMBLE_OUTPUT}")

# ============================================================
# DETAILED EVALUATION OF BEST ENSEMBLE
# ============================================================
print("\n" + "=" * 80)
print("DETAILED EVALUATION - BEST ENSEMBLE")
print("=" * 80)

print(f"\nBest Ensemble: {best_method}")
print(f"Models: {' + '.join(best_models)}")
print(f"Threshold: {best_thr:.6f}")
print(f"N samples: {len(y_true)}")

y_pred = (best_p_oof >= best_thr).astype(int)

cm = confusion_matrix(y_true, y_pred)
tn, fp, fn, tp = cm.ravel()

print(f"\n                 Predicted")
print(f"                 non    oui")
print(f"Actual non      {tn:4d}   {fp:4d}    (N={tn+fp})")
print(f"       oui      {fn:4d}   {tp:4d}    (N={fn+tp})")
print(f"\nTP={tp}  TN={tn}  FP={fp}  FN={fn}")

prec_per_class = precision_score(y_true, y_pred, average=None)
rec_per_class = recall_score(y_true, y_pred, average=None)
f1_per_class = f1_score(y_true, y_pred, average=None)
support = [np.sum(y_true == 0), np.sum(y_true == 1)]

print(f"\n{'Class':<10} {'Precision':>10} {'Recall':>10} {'F1':>10} {'Support':>10}")
print("-" * 52)
print(f"{'non (0)':<10} {prec_per_class[0]:>10.4f} {rec_per_class[0]:>10.4f} {f1_per_class[0]:>10.4f} {support[0]:>10d}")
print(f"{'oui (1)':<10} {prec_per_class[1]:>10.4f} {rec_per_class[1]:>10.4f} {f1_per_class[1]:>10.4f} {support[1]:>10d}")

acc = accuracy_score(y_true, y_pred)
bacc = balanced_accuracy_score(y_true, y_pred)
mcc = matthews_corrcoef(y_true, y_pred)
f1_macro = f1_score(y_true, y_pred, average="macro")

print(f"\n{'Metric':<30} {'Value':>10}")
print("-" * 42)
print(f"{'Accuracy':<30} {acc:>10.4f}")
print(f"{'Balanced Accuracy':<30} {bacc:>10.4f}")
print(f"{'MCC':<30} {mcc:>10.4f}")
print(f"{'F1 (macro)':<30} {f1_macro:>10.4f}")

y_pred_05 = (best_p_oof >= 0.5).astype(int)
mcc_05 = matthews_corrcoef(y_true, y_pred_05)
acc_05 = accuracy_score(y_true, y_pred_05)
bacc_05 = balanced_accuracy_score(y_true, y_pred_05)
f1_05 = f1_score(y_true, y_pred_05, average="macro")

print(f"\n{'Metric':<20} {'thr='+f'{best_thr:.3f}':>12} {'thr=0.5':>12} {'Delta':>12}")
print("-" * 58)
print(f"{'MCC':<20} {mcc:>12.4f} {mcc_05:>12.4f} {mcc-mcc_05:>+12.4f}")
print(f"{'Accuracy':<20} {acc:>12.4f} {acc_05:>12.4f} {acc-acc_05:>+12.4f}")
print(f"{'Balanced Acc':<20} {bacc:>12.4f} {bacc_05:>12.4f} {bacc-bacc_05:>+12.4f}")
print(f"{'F1 (macro)':<20} {f1_macro:>12.4f} {f1_05:>12.4f} {f1_macro-f1_05:>+12.4f}")

print("\n" + "-" * 50)
print(classification_report(y_true, y_pred, target_names=["non", "oui"], digits=4))

metrics_summary = {
    "ensemble": {"method": best_method, "models": list(best_models), "threshold": float(best_thr)},
    "confusion_matrix": {"TP": int(tp), "TN": int(tn), "FP": int(fp), "FN": int(fn)},
    "aggregate": {"accuracy": float(acc), "balanced_accuracy": float(bacc), "mcc": float(mcc), "f1_macro": float(f1_macro)},
    "baselines": baseline_results,
}
with open(os.path.join(BEST_OUTPUT_PATH, "detailed_metrics_best_ensemble_with_lawma.json"), "w") as f:
    json.dump(metrics_summary, f, indent=2)

print(f"\n" + "=" * 80)
print(f"FINAL: MCC = {mcc:.4f} | Threshold = {best_thr:.4f}")
print(f"Best ensemble: {best_method} with {'+'.join(best_models)}")
print("=" * 80)

# LLama Details

In [ ]:
# ============================================================
# DETAILED EVALUATION — TOP 5 ENSEMBLES
# Reproduces the search results and displays the
# complete metrics for the 5 best ensembles
# ============================================================

import os, json, re, warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
from sklearn.metrics import (
    accuracy_score, balanced_accuracy_score, f1_score,
    precision_score, recall_score, confusion_matrix,
    classification_report, matthews_corrcoef
)
from sklearn.linear_model import LogisticRegression
from sklearn.neural_network import MLPClassifier
from sklearn.preprocessing import StandardScaler

# ============================================================
# CONFIG
# ============================================================
BASE_PATH = "artifacts"  # not shipped — see DATA.md
OUTPUT_PATH = os.path.join(BASE_PATH, "oof_proba_final")
EVAL_OUTPUT = os.path.join(OUTPUT_PATH, "top5_detailed_evaluation")
os.makedirs(EVAL_OUTPUT, exist_ok=True)

SEED = 42
N_SPLITS = 5
THR_GRID = np.linspace(0.15, 0.85, 500)
np.random.seed(SEED)

# ============================================================
# TOP 5 ENSEMBLES TO EVALUATE (from the search)
# ============================================================
TOP5 = [
    {
        "rank": 1,
        "method": "WeightedAvg",
        "models": ["CamemBERT", "JuriBERT", "SAUL", "ST-MiniLM", "ST-MPNet", "Lawma"],
        "mcc_ref": 0.5283,
    },
    {
        "rank": 2,
        "method": "WeightedAvg",
        "models": ["CamemBERTav2", "JuriBERT", "SAUL", "Lawma"],
        "mcc_ref": 0.5279,
    },
    {
        "rank": 3,
        "method": "Stacking_LR",
        "models": ["CamemBERT", "JuriBERT", "SAUL", "ST-MiniLM", "ST-MPNet", "Lawma"],
        "mcc_ref": 0.5273,
    },
    {
        "rank": 4,
        "method": "Stacking_LR",
        "models": ["CamemBERTav2", "JuriBERT", "LLaMA", "SAUL"],
        "mcc_ref": 0.5258,
    },
    {
        "rank": 5,
        "method": "Stacking_LR",
        "models": ["CamemBERT", "JuriBERT", "SAUL", "ST-MiniLM", "ST-MPNet", "TF-IDF", "Lawma"],
        "mcc_ref": 0.5253,
    },
]

# ============================================================
# LOAD DATA
# ============================================================
MODELS_FILES = {
    "CamemBERT": "proba_oof_CamemBERT_BEST_cfg3_mean_avg_last_k_4_MLP2.npy",
    "CamemBERTav2": "proba_oof_CamemBERTav2_BEST_cfg1_mean_avg_last_k_4_MLP1.npy",
    "JuriBERT": "proba_oof_JuriBERT-base_BEST_cfg1_cls_last_LR.npy",
    "LLaMA": "proba_oof_LLaMA_BEST_cfg2_mean_layer_-2_MLP1.npy",
    "SAUL": "proba_oof_SAUL_BEST_cfg3_mean_last_k_4_LR.npy",
    "ST-MiniLM": "proba_oof_ST-MiniLM_BEST_cfg3_mean_layer_-2_MLP2.npy",
    "ST-MPNet": "proba_oof_ST-MPNet_BEST_cfg1_mean_avg_last_k_4_MLP2.npy",
}
TFIDF_CSV = "tfidf_oof_probas.csv"
LAWMA_PROBA_PATH = os.path.join(
    BASE_PATH, "outputs", "outputs_lawma_mlp1_mlp2_3cfg",
    "proba_oof_Lawma-8B__cfg1__mean__last__mlp1.npy"
)

print("=" * 100)
print("DETAILED EVALUATION — TOP 5 ENSEMBLES")
print("=" * 100)

# Load labels + folds
def load_labels():
    df0 = pd.read_excel("DATA/outputs/benchmark.csv")
    cols = ["decision_id", "eval_A1", "eval_A2", "eval_A3"]
    df0 = df0[cols].copy()
    def extract(x):
        if pd.isna(x): return np.nan
        s = str(x).lower()
        if re.search(r"\boui\b", s) and not re.search(r"\bnon\b", s): return "oui"
        if re.search(r"\bnon\b", s) and not re.search(r"\boui\b", s): return "non"
        return np.nan
    df0["a"] = df0["eval_A1"].apply(extract)
    df0["t"] = df0["eval_A2"].apply(extract)
    df0["s"] = df0["eval_A3"].apply(extract)
    def resolve(row):
        a, t, s = row["a"], row["t"], row["s"]
        if pd.notna(a) and pd.notna(t) and a == t: return a
        if pd.notna(a) and pd.notna(t) and a != t and pd.notna(s): return s
        return np.nan
    df0["label_str"] = df0.apply(resolve, axis=1)
    df0 = df0[df0["label_str"].isin(["oui", "non"])].copy()
    df0["label"] = df0["label_str"].map({"oui": 1, "non": 0}).astype(int)
    rng = np.random.default_rng(SEED)
    groups = df0.groupby("decision_id").size().to_dict()
    uniq = list(groups.keys())
    rng.shuffle(uniq)
    uniq = sorted(uniq, key=lambda g: groups[g], reverse=True)
    loads = np.zeros(N_SPLITS, dtype=int)
    g2f = {}
    for g in uniq:
        f = int(loads.argmin())
        g2f[g] = f
        loads[f] += groups[g]
    df0["fold"] = df0["decision_id"].map(g2f).astype(int)
    return df0["label"].values.astype(int), df0["fold"].values

y_true, folds = load_labels()
print(f"  {len(y_true)} samples, {N_SPLITS} folds")
print(f"  oui={np.sum(y_true==1)}, non={np.sum(y_true==0)}")

# Load probas
probas = {}
for name, fname in MODELS_FILES.items():
    path = os.path.join(OUTPUT_PATH, fname)
    if os.path.exists(path):
        probas[name] = np.load(path)
tfidf_path = os.path.join(OUTPUT_PATH, TFIDF_CSV)
if os.path.exists(tfidf_path):
    probas["TF-IDF"] = pd.read_csv(tfidf_path)["proba_oui"].values
if os.path.exists(LAWMA_PROBA_PATH):
    probas["Lawma"] = np.load(LAWMA_PROBA_PATH)
else:
    for root, dirs, files in os.walk(os.path.join(BASE_PATH, "outputs")):
        for f in files:
            if "Lawma" in f and "cfg1" in f and "last" in f and "mlp1" in f and f.endswith(".npy"):
                probas["Lawma"] = np.load(os.path.join(root, f))
                break

print(f"  Models loaded: {list(probas.keys())}")

model_names = list(probas.keys())
PROBA_MATRIX = np.column_stack([probas[m] for m in model_names])
MODEL_IDX = {m: i for i, m in enumerate(model_names)}

# ============================================================
# HELPER FUNCTIONS
# ============================================================
def fast_mcc(y_true, y_pred):
    tp = np.sum((y_true == 1) & (y_pred == 1))
    tn = np.sum((y_true == 0) & (y_pred == 0))
    fp = np.sum((y_true == 0) & (y_pred == 1))
    fn = np.sum((y_true == 1) & (y_pred == 0))
    num = float(tp * tn - fp * fn)
    den = np.sqrt(float((tp + fp) * (tp + fn) * (tn + fp) * (tn + fn)))
    return num / den if den > 0 else 0.0

def best_threshold_search(y_true, p_proba):
    best_thr, best_mcc = 0.5, -1
    for thr in THR_GRID:
        y_pred = (p_proba >= thr).astype(int)
        mcc = fast_mcc(y_true, y_pred)
        if mcc > best_mcc:
            best_mcc = mcc
            best_thr = thr
    return best_thr, best_mcc

def combo_to_seed(combo):
    s = "+".join(sorted(combo))
    return sum(ord(c) * (i + 1) for i, c in enumerate(s)) % (2**31)

# ============================================================
# REPRODUCE ENSEMBLE PREDICTIONS
# ============================================================
def reproduce_weighted_avg(combo):
    idxs = [MODEL_IDX[m] for m in combo]
    n_combo = len(combo)
    combo_seed = combo_to_seed(tuple(combo))
    n = len(y_true)
    p_oof = np.zeros(n)
    fold_weights = []

    for fold_id in range(N_SPLITS):
        tr_mask = folds != fold_id
        te_mask = folds == fold_id
        fold_seed = combo_seed + fold_id
        rng = np.random.default_rng(fold_seed)

        best = {"mcc_train": -1, "weights": None, "thr": 0.5}
        P_train = PROBA_MATRIX[tr_mask]
        y_train = y_true[tr_mask]

        if n_combo == 2:
            for w1 in np.linspace(0, 1, 21):
                weights = (w1, 1 - w1)
                p_mix = sum(w * P_train[:, idx] for w, idx in zip(weights, idxs))
                thr, mcc = best_threshold_search(y_train, p_mix)
                if mcc > best["mcc_train"]:
                    best = {"weights": weights, "thr": thr, "mcc_train": mcc}
        elif n_combo == 3:
            for w1 in np.linspace(0, 1, 11):
                for w2 in np.linspace(0, 1 - w1, 11):
                    w3 = 1 - w1 - w2
                    if w3 < -1e-6: continue
                    weights = (w1, w2, w3)
                    p_mix = sum(w * P_train[:, idx] for w, idx in zip(weights, idxs))
                    thr, mcc = best_threshold_search(y_train, p_mix)
                    if mcc > best["mcc_train"]:
                        best = {"weights": weights, "thr": thr, "mcc_train": mcc}
        else:
            for _ in range(1000):
                weights = tuple(rng.dirichlet(np.ones(n_combo)))
                p_mix = sum(w * P_train[:, idx] for w, idx in zip(weights, idxs))
                thr, mcc = best_threshold_search(y_train, p_mix)
                if mcc > best["mcc_train"]:
                    best = {"weights": weights, "thr": thr, "mcc_train": mcc}
            weights = tuple(np.ones(n_combo) / n_combo)
            p_mix = sum(w * P_train[:, idx] for w, idx in zip(weights, idxs))
            thr, mcc = best_threshold_search(y_train, p_mix)
            if mcc > best["mcc_train"]:
                best = {"weights": weights, "thr": thr, "mcc_train": mcc}

        p_test = sum(w * PROBA_MATRIX[te_mask, idx] for w, idx in zip(best["weights"], idxs))
        p_oof[te_mask] = p_test
        fold_weights.append({"fold": fold_id, "weights": best["weights"], "thr_train": best["thr"],
                             "mcc_train": best["mcc_train"]})

    return p_oof, fold_weights

def reproduce_stacking_lr(combo):
    idxs = [MODEL_IDX[m] for m in combo]
    X_meta = PROBA_MATRIX[:, idxs]
    n = len(y_true)
    p_oof = np.zeros(n)
    fold_info = []

    for fold_id in range(N_SPLITS):
        tr_mask = folds != fold_id
        te_mask = folds == fold_id
        scaler = StandardScaler()
        X_train = scaler.fit_transform(X_meta[tr_mask])
        X_test = scaler.transform(X_meta[te_mask])
        clf = LogisticRegression(solver="lbfgs", max_iter=500, C=1.0, random_state=SEED)
        clf.fit(X_train, y_true[tr_mask])
        p_oof[te_mask] = clf.predict_proba(X_test)[:, 1]
        fold_info.append({"fold": fold_id, "coefs": clf.coef_[0].tolist(), "intercept": float(clf.intercept_[0])})

    return p_oof, fold_info

# ============================================================
# EVALUATE EACH ENSEMBLE
# ============================================================
all_summaries = []

for entry in TOP5:
    rank = entry["rank"]
    method = entry["method"]
    models = entry["models"]
    mcc_ref = entry["mcc_ref"]

    print("\n" + "█" * 100)
    print(f"  RANK #{rank} — {method}")
    print(f"  Models: {' + '.join(models)}")
    print(f"  MCC (ref): {mcc_ref:.4f}")
    print("█" * 100)

    # Reproduce
    if method == "WeightedAvg":
        p_oof, fold_detail = reproduce_weighted_avg(models)
    elif method == "Stacking_LR":
        p_oof, fold_detail = reproduce_stacking_lr(models)
    else:
        print(f"  ⚠ Method {method} not implemented for reproduction, skipping")
        continue

    # Optimal threshold
    best_thr, best_mcc = best_threshold_search(y_true, p_oof)
    y_pred = (p_oof >= best_thr).astype(int)

    # Fixed threshold 0.5
    y_pred_05 = (p_oof >= 0.5).astype(int)

    # Verification
    print(f"\n  Reproduced MCC = {best_mcc:.4f} (ref: {mcc_ref:.4f}, Δ={best_mcc - mcc_ref:+.4f})")

    # ---- Confusion Matrix ----
    cm = confusion_matrix(y_true, y_pred)
    tn, fp, fn, tp = cm.ravel()

    print(f"\n  {'─'*60}")
    print(f"  CONFUSION MATRIX (thr={best_thr:.4f})")
    print(f"  {'─'*60}")
    print(f"                   Predicted")
    print(f"                   non    oui")
    print(f"  Actual non      {tn:4d}   {fp:4d}    (N={tn+fp})")
    print(f"         oui      {fn:4d}   {tp:4d}    (N={fn+tp})")
    print(f"  TP={tp}  TN={tn}  FP={fp}  FN={fn}")

    # ---- Per-class ----
    prec_cls = precision_score(y_true, y_pred, average=None)
    rec_cls = recall_score(y_true, y_pred, average=None)
    f1_cls = f1_score(y_true, y_pred, average=None)
    support = [np.sum(y_true == 0), np.sum(y_true == 1)]

    print(f"\n  {'─'*60}")
    print(f"  PER-CLASS METRICS")
    print(f"  {'─'*60}")
    print(f"  {'Class':<12} {'Precision':>10} {'Recall':>10} {'F1':>10} {'Support':>10}")
    print(f"  {'─'*54}")
    print(f"  {'non (0)':<12} {prec_cls[0]:>10.4f} {rec_cls[0]:>10.4f} {f1_cls[0]:>10.4f} {support[0]:>10d}")
    print(f"  {'oui (1)':<12} {prec_cls[1]:>10.4f} {rec_cls[1]:>10.4f} {f1_cls[1]:>10.4f} {support[1]:>10d}")

    # ---- Aggregate ----
    acc = accuracy_score(y_true, y_pred)
    bacc = balanced_accuracy_score(y_true, y_pred)
    mcc = matthews_corrcoef(y_true, y_pred)
    f1_macro = f1_score(y_true, y_pred, average="macro")
    f1_weighted = f1_score(y_true, y_pred, average="weighted")
    prec_macro = precision_score(y_true, y_pred, average="macro")
    rec_macro = recall_score(y_true, y_pred, average="macro")
    specificity = tn / (tn + fp) if (tn + fp) > 0 else 0
    sensitivity = tp / (tp + fn) if (tp + fn) > 0 else 0

    print(f"\n  {'─'*60}")
    print(f"  AGGREGATE METRICS")
    print(f"  {'─'*60}")
    print(f"  {'Metric':<30} {'Value':>10}")
    print(f"  {'─'*42}")
    print(f"  {'Accuracy':<30} {acc:>10.4f}")
    print(f"  {'Balanced Accuracy':<30} {bacc:>10.4f}")
    print(f"  {'MCC':<30} {mcc:>10.4f}")
    print(f"  {'F1 (macro)':<30} {f1_macro:>10.4f}")
    print(f"  {'F1 (weighted)':<30} {f1_weighted:>10.4f}")
    print(f"  {'Precision (macro)':<30} {prec_macro:>10.4f}")
    print(f"  {'Recall (macro)':<30} {rec_macro:>10.4f}")
    print(f"  {'Specificity':<30} {specificity:>10.4f}")
    print(f"  {'Sensitivity':<30} {sensitivity:>10.4f}")

    # ---- Comparison best thr vs 0.5 ----
    mcc_05 = matthews_corrcoef(y_true, y_pred_05)
    acc_05 = accuracy_score(y_true, y_pred_05)
    bacc_05 = balanced_accuracy_score(y_true, y_pred_05)
    f1_05 = f1_score(y_true, y_pred_05, average="macro")

    print(f"\n  {'─'*60}")
    print(f"  THRESHOLD COMPARISON")
    print(f"  {'─'*60}")
    print(f"  {'Metric':<20} {'thr='+f'{best_thr:.3f}':>12} {'thr=0.5':>12} {'Δ':>12}")
    print(f"  {'─'*58}")
    print(f"  {'MCC':<20} {mcc:>12.4f} {mcc_05:>12.4f} {mcc-mcc_05:>+12.4f}")
    print(f"  {'Accuracy':<20} {acc:>12.4f} {acc_05:>12.4f} {acc-acc_05:>+12.4f}")
    print(f"  {'Balanced Acc':<20} {bacc:>12.4f} {bacc_05:>12.4f} {bacc-bacc_05:>+12.4f}")
    print(f"  {'F1 (macro)':<20} {f1_macro:>12.4f} {f1_05:>12.4f} {f1_macro-f1_05:>+12.4f}")

    # ---- Fold details ----
    print(f"\n  {'─'*60}")
    print(f"  FOLD DETAILS")
    print(f"  {'─'*60}")
    if method == "WeightedAvg":
        for fi in fold_detail:
            w_str = ", ".join(f"{m}={w:.3f}" for m, w in zip(models, fi["weights"]))
            print(f"  Fold {fi['fold']}: thr_train={fi['thr_train']:.3f} MCC_train={fi['mcc_train']:.4f}")
            print(f"         weights: {w_str}")
    elif method == "Stacking_LR":
        for fi in fold_detail:
            c_str = ", ".join(f"{m}={c:.3f}" for m, c in zip(models, fi["coefs"]))
            print(f"  Fold {fi['fold']}: intercept={fi['intercept']:.4f}")
            print(f"         coefs: {c_str}")

    # Save probas
    np.save(os.path.join(EVAL_OUTPUT, f"proba_oof_rank{rank}_{method}.npy"), p_oof)

    summary = {
        "rank": rank,
        "method": method,
        "models": models,
        "has_lawma": "Lawma" in models,
        "n_models": len(models),
        "threshold": float(best_thr),
        "mcc": float(mcc),
        "mcc_05": float(mcc_05),
        "accuracy": float(acc),
        "balanced_accuracy": float(bacc),
        "f1_macro": float(f1_macro),
        "precision_macro": float(prec_macro),
        "recall_macro": float(rec_macro),
        "specificity": float(specificity),
        "sensitivity": float(sensitivity),
        "f1_oui": float(f1_cls[1]),
        "f1_non": float(f1_cls[0]),
        "TP": int(tp), "TN": int(tn), "FP": int(fp), "FN": int(fn),
    }
    all_summaries.append(summary)

# ============================================================
# COMPARATIVE TABLE
# ============================================================
print("\n" + "=" * 100)
print("COMPARISON TABLE — TOP 5 ENSEMBLES")
print("=" * 100)

df_comp = pd.DataFrame(all_summaries)
print(f"\n{'Rank':<5} {'Method':<15} {'Models':<55} {'Lawma':>5} {'Thr':>6} {'MCC':>7} {'MCC@.5':>7} {'BAcc':>7} {'F1m':>7} {'F1-oui':>7} {'F1-non':>7} {'Spec':>7} {'Sens':>7}")
print("─" * 145)
for _, row in df_comp.iterrows():
    models_str = "+".join(row["models"])
    if len(models_str) > 53:
        models_str = models_str[:50] + "..."
    lawma = "✓" if row["has_lawma"] else ""
    print(f"#{row['rank']:<4} {row['method']:<15} {models_str:<55} {lawma:>5} {row['threshold']:>6.3f} {row['mcc']:>7.4f} {row['mcc_05']:>7.4f} {row['balanced_accuracy']:>7.4f} {row['f1_macro']:>7.4f} {row['f1_oui']:>7.4f} {row['f1_non']:>7.4f} {row['specificity']:>7.4f} {row['sensitivity']:>7.4f}")

# Save
df_comp_save = df_comp.copy()
df_comp_save["models"] = df_comp_save["models"].apply(lambda x: "+".join(x))
df_comp_save.to_csv(os.path.join(EVAL_OUTPUT, "top5_comparison.csv"), index=False)

with open(os.path.join(EVAL_OUTPUT, "top5_detailed_metrics.json"), "w") as f:
    json.dump(all_summaries, f, indent=2, default=str)

print(f"\n✅ Saved to: {EVAL_OUTPUT}")

print("\n" + "=" * 100)
print("BEST F1-OUI IN THE TOP 5")
print("=" * 100)
best_f1oui = max(all_summaries, key=lambda x: x.get("f1_oui", 0))
print(f"  Rank #{best_f1oui['rank']} — {best_f1oui['method']}")
print(f"  Models: {' + '.join(best_f1oui['models'])}")
print(f"  F1-oui = {best_f1oui['f1_oui']:.4f}  (Sensitivity = {best_f1oui['sensitivity']:.4f})")
print(f"  MCC = {best_f1oui['mcc']:.4f}  |  F1-non = {best_f1oui['f1_non']:.4f}")

print("\n" + "=" * 100)
print("KEY FINDING: LAWMA IMPACT")
print("=" * 100)
with_lawma = [s for s in all_summaries if s["has_lawma"]]
without_lawma = [s for s in all_summaries if not s["has_lawma"]]
if with_lawma and without_lawma:
    best_with = max(with_lawma, key=lambda x: x["mcc"])
    best_without = max(without_lawma, key=lambda x: x["mcc"])
    print(f"  Best WITH Lawma:    #{best_with['rank']} MCC={best_with['mcc']:.4f} ({best_with['method']})")
    print(f"  Best WITHOUT Lawma: #{best_without['rank']} MCC={best_without['mcc']:.4f} ({best_without['method']})")
    print(f"  Δ = {best_with['mcc'] - best_without['mcc']:+.4f}")

# Rebuttal - negative analysis

In [ ]:
# ============================================================
# FN ANALYSIS BY AGREEMENT — STANDALONE VERSION
# ============================================================
import os, re, warnings
warnings.filterwarnings("ignore")
import numpy as np
import pandas as pd
from scipy.stats import fisher_exact
from statsmodels.stats.multitest import multipletests

# ---- CONFIG ----
BASE_PATH = "artifacts"  # not shipped — see DATA.md
OUTPUT_PATH = os.path.join(BASE_PATH, "oof_proba_final")
EVAL_OUTPUT = os.path.join(OUTPUT_PATH, "top5_detailed_evaluation")
SEED = 42
N_SPLITS = 5
THR_GRID = np.linspace(0.15, 0.85, 500)

# ---- HELPERS ----
def fast_mcc(y_true, y_pred):
    tp = np.sum((y_true == 1) & (y_pred == 1))
    tn = np.sum((y_true == 0) & (y_pred == 0))
    fp = np.sum((y_true == 0) & (y_pred == 1))
    fn = np.sum((y_true == 1) & (y_pred == 0))
    num = float(tp * tn - fp * fn)
    den = np.sqrt(float((tp + fp) * (tp + fn) * (tn + fp) * (tn + fn)))
    return num / den if den > 0 else 0.0

def best_threshold_search(y_true, p_proba):
    best_thr, best_mcc = 0.5, -1
    for thr in THR_GRID:
        y_pred = (p_proba >= thr).astype(int)
        mcc = fast_mcc(y_true, y_pred)
        if mcc > best_mcc:
            best_mcc = mcc
            best_thr = thr
    return best_thr, best_mcc

# ---- 1) Load labels + case (agree/disagree) ----
df = pd.read_excel("DATA/outputs/benchmark.csv")
df = df[["decision_id", "eval_A1", "eval_A2", "eval_A3"]].copy()

def extract(x):
    if pd.isna(x): return np.nan
    s = str(x).lower()
    if re.search(r"\boui\b", s) and not re.search(r"\bnon\b", s): return "oui"
    if re.search(r"\bnon\b", s) and not re.search(r"\boui\b", s): return "non"
    return np.nan

df["a"] = df["eval_A1"].apply(extract)
df["t"] = df["eval_A2"].apply(extract)
df["s"] = df["eval_A3"].apply(extract)

def get_case(row):
    a, t = row["a"], row["t"]
    if pd.notna(a) and pd.notna(t) and a == t:
        return "agree"
    if pd.notna(a) and pd.notna(t) and a != t and pd.notna(row["s"]):
        return "disagree"
    return "other"

def get_label(row):
    if row["case"] == "agree":
        return row["a"]
    elif row["case"] == "disagree":
        return row["s"]
    return np.nan

df["case"] = df.apply(get_case, axis=1)
df["label_str"] = df.apply(get_label, axis=1)
df = df[df["label_str"].isin(["oui", "non"])].reset_index(drop=True)
df["label"] = df["label_str"].map({"oui": 1, "non": 0}).astype(int)

y_true = df["label"].values.astype(int)
case_arr = df["case"].values

print(f"n samples: {len(y_true)}")
print(f"  agree:    {(case_arr == 'agree').sum()}")
print(f"  disagree: {(case_arr == 'disagree').sum()}")
print(f"  oui:      {(y_true == 1).sum()}  /  non: {(y_true == 0).sum()}")

# ---- 2) Load OOF probabilities of individual models ----
MODELS_FILES = {
    "CamemBERT": "proba_oof_CamemBERT_BEST_cfg3_mean_avg_last_k_4_MLP2.npy",
    "CamemBERTav2": "proba_oof_CamemBERTav2_BEST_cfg1_mean_avg_last_k_4_MLP1.npy",
    "JuriBERT": "proba_oof_JuriBERT-base_BEST_cfg1_cls_last_LR.npy",
    "LLaMA": "proba_oof_LLaMA_BEST_cfg2_mean_layer_-2_MLP1.npy",
    "SAUL": "proba_oof_SAUL_BEST_cfg3_mean_last_k_4_LR.npy",
    "ST-MiniLM": "proba_oof_ST-MiniLM_BEST_cfg3_mean_layer_-2_MLP2.npy",
    "ST-MPNet": "proba_oof_ST-MPNet_BEST_cfg1_mean_avg_last_k_4_MLP2.npy",
}
TFIDF_CSV = "tfidf_oof_probas.csv"
LAWMA_PATH = os.path.join(BASE_PATH, "outputs", "outputs_lawma_mlp1_mlp2_3cfg",
                          "proba_oof_Lawma-8B__cfg1__mean__last__mlp1.npy")

probas = {}
for name, fname in MODELS_FILES.items():
    path = os.path.join(OUTPUT_PATH, fname)
    if os.path.exists(path):
        probas[name] = np.load(path)

tfidf_path = os.path.join(OUTPUT_PATH, TFIDF_CSV)
if os.path.exists(tfidf_path):
    probas["TF-IDF"] = pd.read_csv(tfidf_path)["proba_oui"].values

if os.path.exists(LAWMA_PATH):
    probas["Lawma"] = np.load(LAWMA_PATH)
else:
    for root, dirs, files in os.walk(os.path.join(BASE_PATH, "outputs")):
        for f in files:
            if "Lawma" in f and "cfg1" in f and "last" in f and "mlp1" in f and f.endswith(".npy"):
                probas["Lawma"] = np.load(os.path.join(root, f))
                break

print(f"\nModels loaded: {list(probas.keys())}")

# Size check
for m, p in probas.items():
    assert len(p) == len(y_true), f"{m}: {len(p)} vs {len(y_true)}"

# ---- 3) Per-model loop ----
print("\n" + "=" * 100)
print("FALSE NEGATIVE RATE BY AGREEMENT — ALL INDIVIDUAL MODELS")
print("=" * 100)

print(f"\n{'Model':<15} {'FNRag':>8} {'FNRdis':>8} {'OR':>6} {'p':>8} {'n_pos_ag':>10} {'n_pos_dis':>10} {'FN_ag':>7} {'FN_dis':>7}")
print("─" * 90)

fn_results = []
pvals = []

for model_name in probas.keys():
    p_proba = probas[model_name]
    best_thr, _ = best_threshold_search(y_true, p_proba)
    y_pred = (p_proba >= best_thr).astype(int)

    pos_mask = (y_true == 1)
    ag_pos = pos_mask & (case_arr == "agree")
    dis_pos = pos_mask & (case_arr == "disagree")

    n_pos_ag = int(ag_pos.sum())
    n_pos_dis = int(dis_pos.sum())

    fn_ag = int((ag_pos & (y_pred == 0)).sum())
    fn_dis = int((dis_pos & (y_pred == 0)).sum())

    fnr_ag = fn_ag / n_pos_ag if n_pos_ag > 0 else 0
    fnr_dis = fn_dis / n_pos_dis if n_pos_dis > 0 else 0

    tp_ag = n_pos_ag - fn_ag
    tp_dis = n_pos_dis - fn_dis

    try:
        _, p_val = fisher_exact([[fn_ag, tp_ag], [fn_dis, tp_dis]], alternative='two-sided')
    except Exception:
        p_val = np.nan

    if tp_ag > 0 and tp_dis > 0 and fn_ag > 0:
        or_manual = (fn_dis / tp_dis) / (fn_ag / tp_ag)
    else:
        or_manual = np.nan

    fn_results.append({
        "model": model_name, "fnr_ag": fnr_ag, "fnr_dis": fnr_dis,
        "odds_ratio": or_manual, "p_value": p_val,
        "n_pos_ag": n_pos_ag, "n_pos_dis": n_pos_dis,
        "fn_ag": fn_ag, "fn_dis": fn_dis,
    })
    pvals.append(p_val)

    or_str = f"{or_manual:.2f}" if not np.isnan(or_manual) else "  N/A"
    p_str = f"{p_val:.3f}" if not np.isnan(p_val) else "  N/A"
    print(f"{model_name:<15} {fnr_ag*100:>7.1f}% {fnr_dis*100:>7.1f}% {or_str:>6} {p_str:>8} {n_pos_ag:>10d} {n_pos_dis:>10d} {fn_ag:>7d} {fn_dis:>7d}")

# ---- 4) FDR correction ----
print("\n" + "─" * 90)
print("AFTER FDR CORRECTION (Benjamini-Hochberg)")
print("─" * 90)

valid_pvals = [p for p in pvals if not np.isnan(p)]
if len(valid_pvals) > 0:
    rejected, pvals_corrected, _, _ = multipletests(valid_pvals, alpha=0.05, method='fdr_bh')
    print(f"{'Model':<15} {'p_raw':>8} {'p_FDR':>8} {'Sig.':>6}")
    print("─" * 45)
    j = 0
    for r, pv in zip(fn_results, pvals):
        if not np.isnan(pv):
            sig = "***" if pvals_corrected[j] < 0.001 else ("**" if pvals_corrected[j] < 0.01 else ("*" if pvals_corrected[j] < 0.05 else "n.s."))
            print(f"{r['model']:<15} {pv:>8.3f} {pvals_corrected[j]:>8.3f} {sig:>6}")
            j += 1

# ---- 5) Ensemble Rank #4 (if available) ----
rank4_path = os.path.join(EVAL_OUTPUT, "proba_oof_rank4_Stacking_LR.npy")
if os.path.exists(rank4_path):
    print("\n" + "=" * 100)
    print("ENSEMBLE Rank #4 (CamemBERTav2 + JuriBERT + LLaMA + SAUL — Stacking_LR)")
    print("=" * 100)

    p_oof_rank4 = np.load(rank4_path)
    best_thr_rank4, _ = best_threshold_search(y_true, p_oof_rank4)
    y_pred_rank4 = (p_oof_rank4 >= best_thr_rank4).astype(int)

    pos_mask = (y_true == 1)
    ag_pos = pos_mask & (case_arr == "agree")
    dis_pos = pos_mask & (case_arr == "disagree")

    n_pos_ag = int(ag_pos.sum())
    n_pos_dis = int(dis_pos.sum())
    fn_ag = int((ag_pos & (y_pred_rank4 == 0)).sum())
    fn_dis = int((dis_pos & (y_pred_rank4 == 0)).sum())
    tp_ag = n_pos_ag - fn_ag
    tp_dis = n_pos_dis - fn_dis

    fnr_ag = fn_ag / n_pos_ag if n_pos_ag > 0 else 0
    fnr_dis = fn_dis / n_pos_dis if n_pos_dis > 0 else 0
    or_ens = (fn_dis / tp_dis) / (fn_ag / tp_ag) if (tp_ag > 0 and tp_dis > 0 and fn_ag > 0) else np.nan
    _, p_ens = fisher_exact([[fn_ag, tp_ag], [fn_dis, tp_dis]], alternative='two-sided')

    print(f"\n{'Subset':<12} {'n_pos':>7} {'FN':>5} {'TP':>5} {'FNR':>8}")
    print("─" * 45)
    print(f"{'Agree':<12} {n_pos_ag:>7d} {fn_ag:>5d} {tp_ag:>5d} {fnr_ag*100:>7.1f}%")
    print(f"{'Disagree':<12} {n_pos_dis:>7d} {fn_dis:>5d} {tp_dis:>5d} {fnr_dis*100:>7.1f}%")
    or_str = f"{or_ens:.2f}" if not np.isnan(or_ens) else "N/A"
    print(f"\n  OR = {or_str}")
    print(f"  p  = {p_ens:.4f}")
else:
    print(f"\n⚠️  {rank4_path} not found — run the TOP5 script first to generate the ensemble probabilities.")

# ---- 6) Save ----
df_fn = pd.DataFrame(fn_results)
out_csv = os.path.join(EVAL_OUTPUT, "fn_by_agreement.csv")
os.makedirs(EVAL_OUTPUT, exist_ok=True)
df_fn.to_csv(out_csv, index=False)
print(f"\n✅ Saved to: {out_csv}")

# Metrics F1+ Lawma

In [ ]:
# ============================================================
# COMPUTE METRICS FOR ALL MODELS + LAWMA - F1 on positive class + Accuracy
# ============================================================

import os, re, warnings, hashlib, json
import numpy as np
import pandas as pd
from sklearn.metrics import (
    precision_score, recall_score, f1_score,
    matthews_corrcoef, accuracy_score, confusion_matrix
)
from sklearn.preprocessing import StandardScaler
from sklearn.neural_network import MLPClassifier

warnings.filterwarnings("ignore")

# ============================================================
# CONFIG
# ============================================================
BASE_PATH = "artifacts"  # not shipped — see DATA.md
EXCEL_PATH = "DATA/outputs/benchmark.csv"

OUTPUT_PATH = os.path.join(BASE_PATH, "oof_proba_final")
BEST_OUTPUT_PATH = os.path.join(OUTPUT_PATH, "best_models")

# LawMA cache directories (from comparison script)
LAWMA_CACHE_DIRS = [
    os.path.join(BASE_PATH, "outputs", "outputs_lawma_vs_llama_full_comparison", "_cache_embs"),
    os.path.join(BASE_PATH, "outputs", "outputs_lawma_lr_llama_mlp1_3cfg", "_cache_embs"),
    os.path.join(BASE_PATH, "outputs", "outputs_lawma_mlp1_mlp2_3cfg", "_cache_embs"),
]

SEED = 42
N_SPLITS = 5
np.random.seed(SEED)

# ============================================================
# MODELS AND FILES
# ============================================================
MODELS = {
    "CamemBERT": "proba_oof_CamemBERT_BEST_cfg3_mean_avg_last_k_4_MLP2.npy",
    "CamemBERTav2": "proba_oof_CamemBERTav2_BEST_cfg1_mean_avg_last_k_4_MLP1.npy",
    "JuriBERT": "proba_oof_JuriBERT-base_BEST_cfg1_cls_last_LR.npy",
    "LLaMA": "proba_oof_LLaMA_BEST_cfg2_mean_layer_-2_MLP1.npy",
    "SAUL": "proba_oof_SAUL_BEST_cfg3_mean_last_k_4_LR.npy",
    "ST-MiniLM": "proba_oof_ST-MiniLM_BEST_cfg3_mean_layer_-2_MLP2.npy",
    "ST-MPNet": "proba_oof_ST-MPNet_BEST_cfg1_mean_avg_last_k_4_MLP2.npy",
}
TFIDF_CSV = "tfidf_oof_probas.csv"

# Best thresholds per model
BEST_THRESHOLDS = {
    "CamemBERT": 0.482666,
    "CamemBERTav2": 0.587369,
    "JuriBERT": 0.739345,
    "LLaMA": 0.614332,
    "SAUL": 0.793622,
    "ST-MiniLM": 0.539045,
    "ST-MPNet": 0.631491,
    "TF-IDF": 0.471811,
    "BEST_ENSEMBLE": 0.610120,
    # LawMA threshold will be computed automatically
}

# Best classifier head per model
HEADS = {
    "CamemBERT": "MLP2",
    "CamemBERTav2": "MLP1",
    "JuriBERT": "LR",
    "LLaMA": "MLP1",
    "LawMA": "MLP1",
    "SAUL": "LR",
    "ST-MiniLM": "MLP2",
    "ST-MPNet": "MLP2",
    "TF-IDF": "LR",
}

# Display names for LaTeX
DISPLAY_NAMES = {
    "SAUL": "SAUL-7B",
    "LLaMA": "LLaMA-3.1-8B",
    "LawMA": "LawMA-8B",
    "BEST_ENSEMBLE": "Ensemble",
}

# ============================================================
# HELPERS (from comparison script, for LawMA recomputation)
# ============================================================
def make_grouped_folds(df, n_splits, seed):
    rng = np.random.default_rng(seed)
    group_sizes = df.groupby("decision_id").size().to_dict()
    uniq_groups = np.array(list(group_sizes.keys()))
    uniq_groups = uniq_groups[rng.permutation(len(uniq_groups))]
    uniq_groups = sorted(uniq_groups, key=lambda g: group_sizes[g], reverse=True)
    fold_loads = np.zeros(n_splits, dtype=int)
    group_to_fold = {}
    for g in uniq_groups:
        f = int(fold_loads.argmin())
        group_to_fold[g] = f
        fold_loads[f] += int(group_sizes[g])
    out = df.copy()
    out["fold"] = out["decision_id"].map(group_to_fold).astype(int)
    return out, fold_loads

def _hash_texts(texts, n=200):
    sample = texts[:n] + texts[-n:] if len(texts) > 2*n else texts
    blob = "\n".join(map(str, sample)) + f"\n__N__{len(texts)}__"
    return hashlib.md5(blob.encode("utf-8")).hexdigest()

def _cache_key(model_name, pooling, layer_strategy, max_len, texts_hash):
    s = json.dumps({"m": model_name, "p": pooling, "l": layer_strategy,
                     "L": int(max_len), "h": texts_hash}, sort_keys=True)
    return hashlib.md5(s.encode("utf-8")).hexdigest()

# ============================================================
# LOAD LABELS
# ============================================================
print("=" * 80)
print("LOADING DATA")
print("=" * 80)

df0 = pd.read_excel(EXCEL_PATH)
cols = ["decision_id", "chunk_id", "eval_A1", "eval_A2", "eval_A3"]
df0 = df0[cols].copy()

def extract(x):
    if pd.isna(x): return np.nan
    s = str(x).lower()
    if re.search(r"\boui\b", s) and not re.search(r"\bnon\b", s): return "oui"
    if re.search(r"\bnon\b", s) and not re.search(r"\boui\b", s): return "non"
    return np.nan

df0["a"] = df0["eval_A1"].apply(extract)
df0["t"] = df0["eval_A2"].apply(extract)
df0["s"] = df0["eval_A3"].apply(extract)

def resolve(row):
    a, t, s = row["a"], row["t"], row["s"]
    if pd.notna(a) and pd.notna(t) and a == t:
        return a, "agree"
    if pd.notna(a) and pd.notna(t) and a != t and pd.notna(s):
        return s, "disagree"
    return np.nan, "other"

res = df0.apply(resolve, axis=1)
df0["label_str"] = res.apply(lambda x: x[0])
df0["case"] = res.apply(lambda x: x[1])

df0 = df0[df0["label_str"].isin(["oui", "non"])].copy()
df0["label"] = df0["label_str"].map({"oui": 1, "non": 0}).astype(int)
df0 = df0.reset_index(drop=True)

y_true = df0["label"].values
print(f"Total samples: {len(df0)}")
print(f"  - OUI (1): {(y_true == 1).sum()}")
print(f"  - NON (0): {(y_true == 0).sum()}")

# ============================================================
# LOAD PROBAS
# ============================================================
print("\n" + "=" * 80)
print("LOADING PROBABILITIES")
print("=" * 80)

probas = {}

# Individual models
for name, fname in MODELS.items():
    path = os.path.join(OUTPUT_PATH, fname)
    if os.path.exists(path):
        probas[name] = np.load(path)
        print(f"  ✓ {name}")
    else:
        print(f"  ✗ {name} NOT FOUND")

# TF-IDF
tfidf_path = os.path.join(OUTPUT_PATH, TFIDF_CSV)
if os.path.exists(tfidf_path):
    tfidf_df = pd.read_csv(tfidf_path)
    probas["TF-IDF"] = tfidf_df["proba_oui"].values
    print(f"  ✓ TF-IDF")

# ============================================================
# LAWMA: Load from saved probas or recompute from cached embeddings
# Best config: MLP1 | cfg1 | mean | last
# ============================================================
print("\n" + "-" * 80)
print("LAWMA — Best config: MLP1 | cfg1 | mean | last")
print("-" * 80)

lawma_proba_path = os.path.join(OUTPUT_PATH, "proba_oof_LawMA_BEST_cfg1_mean_last_MLP1.npy")

if os.path.exists(lawma_proba_path):
    probas["LawMA"] = np.load(lawma_proba_path)
    print(f"  ✓ LawMA (loaded from saved probas)")
else:
    # Recompute from cached embeddings
    df_full = pd.read_excel(EXCEL_PATH)
    cols_needed = ["decision_id", "chunk_id", "pred_art", "text", "article_text",
                   "eval_A1", "eval_A2", "eval_A3"]
    df_full = df_full[[c for c in cols_needed if c in df_full.columns]].copy()
    df_full["a"] = df_full["eval_A1"].apply(extract)
    df_full["t"] = df_full["eval_A2"].apply(extract)
    df_full["s"] = df_full["eval_A3"].apply(extract)

    def resolve_label(r):
        a, t, s = r["a"], r["t"], r["s"]
        if pd.notna(a) and pd.notna(t) and a == t: return a
        if pd.notna(a) and pd.notna(t) and a != t and pd.notna(s): return s
        return np.nan

    df_full["label_str"] = df_full.apply(resolve_label, axis=1)
    df_full = df_full[df_full["label_str"].isin(["oui", "non"])].copy()
    df_full["label"] = df_full["label_str"].map({"oui": 1, "non": 0}).astype(int)

    df_cv, _ = make_grouped_folds(df_full, N_SPLITS, SEED)

    # cfg1 texts: article + [SEP] + chunk
    texts = (df_cv["article_text"].astype(str).str.strip()
             + " [SEP] "
             + df_cv["text"].astype(str).str.strip()).tolist()

    # Find cached embeddings
    LAWMA_MODEL = "ricdomolm/lawma-8b"
    texts_hash = _hash_texts(texts)
    key = _cache_key(LAWMA_MODEL, "mean", "last", 512, texts_hash)
    emb_fname = f"emb_{key}.npy"

    X_lawma = None
    for cache_dir in LAWMA_CACHE_DIRS:
        cache_path = os.path.join(cache_dir, emb_fname)
        if os.path.exists(cache_path):
            X_lawma = np.load(cache_path)
            print(f"  ⚡ Embeddings loaded from: {cache_dir}")
            break

    if X_lawma is None:
        print("  ✗ Embeddings NOT FOUND. Run comparison script first.")
    else:
        print(f"  ✓ Embeddings shape: {X_lawma.shape}")

        # OOF with MLP1
        y_cv = df_cv["label"].values
        proba_oof = np.full(len(df_cv), np.nan, dtype=float)

        for fold in range(N_SPLITS):
            tr = (df_cv["fold"] != fold).values
            te = (df_cv["fold"] == fold).values
            scaler = StandardScaler()
            Xtr = scaler.fit_transform(X_lawma[tr])
            Xte = scaler.transform(X_lawma[te])
            clf = MLPClassifier(
                hidden_layer_sizes=(256,), alpha=1e-3,
                learning_rate_init=5e-4, max_iter=300,
                early_stopping=True, n_iter_no_change=15,
                random_state=SEED
            )
            clf.fit(Xtr, y_cv[tr])
            proba_oof[te] = clf.predict_proba(Xte)[:, 1]

        assert np.isfinite(proba_oof).all()
        np.save(lawma_proba_path, proba_oof)
        print(f"  ✓ OOF probas saved: {lawma_proba_path}")
        probas["LawMA"] = proba_oof

# Compute best threshold for LawMA
if "LawMA" in probas:
    THR_GRID = np.linspace(0.05, 0.95, 181)
    best_thr, best_mcc = 0.5, -1e9
    for t in THR_GRID:
        mcc = matthews_corrcoef(y_true, (probas["LawMA"] >= t).astype(int))
        if mcc > best_mcc:
            best_thr, best_mcc = float(t), float(mcc)
    BEST_THRESHOLDS["LawMA"] = best_thr
    print(f"  ✓ Best threshold: {best_thr:.6f} (MCC={best_mcc:.4f})")

# Best Ensemble
best_ens_path = os.path.join(BEST_OUTPUT_PATH, "proba_oof_BEST_ENSEMBLE_nested_cv.npy")
if os.path.exists(best_ens_path):
    probas["BEST_ENSEMBLE"] = np.load(best_ens_path)
    print(f"  ✓ BEST_ENSEMBLE")
else:
    print(f"  ✗ BEST_ENSEMBLE NOT FOUND")

print(f"\n{len(probas)} models loaded")

# ============================================================
# COMPUTE METRICS (F1 positive class + Accuracy)
# ============================================================
print("\n" + "=" * 80)
print("COMPUTING METRICS (F1 positive class)")
print("=" * 80)

results = []

for model_name, p_oui in probas.items():
    thr = BEST_THRESHOLDS.get(model_name, 0.50)
    y_pred = (p_oui >= thr).astype(int)

    # Positive class metrics (pos_label=1)
    p_pos = precision_score(y_true, y_pred, pos_label=1, average='binary', zero_division=0)
    r_pos = recall_score(y_true, y_pred, pos_label=1, average='binary', zero_division=0)
    f1_pos = f1_score(y_true, y_pred, pos_label=1, average='binary', zero_division=0)

    # MCC and Accuracy
    mcc = matthews_corrcoef(y_true, y_pred)
    acc = accuracy_score(y_true, y_pred)

    # Confusion matrix
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred).ravel()

    head = HEADS.get(model_name, "—")

    results.append({
        "Model": model_name,
        "Head": head,
        "Threshold": thr,
        "P": p_pos,
        "R": r_pos,
        "F1": f1_pos,
        "Acc": acc,
        "MCC": mcc,
        "TP": tp, "TN": tn, "FP": fp, "FN": fn,
    })

# ============================================================
# DISPLAY RESULTS
# ============================================================
results_df = pd.DataFrame(results)
results_df = results_df.sort_values("MCC", ascending=False).reset_index(drop=True)

print("\n" + "=" * 80)
print("RESULTS - Positive class metrics (sorted by MCC)")
print("=" * 80)

print("\n📊 Main Metrics (positive class):\n")
print(f"{'Model':<18} {'Head':<6} {'P':>8} {'R':>8} {'F1':>8} {'Acc':>8} {'MCC':>8}")
print("-" * 72)
for _, row in results_df.iterrows():
    print(f"{row['Model']:<18} {row['Head']:<6} {row['P']:>8.2f} {row['R']:>8.2f} {row['F1']:>8.2f} {row['Acc']:>8.2f} {row['MCC']:>8.2f}")

print("\n\n📊 Confusion Matrices:\n")
print(f"{'Model':<18} {'TP':>6} {'TN':>6} {'FP':>6} {'FN':>6}")
print("-" * 50)
for _, row in results_df.iterrows():
    print(f"{row['Model']:<18} {row['TP']:>6} {row['TN']:>6} {row['FP']:>6} {row['FN']:>6}")

# ============================================================
# LATEX TABLE 1: Model, Head, F1, Acc, MCC
# ============================================================
print("\n\n" + "=" * 80)
print("LATEX TABLE 1 — F1 / Acc / MCC (positive class)")
print("=" * 80)

print("""
\\begin{table}[t]
\\centering
\\small
\\begin{tabular}{llccc}
\\toprule
\\textbf{Model} & \\textbf{Head} & \\textbf{F1} & \\textbf{Acc} & \\textbf{MCC} \\\\
\\midrule""")

for _, row in results_df.iterrows():
    model = row['Model']
    if model == "BEST_ENSEMBLE":
        continue
    display = DISPLAY_NAMES.get(model, model)
    print(f"{display} & {row['Head']} & {row['F1']:.2f} & {row['Acc']:.2f} & {row['MCC']:.2f} \\\\")

print("""\\bottomrule
\\end{tabular}
\\caption{Supervised classification results (5-fold CV, F1 on positive class). Configuration details in Appendix~\\ref{app:gridsearch}.}
\\label{tab:supervised}
\\end{table}
""")

# ============================================================
# LATEX TABLE 2: Confusion Matrices
# ============================================================
print("\n" + "=" * 80)
print("LATEX TABLE 2 — Confusion Matrices")
print("=" * 80)

print("""
\\begin{table}[h]
\\centering
\\small
\\begin{tabular}{lrrrr}
\\toprule
\\textbf{Model} & \\textbf{TP} & \\textbf{TN} & \\textbf{FP} & \\textbf{FN} \\\\
\\midrule""")

for _, row in results_df.iterrows():
    model = row['Model']
    display = DISPLAY_NAMES.get(model, model)
    print(f"{display} & {row['TP']} & {row['TN']} & {row['FP']} & {row['FN']} \\\\")

print("""\\bottomrule
\\end{tabular}
\\caption{Confusion matrices for supervised models (5-fold CV, optimized threshold).}
\\label{tab:supervised-cm}
\\end{table}
""")

# ============================================================
# LATEX TABLE 3: Full metrics — Thr, P, R, F1, MCC (positive class)
# ============================================================
print("\n" + "=" * 80)
print("LATEX TABLE 3 — Thr / P / R / F1 / MCC (positive class)")
print("=" * 80)

print("""
\\begin{table}[h]
\\centering
\\small
\\begin{tabular}{lccccc}
\\toprule
\\textbf{Model} & \\textbf{Thr} & \\textbf{P} & \\textbf{R} & \\textbf{F1} & \\textbf{MCC} \\\\
\\midrule""")

# Individual models (exclude ensemble)
for _, row in results_df.iterrows():
    model = row['Model']
    if model == "BEST_ENSEMBLE":
        continue
    display = DISPLAY_NAMES.get(model, model)
    print(f"{display} & {row['Threshold']:.2f} & {row['P']:.2f} & {row['R']:.2f} & {row['F1']:.2f} & {row['MCC']:.2f} \\\\")

# Ensemble with midrule separator
ens_row = results_df[results_df["Model"] == "BEST_ENSEMBLE"]
if len(ens_row) > 0:
    row = ens_row.iloc[0]
    display = DISPLAY_NAMES.get(row['Model'], row['Model'])
    print(f"\\midrule")
    print(f"{display} & {row['Threshold']:.2f} & {row['P']:.2f} & {row['R']:.2f} & {row['F1']:.2f} & {row['MCC']:.2f} \\\\")

print("""\\bottomrule
\\end{tabular}
\\caption{Supervised metrics (5-fold CV, positive class). Thr = threshold, P = precision, R = recall.}
\\label{tab:supervised-full}
\\end{table}
""")


# # p(FP | disagree) > P (FP | agree)


In [ ]:
# ============================================================
# A1 — TEST HYPOTHESIS: DISAGREE -> FP ?
# Uses OOF probas + best ensemble from NESTED CV
# ============================================================

import os, re, warnings, json
import numpy as np
import pandas as pd
import statsmodels.api as sm
import matplotlib.pyplot as plt

warnings.filterwarnings("ignore")

# ============================================================
# CONFIG
# ============================================================
BASE_PATH = "artifacts"  # not shipped — see DATA.md
EXCEL_PATH = "DATA/outputs/benchmark.csv"

OUTPUT_PATH = os.path.join(BASE_PATH, "oof_proba_final")
BEST_OUTPUT_PATH = os.path.join(OUTPUT_PATH, "best_models")
ENSEMBLE_OUTPUT = os.path.join(OUTPUT_PATH, "ensemble_search_nested_cv")

# Output for A1
A1_OUTPUT = os.path.join(OUTPUT_PATH, "A1_analysis_nested_cv")
os.makedirs(A1_OUTPUT, exist_ok=True)

SEED = 42
N_SPLITS = 5
N_PERM = 20000

np.random.seed(SEED)

# ============================================================
# MODELS AND FILES
# ============================================================
MODELS = {
    "CamemBERT": "proba_oof_CamemBERT_BEST_cfg3_mean_avg_last_k_4_MLP2.npy",
    "CamemBERTav2": "proba_oof_CamemBERTav2_BEST_cfg1_mean_avg_last_k_4_MLP1.npy",
    "JuriBERT": "proba_oof_JuriBERT-base_BEST_cfg1_cls_last_LR.npy",
    "LLaMA": "proba_oof_LLaMA_BEST_cfg2_mean_layer_-2_MLP1.npy",
    "SAUL": "proba_oof_SAUL_BEST_cfg3_mean_last_k_4_LR.npy",
    "ST-MiniLM": "proba_oof_ST-MiniLM_BEST_cfg3_mean_layer_-2_MLP2.npy",
    "ST-MPNet": "proba_oof_ST-MPNet_BEST_cfg1_mean_avg_last_k_4_MLP2.npy",
}
TFIDF_CSV = "tfidf_oof_probas.csv"

# Best thresholds per model (exact values from latest search - 2000 points grid)
BEST_THRESHOLDS = {
    "CamemBERT": 0.482666,
    "CamemBERTav2": 0.587369,
    "JuriBERT": 0.739345,
    "LLaMA": 0.614332,
    "SAUL": 0.793622,
    "ST-MiniLM": 0.539045,
    "ST-MPNet": 0.631491,
    "TF-IDF": 0.471811,
}

# MCC per model (best threshold)
BEST_MCC = {
    "CamemBERT": 0.4281,
    "CamemBERTav2": 0.4160,
    "JuriBERT": 0.3706,
    "LLaMA": 0.4633,
    "SAUL": 0.4716,
    "ST-MiniLM": 0.3685,
    "ST-MPNet": 0.4292,
    "TF-IDF": 0.4148,
}

# BEST ENSEMBLE (Nested CV - Stacking_LR, MCC=0.5258)
# CamemBERTav2+JuriBERT+LLaMA+SAUL
BEST_ENSEMBLE_MODELS = ["CamemBERTav2", "JuriBERT", "LLaMA", "SAUL"]
BEST_ENSEMBLE_METHOD = "Stacking_LR"
BEST_ENSEMBLE_THRESHOLD = 0.610120
BEST_ENSEMBLE_MCC = 0.5258

# ============================================================
# LOAD DATA
# ============================================================
print("=" * 80)
print("LOADING DATA")
print("=" * 80)

# Labels + case (agree/disagree)
df0 = pd.read_excel(EXCEL_PATH)
cols = ["decision_id", "chunk_id", "eval_A1", "eval_A2", "eval_A3"]
df0 = df0[cols].copy()

def extract(x):
    if pd.isna(x): return np.nan
    s = str(x).lower()
    if re.search(r"\boui\b", s) and not re.search(r"\bnon\b", s): return "oui"
    if re.search(r"\bnon\b", s) and not re.search(r"\boui\b", s): return "non"
    return np.nan

df0["a"] = df0["eval_A1"].apply(extract)
df0["t"] = df0["eval_A2"].apply(extract)
df0["s"] = df0["eval_A3"].apply(extract)

def resolve(row):
    a, t, s = row["a"], row["t"], row["s"]
    if pd.notna(a) and pd.notna(t) and a == t:
        return a, "agree"
    if pd.notna(a) and pd.notna(t) and a != t and pd.notna(s):
        return s, "disagree"
    return np.nan, "other"

res = df0.apply(resolve, axis=1)
df0["label_str"] = res.apply(lambda x: x[0])
df0["case"] = res.apply(lambda x: x[1])

df0 = df0[df0["label_str"].isin(["oui", "non"])].copy()
df0["label"] = df0["label_str"].map({"oui": 1, "non": 0}).astype(int)

# Folds
rng = np.random.default_rng(SEED)
groups = df0.groupby("decision_id").size().to_dict()
uniq = list(groups.keys())
rng.shuffle(uniq)
uniq = sorted(uniq, key=lambda g: groups[g], reverse=True)
loads = np.zeros(N_SPLITS, dtype=int)
g2f = {}
for g in uniq:
    f = int(loads.argmin())
    g2f[g] = f
    loads[f] += groups[g]
df0["fold"] = df0["decision_id"].map(g2f).astype(int)
df0 = df0.reset_index(drop=True)

print(f"Total: {len(df0)} samples")
print(f"  - agree: {(df0['case'] == 'agree').sum()}")
print(f"  - disagree: {(df0['case'] == 'disagree').sum()}")

# ============================================================
# LOAD PROBAS
# ============================================================
print("\nLoading probabilities...")

probas = {}

# Individual models
for name, fname in MODELS.items():
    path = os.path.join(OUTPUT_PATH, fname)
    if os.path.exists(path):
        probas[name] = np.load(path)
        print(f"  ✓ {name} (thr={BEST_THRESHOLDS[name]:.4f})")

# TF-IDF
tfidf_path = os.path.join(OUTPUT_PATH, TFIDF_CSV)
if os.path.exists(tfidf_path):
    tfidf_df = pd.read_csv(tfidf_path)
    probas["TF-IDF"] = tfidf_df["proba_oui"].values
    print(f"  ✓ TF-IDF (thr={BEST_THRESHOLDS['TF-IDF']:.4f})")

# Best Ensemble (Nested CV)
best_ens_path = os.path.join(BEST_OUTPUT_PATH, "proba_oof_BEST_ENSEMBLE_nested_cv.npy")
best_config_path = os.path.join(BEST_OUTPUT_PATH, "detailed_metrics_best_ensemble.json")

if os.path.exists(best_ens_path):
    probas["BEST_ENSEMBLE"] = np.load(best_ens_path)

    # Load config if available
    if os.path.exists(best_config_path):
        with open(best_config_path, "r") as f:
            best_config = json.load(f)
        # Use threshold from config
        BEST_THRESHOLDS["BEST_ENSEMBLE"] = best_config.get("ensemble", {}).get("threshold", BEST_ENSEMBLE_THRESHOLD)
        loaded_models = best_config.get("ensemble", {}).get("models", BEST_ENSEMBLE_MODELS)
        loaded_method = best_config.get("ensemble", {}).get("method", BEST_ENSEMBLE_METHOD)
        print(f"  ✓ BEST_ENSEMBLE (thr={BEST_THRESHOLDS['BEST_ENSEMBLE']:.6f})")
        print(f"    Method: {loaded_method}")
        print(f"    Models: {'+'.join(loaded_models)}")
    else:
        BEST_THRESHOLDS["BEST_ENSEMBLE"] = BEST_ENSEMBLE_THRESHOLD
        print(f"  ✓ BEST_ENSEMBLE (thr={BEST_ENSEMBLE_THRESHOLD:.6f})")
        print(f"    Method: {BEST_ENSEMBLE_METHOD}")
        print(f"    Models: {'+'.join(BEST_ENSEMBLE_MODELS)}")
else:
    print(f"  ✗ BEST_ENSEMBLE not found at {best_ens_path}")
    print("    Skipping ensemble analysis...")

print(f"\n{len(probas)} models loaded")

# Display threshold summary
print("\n📊 Thresholds used:")
for name in probas.keys():
    thr = BEST_THRESHOLDS.get(name, 0.5)
    mcc = BEST_MCC.get(name, BEST_ENSEMBLE_MCC if name == "BEST_ENSEMBLE" else "N/A")
    print(f"  {name:20s}: thr={thr:.4f}, MCC={mcc}")

# ============================================================
# HELPERS
# ============================================================
def norm_cdf(x):
    import math
    return 0.5 * (1.0 + math.erf(x / math.sqrt(2.0)))

def cluster_perm_test(values, group_labels, cluster_ids, n_perm=N_PERM, seed=SEED):
    """Cluster permutation test for diff of means."""
    rng = np.random.default_rng(seed)
    values = np.asarray(values, dtype=float)
    group_labels = np.asarray(group_labels, dtype=int)
    cluster_ids = np.asarray(cluster_ids)

    uniq = np.unique(cluster_ids)
    cl_means = []
    cl_group = []
    for c in uniq:
        m = cluster_ids == c
        cl_means.append(values[m].mean())
        cl_group.append(group_labels[m][0])
    cl_means = np.asarray(cl_means)
    cl_group = np.asarray(cl_group)

    obs = cl_means[cl_group == 1].mean() - cl_means[cl_group == 0].mean()

    idx = np.arange(len(uniq))
    nA = int((cl_group == 1).sum())

    perm_stats = np.empty(n_perm, dtype=float)
    for i in range(n_perm):
        perm = rng.permutation(idx)
        A_idx = perm[:nA]
        B_idx = perm[nA:]
        perm_stats[i] = cl_means[A_idx].mean() - cl_means[B_idx].mean()

    p = float(np.mean(np.abs(perm_stats) >= np.abs(obs)))
    ci = (float(np.percentile(perm_stats, 2.5)), float(np.percentile(perm_stats, 97.5)))
    return float(obs), p, ci

# ============================================================
# A1 ANALYSIS: DISAGREE -> FP ?
# ============================================================
print("\n" + "=" * 80)
print("A1 — DISAGREE -> FP ? (true NON only)")
print("=" * 80)

a1_rows = []

for model_name, p_oui in probas.items():
    thr = BEST_THRESHOLDS.get(model_name, 0.50)
    y_pred = (p_oui >= thr).astype(int)
    y_true = df0["label"].values

    # Build prediction df
    pred_df = pd.DataFrame({
        "decision_id": df0["decision_id"].values,
        "case": df0["case"].values,
        "y_true": y_true,
        "y_pred": y_pred,
    })
    pred_df["is_FP"] = ((pred_df["y_true"] == 0) & (pred_df["y_pred"] == 1)).astype(int)

    # Filter: true NON only
    true_non = pred_df[pred_df["y_true"] == 0].copy()
    true_non["case_binary"] = (true_non["case"] == "disagree").astype(int)

    n_agree = (true_non["case"] == "agree").sum()
    n_disagree = (true_non["case"] == "disagree").sum()
    fp_agree = true_non[true_non["case"] == "agree"]["is_FP"].mean()
    fp_disagree = true_non[true_non["case"] == "disagree"]["is_FP"].mean()

    if true_non["is_FP"].nunique() < 2 or true_non["case_binary"].nunique() < 2:
        a1_rows.append({
            "model": model_name,
            "threshold": thr,
            "n_true_non": len(true_non),
            "n_agree": n_agree,
            "n_disagree": n_disagree,
            "FP_rate_agree": fp_agree,
            "FP_rate_disagree": fp_disagree,
            "OR": np.nan,
            "OR_CI95": "NA",
            "p_cluster": np.nan,
            "FP_rate_diff": np.nan,
            "perm_p": np.nan,
        })
        print(f"  {model_name}: degenerate, skipped.")
        continue

    # Logistic regression with cluster-robust SE
    X = sm.add_constant(true_non[["case_binary"]])
    y = true_non["is_FP"].astype(int)
    groups = true_non["decision_id"].astype(str).values

    try:
        logit = sm.Logit(y, X)
        result = logit.fit(cov_type="cluster", cov_kwds={"groups": groups}, disp=0)
        coef = float(result.params["case_binary"])
        se = float(result.bse["case_binary"])
        p_cluster = float(result.pvalues["case_binary"])
    except Exception as e:
        logit = sm.Logit(y, X).fit(disp=0)
        coef = float(logit.params["case_binary"])
        se = float(logit.bse["case_binary"])
        z = coef / se
        p_cluster = 2 * (1 - norm_cdf(abs(z)))

    OR = float(np.exp(coef))
    ci_lo = float(np.exp(coef - 1.96 * se))
    ci_hi = float(np.exp(coef + 1.96 * se))

    # Permutation test
    obs_diff, p_perm, _ = cluster_perm_test(
        values=true_non["is_FP"].values,
        group_labels=true_non["case_binary"].values,
        cluster_ids=true_non["decision_id"].astype(str).values,
    )

    a1_rows.append({
        "model": model_name,
        "threshold": thr,
        "n_true_non": len(true_non),
        "n_agree": n_agree,
        "n_disagree": n_disagree,
        "FP_rate_agree": fp_agree,
        "FP_rate_disagree": fp_disagree,
        "OR": OR,
        "OR_CI95": f"[{ci_lo:.2f}, {ci_hi:.2f}]",
        "p_cluster": p_cluster,
        "FP_rate_diff": obs_diff,
        "perm_p": p_perm,
    })

    print(f"  {model_name}: OR={OR:.2f} {f'[{ci_lo:.2f}, {ci_hi:.2f}]':20s} p_cluster={p_cluster:.4f} | FP_diff={obs_diff:+.3f} perm_p={p_perm:.4f}")

# ============================================================
# RESULTS
# ============================================================
print("\n" + "=" * 80)
print("RESULTS")
print("=" * 80)

a1_df = pd.DataFrame(a1_rows).sort_values("p_cluster", ascending=True)
print(a1_df.round(4).to_string(index=False))

# Save
a1_df.to_csv(os.path.join(A1_OUTPUT, "A1_results.csv"), index=False)
print(f"\n✓ Saved: {os.path.join(A1_OUTPUT, 'A1_results.csv')}")

# Save config used
config_used = {
    "thresholds": {k: float(v) for k, v in BEST_THRESHOLDS.items() if k in probas},
    "best_ensemble": {
        "models": BEST_ENSEMBLE_MODELS,
        "method": BEST_ENSEMBLE_METHOD,
        "threshold": BEST_ENSEMBLE_THRESHOLD,
        "mcc": BEST_ENSEMBLE_MCC,
        "source_file": "proba_oof_BEST_ENSEMBLE_nested_cv.npy",
    },
    "seed": SEED,
    "n_perm": N_PERM,
}
with open(os.path.join(A1_OUTPUT, "A1_config.json"), "w") as f:
    json.dump(config_used, f, indent=2)
print(f"✓ Saved: {os.path.join(A1_OUTPUT, 'A1_config.json')}")

# ============================================================
# FIGURES (ALL IN ENGLISH)
# ============================================================
print("\nGenerating figures...")

# Figure 1: Odds Ratios
fig, ax = plt.subplots(figsize=(12, 5))
a1_plot = a1_df[a1_df["OR"].notna()].copy()

ci = a1_plot["OR_CI95"].str.extract(r"\[(.*), (.*)\]")
ci = ci.apply(pd.to_numeric, errors="coerce")

x = np.arange(len(a1_plot))

# Highlight best ensemble in red
colors = ['#e74c3c' if m == "BEST_ENSEMBLE" else '#3498db' for m in a1_plot["model"]]
for i, (xi, yi, lo, hi, c) in enumerate(zip(x, a1_plot["OR"].values, ci[0].values, ci[1].values, colors)):
    ax.errorbar(x=xi, y=yi, yerr=[[yi - lo], [hi - yi]], fmt="o", capsize=4, color=c, ecolor=c, markersize=8)

ax.axhline(1.0, linestyle="--", linewidth=1, color='gray', alpha=0.7)
ax.set_xticks(x)
ax.set_xticklabels(a1_plot["model"].tolist(), rotation=45, ha="right")
ax.set_ylabel("Odds Ratio (FP | disagree vs agree)", fontsize=11)
ax.set_xlabel("Model", fontsize=11)
ax.set_title("Effect of Annotator Disagreement on False Positive Rate\n(Among true negative samples)", fontsize=12)

# Add legend
from matplotlib.lines import Line2D
legend_elements = [
    Line2D([0], [0], marker='o', color='w', markerfacecolor='#3498db', markersize=10, label='Individual models'),
    Line2D([0], [0], marker='o', color='w', markerfacecolor='#e74c3c', markersize=10, label='Best ensemble'),
    Line2D([0], [0], linestyle='--', color='gray', label='OR = 1 (no effect)')
]
ax.legend(handles=legend_elements, loc='upper right')

plt.tight_layout()
fig.savefig(os.path.join(A1_OUTPUT, "fig_A1_OR.png"), dpi=150, bbox_inches='tight')
fig.savefig(os.path.join(A1_OUTPUT, "fig_A1_OR.pdf"), bbox_inches='tight')
plt.show()
print(f"✓ fig_A1_OR.png / .pdf")

# Figure 2: FP rates comparison
fig, ax = plt.subplots(figsize=(12, 5))
x = np.arange(len(a1_df))
width = 0.35

bars1 = ax.bar(x - width/2, a1_df["FP_rate_agree"], width, label="Agreement", color='#2ecc71', edgecolor='white')
bars2 = ax.bar(x + width/2, a1_df["FP_rate_disagree"], width, label="Disagreement", color='#e74c3c', edgecolor='white')

# Display percentages above bars
for i, (bar, val) in enumerate(zip(bars1, a1_df["FP_rate_agree"])):
    if not np.isnan(val):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
                f'{val*100:.1f}%', ha='center', va='bottom', fontsize=9)

for i, (bar, val) in enumerate(zip(bars2, a1_df["FP_rate_disagree"])):
    if not np.isnan(val):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
                f'{val*100:.1f}%', ha='center', va='bottom', fontsize=9)

ax.set_xticks(x)
ax.set_xticklabels(a1_df["model"].tolist(), rotation=45, ha="right")
ax.set_ylabel("False Positive Rate", fontsize=11)
ax.set_xlabel("Model", fontsize=11)
ax.set_title("False Positive Rate by Annotator Agreement Status\n(Among true negative samples)", fontsize=12)
ax.legend(loc='upper right')
ax.set_ylim(0, max(a1_df["FP_rate_disagree"].max(), a1_df["FP_rate_agree"].max()) * 1.2)

plt.tight_layout()
fig.savefig(os.path.join(A1_OUTPUT, "fig_A1_FP_rates.png"), dpi=150, bbox_inches='tight')
fig.savefig(os.path.join(A1_OUTPUT, "fig_A1_FP_rates.pdf"), bbox_inches='tight')
plt.show()
print(f"✓ fig_A1_FP_rates.png / .pdf")

# Figure 3: Summary forest plot with p-values
fig, ax = plt.subplots(figsize=(10, 6))
a1_sorted = a1_df[a1_df["OR"].notna()].sort_values("OR", ascending=True).reset_index(drop=True)

y_pos = np.arange(len(a1_sorted))
ci = a1_sorted["OR_CI95"].str.extract(r"\[(.*), (.*)\]")
ci = ci.apply(pd.to_numeric, errors="coerce")

colors = ['#e74c3c' if m == "BEST_ENSEMBLE" else '#3498db' for m in a1_sorted["model"]]

for i, (yi, or_val, lo, hi, c, p) in enumerate(zip(y_pos, a1_sorted["OR"].values, ci[0].values, ci[1].values, colors, a1_sorted["p_cluster"].values)):
    ax.errorbar(x=or_val, y=yi, xerr=[[or_val - lo], [hi - or_val]], fmt="o", capsize=4, color=c, ecolor=c, markersize=8)
    # Add p-value annotation
    sig = "***" if p < 0.001 else "**" if p < 0.01 else "*" if p < 0.05 else ""
    ax.text(hi + 0.1, yi, f"p={p:.3f}{sig}", va='center', fontsize=9)

ax.axvline(1.0, linestyle="--", linewidth=1, color='gray', alpha=0.7)
ax.set_yticks(y_pos)
ax.set_yticklabels(a1_sorted["model"].tolist())
ax.set_xlabel("Odds Ratio (95% CI)", fontsize=11)
ax.set_title("Forest Plot: Effect of Disagreement on False Positives\n(Cluster-robust logistic regression)", fontsize=12)

# Extend x-axis to fit p-values
ax.set_xlim(0, ax.get_xlim()[1] + 1.5)

plt.tight_layout()
fig.savefig(os.path.join(A1_OUTPUT, "fig_A1_forest.png"), dpi=150, bbox_inches='tight')
fig.savefig(os.path.join(A1_OUTPUT, "fig_A1_forest.pdf"), bbox_inches='tight')
plt.show()
print(f"✓ fig_A1_forest.png / .pdf")

print("\n" + "=" * 80)
print("DONE")
print("=" * 80)

# FP LLAwMA

In [ ]:
# ============================================================
# A1 — TEST HYPOTHESIS: DISAGREE -> FP ?
# Uses OOF probas + best ensemble from NESTED CV
# WITH LAWMA + NEW BEST ENSEMBLE (WeightedAvg rank #2)
# ============================================================

import os, re, warnings, json
import numpy as np
import pandas as pd
import statsmodels.api as sm
import matplotlib.pyplot as plt

warnings.filterwarnings("ignore")

# ============================================================
# CONFIG
# ============================================================
BASE_PATH = "artifacts"  # not shipped — see DATA.md
EXCEL_PATH = "DATA/outputs/benchmark.csv"

OUTPUT_PATH = os.path.join(BASE_PATH, "oof_proba_final")
BEST_OUTPUT_PATH = os.path.join(OUTPUT_PATH, "best_models")

# Output for A1
A1_OUTPUT = os.path.join(OUTPUT_PATH, "A1_analysis_nested_cv_with_lawma")
os.makedirs(A1_OUTPUT, exist_ok=True)

SEED = 42
N_SPLITS = 5
N_PERM = 20000
THR_GRID = np.linspace(0.15, 0.85, 500)

np.random.seed(SEED)

# ============================================================
# MODELS AND FILES
# ============================================================
MODELS = {
    "CamemBERT": "proba_oof_CamemBERT_BEST_cfg3_mean_avg_last_k_4_MLP2.npy",
    "CamemBERTav2": "proba_oof_CamemBERTav2_BEST_cfg1_mean_avg_last_k_4_MLP1.npy",
    "JuriBERT": "proba_oof_JuriBERT-base_BEST_cfg1_cls_last_LR.npy",
    "LLaMA": "proba_oof_LLaMA_BEST_cfg2_mean_layer_-2_MLP1.npy",
    "SAUL": "proba_oof_SAUL_BEST_cfg3_mean_last_k_4_LR.npy",
    "ST-MiniLM": "proba_oof_ST-MiniLM_BEST_cfg3_mean_layer_-2_MLP2.npy",
    "ST-MPNet": "proba_oof_ST-MPNet_BEST_cfg1_mean_avg_last_k_4_MLP2.npy",
}
TFIDF_CSV = "tfidf_oof_probas.csv"

# Lawma: best config = MLP1, cfg1, mean, last (MCC=0.4576)
LAWMA_PROBA_PATH = os.path.join(
    BASE_PATH, "outputs", "outputs_lawma_mlp1_mlp2_3cfg",
    "proba_oof_Lawma-8B__cfg1__mean__last__mlp1.npy"
)

# Best thresholds per model (exact values from latest search)
BEST_THRESHOLDS = {
    "CamemBERT": 0.482666,
    "CamemBERTav2": 0.587369,
    "JuriBERT": 0.739345,
    "LLaMA": 0.614332,
    "SAUL": 0.793622,
    "ST-MiniLM": 0.539045,
    "ST-MPNet": 0.631491,
    "TF-IDF": 0.471811,
    "Lawma": 0.845,  # from run: best thr=0.845 MCC=0.4576
}

# MCC per model (best threshold)
BEST_MCC = {
    "CamemBERT": 0.4281,
    "CamemBERTav2": 0.4160,
    "JuriBERT": 0.3706,
    "LLaMA": 0.4633,
    "SAUL": 0.4716,
    "ST-MiniLM": 0.3685,
    "ST-MPNet": 0.4292,
    "TF-IDF": 0.4148,
    "Lawma": 0.4576,
}


BEST_ENSEMBLE_MODELS = ["CamemBERTav2", "JuriBERT", "LLaMA", "SAUL"]
BEST_ENSEMBLE_METHOD = "Stacking_LR"
BEST_ENSEMBLE_THRESHOLD = 0.610120
BEST_ENSEMBLE_MCC = 0.5258

# NEW BEST ENSEMBLE: WeightedAvg rank #2
# CamemBERTav2+JuriBERT+SAUL+Lawma, MCC=0.5279
NEW_ENSEMBLE_MODELS = ["CamemBERTav2", "JuriBERT", "SAUL", "Lawma"]
NEW_ENSEMBLE_METHOD = "WeightedAvg"
NEW_ENSEMBLE_MCC = 0.5279

# ============================================================
# LOAD DATA
# ============================================================
print("=" * 80)
print("LOADING DATA")
print("=" * 80)

df0 = pd.read_excel(EXCEL_PATH)
cols = ["decision_id", "chunk_id", "eval_A1", "eval_A2", "eval_A3"]
df0 = df0[cols].copy()

def extract(x):
    if pd.isna(x): return np.nan
    s = str(x).lower()
    if re.search(r"\boui\b", s) and not re.search(r"\bnon\b", s): return "oui"
    if re.search(r"\bnon\b", s) and not re.search(r"\boui\b", s): return "non"
    return np.nan

df0["a"] = df0["eval_A1"].apply(extract)
df0["t"] = df0["eval_A2"].apply(extract)
df0["s"] = df0["eval_A3"].apply(extract)

def resolve(row):
    a, t, s = row["a"], row["t"], row["s"]
    if pd.notna(a) and pd.notna(t) and a == t:
        return a, "agree"
    if pd.notna(a) and pd.notna(t) and a != t and pd.notna(s):
        return s, "disagree"
    return np.nan, "other"

res = df0.apply(resolve, axis=1)
df0["label_str"] = res.apply(lambda x: x[0])
df0["case"] = res.apply(lambda x: x[1])

df0 = df0[df0["label_str"].isin(["oui", "non"])].copy()
df0["label"] = df0["label_str"].map({"oui": 1, "non": 0}).astype(int)

# Folds
rng = np.random.default_rng(SEED)
groups = df0.groupby("decision_id").size().to_dict()
uniq = list(groups.keys())
rng.shuffle(uniq)
uniq = sorted(uniq, key=lambda g: groups[g], reverse=True)
loads = np.zeros(N_SPLITS, dtype=int)
g2f = {}
for g in uniq:
    f = int(loads.argmin())
    g2f[g] = f
    loads[f] += groups[g]
df0["fold"] = df0["decision_id"].map(g2f).astype(int)
df0 = df0.reset_index(drop=True)

print(f"Total: {len(df0)} samples")
print(f"  - agree: {(df0['case'] == 'agree').sum()}")
print(f"  - disagree: {(df0['case'] == 'disagree').sum()}")

# ============================================================
# LOAD PROBAS
# ============================================================
print("\nLoading probabilities...")

probas = {}

# Individual models
for name, fname in MODELS.items():
    path = os.path.join(OUTPUT_PATH, fname)
    if os.path.exists(path):
        probas[name] = np.load(path)
        print(f"  ✓ {name} (thr={BEST_THRESHOLDS[name]:.4f})")

# TF-IDF
tfidf_path = os.path.join(OUTPUT_PATH, TFIDF_CSV)
if os.path.exists(tfidf_path):
    tfidf_df = pd.read_csv(tfidf_path)
    probas["TF-IDF"] = tfidf_df["proba_oui"].values
    print(f"  ✓ TF-IDF (thr={BEST_THRESHOLDS['TF-IDF']:.4f})")

# Lawma
lawma_loaded = False
for lpath in [LAWMA_PROBA_PATH]:
    if os.path.exists(lpath):
        probas["Lawma"] = np.load(lpath)
        print(f"  ✓ Lawma (thr={BEST_THRESHOLDS['Lawma']:.4f})")
        lawma_loaded = True
        break
if not lawma_loaded:
    for root, dirs, files in os.walk(os.path.join(BASE_PATH, "outputs")):
        for f in files:
            if "Lawma" in f and "cfg1" in f and "last" in f and "mlp1" in f and f.endswith(".npy"):
                probas["Lawma"] = np.load(os.path.join(root, f))
                print(f"  ✓ Lawma (found via search)")
                lawma_loaded = True
                break
        if lawma_loaded: break

# ============================================================
# REPRODUCE BEST ENSEMBLE OOF PROBAS
# WeightedAvg: CamemBERTav2+JuriBERT+SAUL+Lawma
# ============================================================
"""
print("\nReproducing best ensemble (WeightedAvg)...")

def fast_mcc(y_true, y_pred):
    tp = np.sum((y_true == 1) & (y_pred == 1))
    tn = np.sum((y_true == 0) & (y_pred == 0))
    fp = np.sum((y_true == 0) & (y_pred == 1))
    fn = np.sum((y_true == 1) & (y_pred == 0))
    num = float(tp * tn - fp * fn)
    den = np.sqrt(float((tp + fp) * (tp + fn) * (tn + fp) * (tn + fn)))
    return num / den if den > 0 else 0.0

def best_threshold_search(y_true, p_proba):
    best_thr, best_mcc = 0.5, -1
    for thr in THR_GRID:
        y_pred = (p_proba >= thr).astype(int)
        mcc = fast_mcc(y_true, y_pred)
        if mcc > best_mcc:
            best_mcc = mcc
            best_thr = thr
    return best_thr, best_mcc

def combo_to_seed(combo):
    s = "+".join(sorted(combo))
    return sum(ord(c) * (i + 1) for i, c in enumerate(s)) % (2**31)

# Build proba matrix for ensemble models
ens_models = NEW_ENSEMBLE_MODELS
ens_probas = np.column_stack([probas[m] for m in ens_models])
ens_idx = {m: i for i, m in enumerate(ens_models)}

y_all = df0["label"].values
folds_all = df0["fold"].values

combo_seed = combo_to_seed(tuple(ens_models))
n_combo = len(ens_models)
n = len(y_all)
p_oof_ensemble = np.zeros(n)
fold_details = []

for fold_id in range(N_SPLITS):
    tr_mask = folds_all != fold_id
    te_mask = folds_all == fold_id
    fold_seed = combo_seed + fold_id
    fold_rng = np.random.default_rng(fold_seed)

    y_train = y_all[tr_mask]
    P_train = ens_probas[tr_mask]

    best = {"mcc_train": -1, "weights": None, "thr": 0.5}

    # n_combo == 4 -> random search (same as original)
    for _ in range(1000):
        weights = tuple(fold_rng.dirichlet(np.ones(n_combo)))
        p_mix = sum(w * P_train[:, i] for w, i in zip(weights, range(n_combo)))
        thr, mcc = best_threshold_search(y_train, p_mix)
        if mcc > best["mcc_train"]:
            best = {"weights": weights, "thr": thr, "mcc_train": mcc}

    # Also try uniform
    weights = tuple(np.ones(n_combo) / n_combo)
    p_mix = sum(w * P_train[:, i] for w, i in zip(weights, range(n_combo)))
    thr, mcc = best_threshold_search(y_train, p_mix)
    if mcc > best["mcc_train"]:
        best = {"weights": weights, "thr": thr, "mcc_train": mcc}

    # Apply to test
    p_test = sum(w * ens_probas[te_mask, i] for w, i in zip(best["weights"], range(n_combo)))
    p_oof_ensemble[te_mask] = p_test

    w_str = ", ".join(f"{m}={w:.3f}" for m, w in zip(ens_models, best["weights"]))
    print(f"  Fold {fold_id}: thr_train={best['thr']:.3f} MCC_train={best['mcc_train']:.4f} | {w_str}")
    fold_details.append(best)

# Find best threshold on full OOF
ens_thr, ens_mcc = best_threshold_search(y_all, p_oof_ensemble)
print(f"  -> Ensemble OOF: thr={ens_thr:.6f} MCC={ens_mcc:.4f} (ref: {NEW_ENSEMBLE_MCC:.4f})")

probas["BEST_ENSEMBLE"] = p_oof_ensemble
BEST_THRESHOLDS["BEST_ENSEMBLE"] = ens_thr
BEST_MCC["BEST_ENSEMBLE"] = ens_mcc

print(f"\n{len(probas)} models loaded (incl. ensemble)")
"""

# ============================================================
# LOAD BEST ENSEMBLE (Stacking-LR, nested CV)
# ============================================================
best_ens_path = os.path.join(BEST_OUTPUT_PATH, "proba_oof_BEST_ENSEMBLE_nested_cv.npy")
if os.path.exists(best_ens_path):
    probas["BEST_ENSEMBLE"] = np.load(best_ens_path)
    BEST_THRESHOLDS["BEST_ENSEMBLE"] = BEST_ENSEMBLE_THRESHOLD
    BEST_MCC["BEST_ENSEMBLE"] = BEST_ENSEMBLE_MCC
    print(f"  ✓ BEST_ENSEMBLE (Stacking-LR, thr={BEST_ENSEMBLE_THRESHOLD:.6f})")
    print(f"    Models: {'+'.join(BEST_ENSEMBLE_MODELS)}")
else:
    print(f"  ✗ BEST_ENSEMBLE not found at {best_ens_path}")

print(f"\n{len(probas)} models loaded (incl. ensemble)")

# ============================================================
# HELPERS
# ============================================================
def norm_cdf(x):
    import math
    return 0.5 * (1.0 + math.erf(x / math.sqrt(2.0)))

def cluster_perm_test(values, group_labels, cluster_ids, n_perm=N_PERM, seed=SEED):
    rng = np.random.default_rng(seed)
    values = np.asarray(values, dtype=float)
    group_labels = np.asarray(group_labels, dtype=int)
    cluster_ids = np.asarray(cluster_ids)

    uniq = np.unique(cluster_ids)
    cl_means = []
    cl_group = []
    for c in uniq:
        m = cluster_ids == c
        cl_means.append(values[m].mean())
        cl_group.append(group_labels[m][0])
    cl_means = np.asarray(cl_means)
    cl_group = np.asarray(cl_group)

    obs = cl_means[cl_group == 1].mean() - cl_means[cl_group == 0].mean()

    idx = np.arange(len(uniq))
    nA = int((cl_group == 1).sum())

    perm_stats = np.empty(n_perm, dtype=float)
    for i in range(n_perm):
        perm = rng.permutation(idx)
        A_idx = perm[:nA]
        B_idx = perm[nA:]
        perm_stats[i] = cl_means[A_idx].mean() - cl_means[B_idx].mean()

    p = float(np.mean(np.abs(perm_stats) >= np.abs(obs)))
    ci = (float(np.percentile(perm_stats, 2.5)), float(np.percentile(perm_stats, 97.5)))
    return float(obs), p, ci

# ============================================================
# A1 ANALYSIS: DISAGREE -> FP ?
# ============================================================
print("\n" + "=" * 80)
print("A1 — DISAGREE -> FP ? (true NON only)")
print("=" * 80)

a1_rows = []

for model_name, p_oui in probas.items():
    thr = BEST_THRESHOLDS.get(model_name, 0.50)
    y_pred = (p_oui >= thr).astype(int)
    y_true = df0["label"].values

    pred_df = pd.DataFrame({
        "decision_id": df0["decision_id"].values,
        "case": df0["case"].values,
        "y_true": y_true,
        "y_pred": y_pred,
    })
    pred_df["is_FP"] = ((pred_df["y_true"] == 0) & (pred_df["y_pred"] == 1)).astype(int)

    true_non = pred_df[pred_df["y_true"] == 0].copy()
    true_non["case_binary"] = (true_non["case"] == "disagree").astype(int)

    n_agree = (true_non["case"] == "agree").sum()
    n_disagree = (true_non["case"] == "disagree").sum()
    fp_agree = true_non[true_non["case"] == "agree"]["is_FP"].mean()
    fp_disagree = true_non[true_non["case"] == "disagree"]["is_FP"].mean()

    if true_non["is_FP"].nunique() < 2 or true_non["case_binary"].nunique() < 2:
        a1_rows.append({
            "model": model_name,
            "threshold": thr,
            "n_true_non": len(true_non),
            "n_agree": n_agree,
            "n_disagree": n_disagree,
            "FP_rate_agree": fp_agree,
            "FP_rate_disagree": fp_disagree,
            "OR": np.nan,
            "OR_CI95": "NA",
            "p_cluster": np.nan,
            "FP_rate_diff": np.nan,
            "perm_p": np.nan,
        })
        print(f"  {model_name}: degenerate, skipped.")
        continue

    # Logistic regression with cluster-robust SE
    X = sm.add_constant(true_non[["case_binary"]])
    y = true_non["is_FP"].astype(int)
    cluster_groups = true_non["decision_id"].astype(str).values

    try:
        logit = sm.Logit(y, X)
        result = logit.fit(cov_type="cluster", cov_kwds={"groups": cluster_groups}, disp=0)
        coef = float(result.params["case_binary"])
        se = float(result.bse["case_binary"])
        p_cluster = float(result.pvalues["case_binary"])
    except Exception as e:
        logit = sm.Logit(y, X).fit(disp=0)
        coef = float(logit.params["case_binary"])
        se = float(logit.bse["case_binary"])
        z = coef / se
        p_cluster = 2 * (1 - norm_cdf(abs(z)))

    OR = float(np.exp(coef))
    ci_lo = float(np.exp(coef - 1.96 * se))
    ci_hi = float(np.exp(coef + 1.96 * se))

    # Permutation test
    obs_diff, p_perm, _ = cluster_perm_test(
        values=true_non["is_FP"].values,
        group_labels=true_non["case_binary"].values,
        cluster_ids=true_non["decision_id"].astype(str).values,
    )

    a1_rows.append({
        "model": model_name,
        "threshold": thr,
        "n_true_non": len(true_non),
        "n_agree": n_agree,
        "n_disagree": n_disagree,
        "FP_rate_agree": fp_agree,
        "FP_rate_disagree": fp_disagree,
        "OR": OR,
        "OR_CI95": f"[{ci_lo:.2f}, {ci_hi:.2f}]",
        "p_cluster": p_cluster,
        "FP_rate_diff": obs_diff,
        "perm_p": p_perm,
    })

    print(f"  {model_name}: OR={OR:.2f} {f'[{ci_lo:.2f}, {ci_hi:.2f}]':20s} p_cluster={p_cluster:.4f} | FP_diff={obs_diff:+.3f} perm_p={p_perm:.4f}")

# ============================================================
# RESULTS
# ============================================================
print("\n" + "=" * 80)
print("RESULTS")
print("=" * 80)

a1_df = pd.DataFrame(a1_rows).sort_values("p_cluster", ascending=True)
print(a1_df.round(4).to_string(index=False))

# Save
a1_df.to_csv(os.path.join(A1_OUTPUT, "A1_results.csv"), index=False)
print(f"\n✓ Saved: {os.path.join(A1_OUTPUT, 'A1_results.csv')}")

# Save config
config_used = {
    "thresholds": {k: float(v) for k, v in BEST_THRESHOLDS.items() if k in probas},
    "best_ensemble": {
        "models": BEST_ENSEMBLE_MODELS,
        "method": BEST_ENSEMBLE_METHOD,
        "threshold": BEST_ENSEMBLE_THRESHOLD,
        "mcc": BEST_ENSEMBLE_MCC,
        "source_file": "proba_oof_BEST_ENSEMBLE_nested_cv.npy",
    },
    "seed": SEED,
    "n_perm": N_PERM,
}
with open(os.path.join(A1_OUTPUT, "A1_config.json"), "w") as f:
    json.dump(config_used, f, indent=2)
print(f"✓ Saved: {os.path.join(A1_OUTPUT, 'A1_config.json')}")

# ============================================================
# FIGURES
# ============================================================
print("\nGenerating figures...")

# Figure 1: Odds Ratios
fig, ax = plt.subplots(figsize=(12, 5))
a1_plot = a1_df[a1_df["OR"].notna()].copy()

ci = a1_plot["OR_CI95"].str.extract(r"\[(.*), (.*)\]")
ci = ci.apply(pd.to_numeric, errors="coerce")

x = np.arange(len(a1_plot))
colors = ['#e74c3c' if m == "BEST_ENSEMBLE" else ('#9b59b6' if m == "Lawma" else '#3498db') for m in a1_plot["model"]]

for i, (xi, yi, lo, hi, c) in enumerate(zip(x, a1_plot["OR"].values, ci[0].values, ci[1].values, colors)):
    ax.errorbar(x=xi, y=yi, yerr=[[yi - lo], [hi - yi]], fmt="o", capsize=4, color=c, ecolor=c, markersize=8)

ax.axhline(1.0, linestyle="--", linewidth=1, color='gray', alpha=0.7)
ax.set_xticks(x)
ax.set_xticklabels(a1_plot["model"].tolist(), rotation=45, ha="right")
ax.set_ylabel("Odds Ratio (FP | disagree vs agree)", fontsize=11)
ax.set_xlabel("Model", fontsize=11)
ax.set_title("Effect of Annotator Disagreement on False Positive Rate\n(Among true negative samples)", fontsize=12)

from matplotlib.lines import Line2D
legend_elements = [
    Line2D([0], [0], marker='o', color='w', markerfacecolor='#3498db', markersize=10, label='Individual models'),
    Line2D([0], [0], marker='o', color='w', markerfacecolor='#9b59b6', markersize=10, label='Lawma (legal fine-tuned)'),
    Line2D([0], [0], marker='o', color='w', markerfacecolor='#e74c3c', markersize=10, label='Best ensemble'),
    Line2D([0], [0], linestyle='--', color='gray', label='OR = 1 (no effect)')
]
ax.legend(handles=legend_elements, loc='upper right')

plt.tight_layout()
fig.savefig(os.path.join(A1_OUTPUT, "fig_A1_OR.png"), dpi=150, bbox_inches='tight')
fig.savefig(os.path.join(A1_OUTPUT, "fig_A1_OR.pdf"), bbox_inches='tight')
plt.show()
print(f"✓ fig_A1_OR.png / .pdf")

# Figure 2: FP rates comparison
fig, ax = plt.subplots(figsize=(12, 5))
x = np.arange(len(a1_df))
width = 0.35

bars1 = ax.bar(x - width/2, a1_df["FP_rate_agree"], width, label="Agreement", color='#2ecc71', edgecolor='white')
bars2 = ax.bar(x + width/2, a1_df["FP_rate_disagree"], width, label="Disagreement", color='#e74c3c', edgecolor='white')

for bar, val in zip(bars1, a1_df["FP_rate_agree"]):
    if not np.isnan(val):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
                f'{val*100:.1f}%', ha='center', va='bottom', fontsize=9)

for bar, val in zip(bars2, a1_df["FP_rate_disagree"]):
    if not np.isnan(val):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
                f'{val*100:.1f}%', ha='center', va='bottom', fontsize=9)

ax.set_xticks(x)
ax.set_xticklabels(a1_df["model"].tolist(), rotation=45, ha="right")
ax.set_ylabel("False Positive Rate", fontsize=11)
ax.set_xlabel("Model", fontsize=11)
ax.set_title("False Positive Rate by Annotator Agreement Status\n(Among true negative samples)", fontsize=12)
ax.legend(loc='upper right')
ax.set_ylim(0, max(a1_df["FP_rate_disagree"].max(), a1_df["FP_rate_agree"].max()) * 1.2)

plt.tight_layout()
fig.savefig(os.path.join(A1_OUTPUT, "fig_A1_FP_rates.png"), dpi=150, bbox_inches='tight')
fig.savefig(os.path.join(A1_OUTPUT, "fig_A1_FP_rates.pdf"), bbox_inches='tight')
plt.show()
print(f"✓ fig_A1_FP_rates.png / .pdf")

# Figure 3: Forest plot
fig, ax = plt.subplots(figsize=(10, 7))
a1_sorted = a1_df[a1_df["OR"].notna()].sort_values("OR", ascending=True).reset_index(drop=True)

y_pos = np.arange(len(a1_sorted))
ci = a1_sorted["OR_CI95"].str.extract(r"\[(.*), (.*)\]")
ci = ci.apply(pd.to_numeric, errors="coerce")

colors = ['#e74c3c' if m == "BEST_ENSEMBLE" else ('#9b59b6' if m == "Lawma" else '#3498db') for m in a1_sorted["model"]]

for i, (yi, or_val, lo, hi, c, p) in enumerate(zip(y_pos, a1_sorted["OR"].values, ci[0].values, ci[1].values, colors, a1_sorted["p_cluster"].values)):
    ax.errorbar(x=or_val, y=yi, xerr=[[or_val - lo], [hi - or_val]], fmt="o", capsize=4, color=c, ecolor=c, markersize=8)
    sig = "***" if p < 0.001 else "**" if p < 0.01 else "*" if p < 0.05 else ""
    ax.text(hi + 0.1, yi, f"p={p:.3f}{sig}", va='center', fontsize=9)

ax.axvline(1.0, linestyle="--", linewidth=1, color='gray', alpha=0.7)
ax.set_yticks(y_pos)
ax.set_yticklabels(a1_sorted["model"].tolist())
ax.set_xlabel("Odds Ratio (95% CI)", fontsize=11)
ax.set_title("Forest Plot: Effect of Disagreement on False Positives\n(Cluster-robust logistic regression)", fontsize=12)
ax.set_xlim(0, ax.get_xlim()[1] + 1.5)

plt.tight_layout()
fig.savefig(os.path.join(A1_OUTPUT, "fig_A1_forest.png"), dpi=150, bbox_inches='tight')
fig.savefig(os.path.join(A1_OUTPUT, "fig_A1_forest.pdf"), bbox_inches='tight')
plt.show()
print(f"✓ fig_A1_forest.png / .pdf")

print("\n" + "=" * 80)
print("DONE")
print("=" * 80)

from statsmodels.stats.multitest import multipletests

# FDR on the 10 models (9 individual + ensemble)
reject, pvals_fdr, _, _ = multipletests(
    a1_df["p_cluster"], alpha=0.05, method="fdr_bh"
)
a1_df["p_fdr"] = pvals_fdr
a1_df["sig_fdr"] = reject

print("\n📊 FDR correction (Benjamini-Hochberg, 10 tests):\n")
print(f"{'Model':20s} {'p_raw':>10s} {'p_fdr':>10s} {'Sig':>5s}")
print("-" * 50)
for _, row in a1_df.iterrows():
    sig = "✓" if row["sig_fdr"] else ""
    print(f"{row['model']:20s} {row['p_cluster']:10.4f} {row['p_fdr']:10.4f} {sig:>5s}")

n_sig = a1_df["sig_fdr"].sum()
print(f"\n{n_sig} models significant after FDR correction (p < 0.05)")

# Analyse predictions


In [ ]:
# ============================================================
# PREDICTION ANALYSIS — Uses oof_proba_final
#
# NO GPU NEEDED — Loads pre-computed OOF probabilities
#
# Adapted to use files from oof_proba_final/
# ============================================================

import os
import numpy as np
import pandas as pd
import re
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from scipy.ndimage import gaussian_filter1d
from scipy.stats import pointbiserialr

# Plot configuration
plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams['figure.facecolor'] = 'white'
plt.rcParams['axes.facecolor'] = 'white'
plt.rcParams['font.size'] = 10

# ============================================================
# CONFIG — Paths to oof_proba_final
# ============================================================
BASE_PATH = "artifacts"  # not shipped — see DATA.md
EXCEL_PATH = "DATA/outputs/benchmark.csv"

# Probas in oof_proba_final
OUTPUT_PATH = os.path.join(BASE_PATH, "oof_proba_final")
BEST_OUTPUT_PATH = os.path.join(OUTPUT_PATH, "best_models")

# Output for analyses
ANALYSIS_OUTPUT = os.path.join(OUTPUT_PATH, "analysis_predictions")
os.makedirs(ANALYSIS_OUTPUT, exist_ok=True)

N_SPLITS = 5
SEED = 42

# ============================================================
# MODELS AND FILES
# ============================================================
MODELS = {
    "CamemBERT": "proba_oof_CamemBERT_BEST_cfg3_mean_avg_last_k_4_MLP2.npy",
    "CamemBERTav2": "proba_oof_CamemBERTav2_BEST_cfg1_mean_avg_last_k_4_MLP1.npy",
    "JuriBERT": "proba_oof_JuriBERT-base_BEST_cfg1_cls_last_LR.npy",
    "LLaMA": "proba_oof_LLaMA_BEST_cfg2_mean_layer_-2_MLP1.npy",
    "SAUL": "proba_oof_SAUL_BEST_cfg3_mean_last_k_4_LR.npy",
    "ST-MiniLM": "proba_oof_ST-MiniLM_BEST_cfg3_mean_layer_-2_MLP2.npy",
    "ST-MPNet": "proba_oof_ST-MPNet_BEST_cfg1_mean_avg_last_k_4_MLP2.npy",
}
TFIDF_CSV = "tfidf_oof_probas.csv"

# Best thresholds per model (exact values from latest search - 2000 points grid)
BEST_THRESHOLDS = {
    "CamemBERT": 0.482666,
    "CamemBERTav2": 0.587369,
    "JuriBERT": 0.739345,
    "LLaMA": 0.614332,
    "SAUL": 0.793622,
    "ST-MiniLM": 0.539045,
    "ST-MPNet": 0.631491,
    "TF-IDF": 0.471811,
}

# MCC per model (best threshold)
BEST_MCC = {
    "CamemBERT": 0.4281,
    "CamemBERTav2": 0.4160,
    "JuriBERT": 0.3706,
    "LLaMA": 0.4633,
    "SAUL": 0.4716,
    "ST-MiniLM": 0.3685,
    "ST-MPNet": 0.4292,
    "TF-IDF": 0.4148,
}

# MCC @ 0.5 per model
MCC_AT_05 = {
    "CamemBERT": 0.4209,
    "CamemBERTav2": 0.3799,
    "JuriBERT": 0.3603,
    "LLaMA": 0.4300,
    "SAUL": 0.4529,
    "ST-MiniLM": 0.3501,
    "ST-MPNet": 0.4013,
    "TF-IDF": 0.3964,
}

# BEST ENSEMBLE (Nested CV - Stacking_LR, MCC=0.5258)
# CamemBERTav2+JuriBERT+LLaMA+SAUL
BEST_ENSEMBLE_MODELS = ["CamemBERTav2", "JuriBERT", "LLaMA", "SAUL"]
BEST_ENSEMBLE_METHOD = "Stacking_LR"
BEST_ENSEMBLE_THRESHOLD = 0.610120
BEST_ENSEMBLE_MCC = 0.5258

# Threshold grid used
THRESHOLD_GRID = {
    "method": "np.linspace",
    "n_points": 2000,
}


# ============================================================
# HELPERS
# ============================================================
def plot_and_save(fig, path_png):
    fig.tight_layout()
    fig.savefig(path_png, dpi=150, bbox_inches='tight')
    plt.show()
    print(f"[Saved] {path_png}")


# ============================================================
# 1) LOAD DATA (labels, folds, case)
# ============================================================
print("=" * 80)
print("LOADING DATA")
print("=" * 80)

df0 = pd.read_excel(EXCEL_PATH)

cols_needed = ["decision_id", "chunk_id", "pred_art", "text", "article_text",
               "eval_A1", "eval_A2", "eval_A3"]
df0 = df0[cols_needed].copy()
print(f"Loaded: {len(df0)} rows")


def extract_oui_non(x):
    if pd.isna(x):
        return np.nan
    s = str(x).lower()
    has_oui = re.search(r"\boui\b", s) is not None
    has_non = re.search(r"\bnon\b", s) is not None
    if has_oui and not has_non:
        return "oui"
    if has_non and not has_oui:
        return "non"
    return np.nan


df0["a"] = df0["eval_A1"].apply(extract_oui_non)
df0["t"] = df0["eval_A2"].apply(extract_oui_non)
df0["s"] = df0["eval_A3"].apply(extract_oui_non)


def resolve_gold(row):
    a, t, s = row["a"], row["t"], row["s"]
    if pd.notna(a) and pd.notna(t) and a == t:
        return a, "agree"
    if pd.notna(a) and pd.notna(t) and a != t and pd.notna(s):
        return s, "disagree"
    return np.nan, "other"


res = df0.apply(resolve_gold, axis=1)
df0["label_str"] = res.apply(lambda x: x[0])
df0["case"] = res.apply(lambda x: x[1])

df0 = df0[df0["label_str"].isin(["oui", "non"])].copy()
df0["label_gold"] = df0["label_str"].map({"oui": 1, "non": 0}).astype(int)

for c in ["text", "article_text", "pred_art"]:
    df0[c] = df0[c].fillna("").astype(str)


def make_grouped_folds(df, n_splits, seed):
    rng = np.random.default_rng(seed)
    group_sizes = df.groupby("decision_id").size().to_dict()
    uniq_groups = np.array(list(group_sizes.keys()))
    uniq_groups = uniq_groups[rng.permutation(len(uniq_groups))]
    uniq_groups = sorted(uniq_groups, key=lambda g: group_sizes[g], reverse=True)

    fold_loads = np.zeros(n_splits, dtype=int)
    group_to_fold = {}
    for g in uniq_groups:
        f = int(fold_loads.argmin())
        group_to_fold[g] = f
        fold_loads[f] += int(group_sizes[g])

    out = df.copy()
    out["fold"] = out["decision_id"].map(group_to_fold).astype(int)
    return out, fold_loads


df_work, fold_loads = make_grouped_folds(df0, N_SPLITS, seed=SEED)
print(f"Usable: {len(df_work)} examples")
print(f"Fold loads: {fold_loads.tolist()}")

print("\nGold distribution by case:")
print(df_work.groupby("case")["label_gold"].value_counts(normalize=True).mul(100).round(1))


# ============================================================
# 2) LOAD OOF PROBABILITIES FROM oof_proba_final
# ============================================================
print("\n" + "=" * 80)
print("LOADING OOF PROBABILITIES (from oof_proba_final)")
print("=" * 80)

all_predictions = {}
y_true = df_work["label_gold"].values.astype(int)

# Load individual models
for model_name, fname in MODELS.items():
    path = os.path.join(OUTPUT_PATH, fname)

    if not os.path.exists(path):
        print(f"  [MISSING] {model_name}: {path}")
        continue

    p_oui = np.load(path)
    threshold = BEST_THRESHOLDS.get(model_name, 0.50)
    y_pred = (p_oui >= threshold).astype(int)

    # Build predictions DataFrame
    pred_df = pd.DataFrame({
        "orig_idx": np.arange(len(df_work)),
        "fold": df_work["fold"].values,
        "decision_id": df_work["decision_id"].astype(str).values,
        "case": df_work["case"].astype(str).values,
        "y_true": y_true,
        "y_pred": y_pred,
        "p_oui": p_oui,
        "p_non": 1.0 - p_oui,
    })
    pred_df["correct"] = (pred_df["y_true"] == pred_df["y_pred"]).astype(int)
    pred_df["is_FP"] = ((pred_df["y_true"] == 0) & (pred_df["y_pred"] == 1)).astype(int)
    pred_df["is_FN"] = ((pred_df["y_true"] == 1) & (pred_df["y_pred"] == 0)).astype(int)

    all_predictions[model_name] = pred_df
    print(f"  [OK] {model_name} (thr={threshold})")

# Load TF-IDF from CSV
tfidf_path = os.path.join(OUTPUT_PATH, TFIDF_CSV)
if os.path.exists(tfidf_path):
    tfidf_df = pd.read_csv(tfidf_path)
    p_oui = tfidf_df["proba_oui"].values
    threshold = BEST_THRESHOLDS["TF-IDF"]
    y_pred = (p_oui >= threshold).astype(int)

    pred_df = pd.DataFrame({
        "orig_idx": np.arange(len(df_work)),
        "fold": df_work["fold"].values,
        "decision_id": df_work["decision_id"].astype(str).values,
        "case": df_work["case"].astype(str).values,
        "y_true": y_true,
        "y_pred": y_pred,
        "p_oui": p_oui,
        "p_non": 1.0 - p_oui,
    })
    pred_df["correct"] = (pred_df["y_true"] == pred_df["y_pred"]).astype(int)
    pred_df["is_FP"] = ((pred_df["y_true"] == 0) & (pred_df["y_pred"] == 1)).astype(int)
    pred_df["is_FN"] = ((pred_df["y_true"] == 1) & (pred_df["y_pred"] == 0)).astype(int)

    all_predictions["TF-IDF"] = pred_df
    print(f"  [OK] TF-IDF (thr={threshold})")
else:
    print(f"  [MISSING] TF-IDF: {tfidf_path}")

# Load BEST ENSEMBLE from saved file (NESTED CV)
best_ens_path = os.path.join(BEST_OUTPUT_PATH, "proba_oof_BEST_ENSEMBLE_nested_cv.npy")
config_path = os.path.join(BEST_OUTPUT_PATH, "detailed_metrics_best_ensemble.json")

if os.path.exists(best_ens_path):
    import json
    p_mix = np.load(best_ens_path)

    # Load config if available
    if os.path.exists(config_path):
        with open(config_path, "r") as f:
            config = json.load(f)
        mix_thr = config.get("ensemble", {}).get("threshold", BEST_ENSEMBLE_THRESHOLD)
        models_str = config.get("ensemble", {}).get("models", BEST_ENSEMBLE_MODELS)
        method_str = config.get("ensemble", {}).get("method", BEST_ENSEMBLE_METHOD)
    else:
        mix_thr = BEST_ENSEMBLE_THRESHOLD
        models_str = BEST_ENSEMBLE_MODELS
        method_str = BEST_ENSEMBLE_METHOD

    y_pred_mix = (p_mix >= mix_thr).astype(int)

    pred_df = pd.DataFrame({
        "orig_idx": np.arange(len(df_work)),
        "fold": df_work["fold"].values,
        "decision_id": df_work["decision_id"].astype(str).values,
        "case": df_work["case"].astype(str).values,
        "y_true": y_true,
        "y_pred": y_pred_mix,
        "p_oui": p_mix,
        "p_non": 1.0 - p_mix,
    })
    pred_df["correct"] = (pred_df["y_true"] == pred_df["y_pred"]).astype(int)
    pred_df["is_FP"] = ((pred_df["y_true"] == 0) & (pred_df["y_pred"] == 1)).astype(int)
    pred_df["is_FN"] = ((pred_df["y_true"] == 1) & (pred_df["y_pred"] == 0)).astype(int)

    all_predictions["BEST_ENSEMBLE"] = pred_df
    print(f"  [OK] BEST_ENSEMBLE ({method_str}, thr={mix_thr:.6f}, models={'+'.join(models_str)})")

else:
    print(f"  [MISSING] BEST_ENSEMBLE: {best_ens_path}")
    print(f"    Note: Run nested_cv_ensemble_search.py first to generate this file")

print(f"\n✓ {len(all_predictions)} models loaded")


# ============================================================
# 3) NORMALIZED DISTRIBUTION (% OF TOTAL PER GROUP)
# ============================================================
print("\n" + "=" * 80)
print("1) NORMALIZED DISTRIBUTION BY GROUP")
print("=" * 80)

def plot_normalized_distribution(pred_df, model_name, ax, n_bins=50):
    """
    Normalized histogram: each bar = % of group total
    Allows comparison of agree vs disagree despite different sizes
    """
    pred_non = pred_df[pred_df["y_true"] == 0]

    agree = pred_non[pred_non["case"] == "agree"]["p_non"].values
    disagree = pred_non[pred_non["case"] == "disagree"]["p_non"].values

    bins = np.linspace(0, 1, n_bins + 1)

    # Calculate histograms as percentages
    agree_hist, _ = np.histogram(agree, bins=bins)
    disagree_hist, _ = np.histogram(disagree, bins=bins)

    agree_pct = 100 * agree_hist / len(agree) if len(agree) > 0 else agree_hist
    disagree_pct = 100 * disagree_hist / len(disagree) if len(disagree) > 0 else disagree_hist

    # Gaussian smoothing
    agree_smooth = gaussian_filter1d(agree_pct, sigma=1.5)
    disagree_smooth = gaussian_filter1d(disagree_pct, sigma=1.5)

    bin_centers = (bins[:-1] + bins[1:]) / 2

    # Plot
    ax.fill_between(bin_centers, agree_smooth, alpha=0.4, color='#3498db', label=f'Agreement (n={len(agree)})')
    ax.fill_between(bin_centers, disagree_smooth, alpha=0.4, color='#e74c3c', label=f'Disagreement (n={len(disagree)})')
    ax.plot(bin_centers, agree_smooth, color='#2980b9', linewidth=2)
    ax.plot(bin_centers, disagree_smooth, color='#c0392b', linewidth=2)

    # Means
    if len(agree) > 0:
        ax.axvline(agree.mean(), color='#2980b9', linestyle='--', linewidth=2, alpha=0.8)
    if len(disagree) > 0:
        ax.axvline(disagree.mean(), color='#c0392b', linestyle='--', linewidth=2, alpha=0.8)

    ax.set_xlim(0, 1)
    ax.set_xlabel('p(non)')
    ax.set_ylabel('% of group')
    ax.set_title(f'{model_name}')
    ax.legend(fontsize=8, loc='upper left')

    return agree.mean() if len(agree) > 0 else np.nan, disagree.mean() if len(disagree) > 0 else np.nan


# Create figure for all models
n_models = len(all_predictions)
n_cols = 3
n_rows = (n_models + n_cols - 1) // n_cols

fig, axes = plt.subplots(n_rows, n_cols, figsize=(15, 4*n_rows))
axes = axes.flatten()

for i, (model_name, pred_df) in enumerate(all_predictions.items()):
    plot_normalized_distribution(pred_df, model_name, axes[i])

# Hide empty axes
for j in range(i+1, len(axes)):
    axes[j].set_visible(False)

plt.suptitle("Normalized p(non) Distribution — Class 'non' (y_true=0)\n"
             "Each curve = % of group | Dashed lines = means",
             fontsize=14, y=1.02)
plt.tight_layout()
plot_and_save(fig, os.path.join(ANALYSIS_OUTPUT, "distribution_normalized.png"))


# ============================================================
# 4) ERROR RATE (FP among true NON) BY GROUP
# ============================================================
print("\n" + "=" * 80)
print("2) ERROR RATE: 'NON' PREDICTED AS 'OUI' (FP)")
print("=" * 80)

print("""
For class 'non' (y_true=0):
  • FP (False Positive) = Model predicts 'oui' when it's 'non'
  • FP Rate = FP / total_true_non = % of 'non' misclassified
""")

error_rates = []

for model_name, pred_df in all_predictions.items():
    pred_non = pred_df[pred_df["y_true"] == 0]

    for case in ["agree", "disagree"]:
        subset = pred_non[pred_non["case"] == case]
        n_total = len(subset)
        n_fp = (subset["y_pred"] == 1).sum()  # Predicted as 'oui' (FP)
        n_tn = (subset["y_pred"] == 0).sum()  # Predicted as 'non' (TN, correct)

        rate = 100 * n_fp / n_total if n_total > 0 else 0

        error_rates.append({
            "model": model_name,
            "case": case,
            "n_total": n_total,
            "n_correct": n_tn,
            "n_error": n_fp,
            "error_rate": rate
        })

error_df = pd.DataFrame(error_rates)

# Formatted display
print("\n📊 FP Rate (true 'non' predicted as 'oui'):\n")
print(f"{'Model':20s} | {'Agreement':^30s} | {'Disagreement':^30s} | {'Δ':>6s}")
print(f"{'':20s} | {'n_err/n_tot':^15s} {'rate':^12s} | {'n_err/n_tot':^15s} {'rate':^12s} |")
print("-" * 95)

for model_name in all_predictions.keys():
    agree_row = error_df[(error_df["model"] == model_name) & (error_df["case"] == "agree")].iloc[0]
    disagree_row = error_df[(error_df["model"] == model_name) & (error_df["case"] == "disagree")].iloc[0]

    delta = disagree_row["error_rate"] - agree_row["error_rate"]

    print(f"{model_name:20s} | {agree_row['n_error']:3.0f}/{agree_row['n_total']:3.0f}        "
          f"{agree_row['error_rate']:5.1f}%      | {disagree_row['n_error']:3.0f}/{disagree_row['n_total']:3.0f}        "
          f"{disagree_row['error_rate']:5.1f}%      | {delta:+5.1f}%")

# Visualization
fig, ax = plt.subplots(figsize=(14, 6))

models = list(all_predictions.keys())
x = np.arange(len(models))
width = 0.35

agree_rates = [error_df[(error_df["model"] == m) & (error_df["case"] == "agree")]["error_rate"].values[0] for m in models]
disagree_rates = [error_df[(error_df["model"] == m) & (error_df["case"] == "disagree")]["error_rate"].values[0] for m in models]

bars1 = ax.bar(x - width/2, agree_rates, width, label='Agreement', color='#3498db', alpha=0.7)
bars2 = ax.bar(x + width/2, disagree_rates, width, label='Disagreement', color='#e74c3c', alpha=0.7)

# Add values on bars
for bar, rate in zip(bars1, agree_rates):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.5, f'{rate:.1f}%',
            ha='center', va='bottom', fontsize=9)
for bar, rate in zip(bars2, disagree_rates):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.5, f'{rate:.1f}%',
            ha='center', va='bottom', fontsize=9)

ax.set_ylabel("FP Rate (%)")
ax.set_xlabel("Model")
ax.set_title("False Positive Rate: 'non' Predicted as 'oui'\n(Higher rate = more model errors)")
ax.set_xticks(x)
ax.set_xticklabels(models, rotation=45, ha='right')
ax.legend()
ax.set_ylim(0, max(max(agree_rates), max(disagree_rates)) * 1.3)

plt.tight_layout()
plot_and_save(fig, os.path.join(ANALYSIS_OUTPUT, "error_rate_fp.png"))


# ============================================================
# 5) DISTRIBUTION OVERLAP (Bhattacharyya Coefficient)
# ============================================================
print("\n" + "=" * 80)
print("3) DISTRIBUTION OVERLAP MEASUREMENT")
print("=" * 80)

print("""
The Bhattacharyya coefficient measures similarity between two distributions.
Lower overlap means agree/disagree distributions are more separated.
""")

def bhattacharyya_coefficient(p, q, bins=50):
    """Calculate Bhattacharyya coefficient between two samples"""
    range_min = min(p.min(), q.min())
    range_max = max(p.max(), q.max())

    p_hist, _ = np.histogram(p, bins=bins, range=(range_min, range_max), density=True)
    q_hist, _ = np.histogram(q, bins=bins, range=(range_min, range_max), density=True)

    # Normalize to probabilities
    p_hist = p_hist / (p_hist.sum() + 1e-10)
    q_hist = q_hist / (q_hist.sum() + 1e-10)

    # Bhattacharyya coefficient
    bc = np.sum(np.sqrt(p_hist * q_hist))

    # Bhattacharyya distance
    bd = -np.log(bc + 1e-10)

    return bc, bd


overlap_results = []

for model_name, pred_df in all_predictions.items():
    pred_non = pred_df[pred_df["y_true"] == 0]

    agree = pred_non[pred_non["case"] == "agree"]["p_non"].values
    disagree = pred_non[pred_non["case"] == "disagree"]["p_non"].values

    if len(agree) > 10 and len(disagree) > 10:
        bc, bd = bhattacharyya_coefficient(agree, disagree)

        # Kolmogorov-Smirnov test
        ks_stat, ks_pval = stats.ks_2samp(agree, disagree)

        overlap_results.append({
            "model": model_name,
            "bhattacharyya_coef": bc,
            "bhattacharyya_dist": bd,
            "ks_statistic": ks_stat,
            "ks_pvalue": ks_pval,
            "separation": 1 - bc,
        })

overlap_df = pd.DataFrame(overlap_results)

print("\n📊 Distribution separation measures:\n")
print(f"{'Model':20s} | {'Bhatt. Coef':>12s} | {'Separation':>10s} | {'KS stat':>10s} | {'KS p-val':>10s}")
print("-" * 75)
for _, row in overlap_df.iterrows():
    print(f"{row['model']:20s} | {row['bhattacharyya_coef']:12.4f} | {row['separation']:10.4f} | "
          f"{row['ks_statistic']:10.4f} | {row['ks_pvalue']:.2e}")

# Visualization
fig, ax = plt.subplots(figsize=(12, 6))

x = np.arange(len(overlap_df))
colors = plt.cm.RdYlGn(overlap_df["separation"].values / overlap_df["separation"].max())

bars = ax.bar(x, overlap_df["separation"], color=colors, alpha=0.8)
ax.set_ylabel("Separation (1 - Bhattacharyya)")
ax.set_xlabel("Model")
ax.set_title("Distribution Separation: Agreement vs Disagreement\n(Higher = more different distributions)")
ax.set_xticks(x)
ax.set_xticklabels(overlap_df["model"], rotation=45, ha='right')

plt.tight_layout()
plot_and_save(fig, os.path.join(ANALYSIS_OUTPUT, "distribution_separation.png"))


# ============================================================
# 6) CONFIDENCE-DISAGREEMENT CORRELATION
# ============================================================
print("\n" + "=" * 80)
print("4) CONFIDENCE-DISAGREEMENT CORRELATION")
print("=" * 80)

print("""
Question: Are low-confidence cases more often 'disagree'?
Confidence = distance to threshold (|p_oui - threshold|)
""")

correlation_results = []

for model_name, pred_df in all_predictions.items():
    pred_non = pred_df[pred_df["y_true"] == 0].copy()

    # Get model threshold
    if model_name == "BEST_ENSEMBLE":
        threshold = BEST_ENSEMBLE_THRESHOLD
    else:
        threshold = BEST_THRESHOLDS.get(model_name, 0.50)

    # Confidence = distance to threshold
    pred_non["confidence"] = np.abs(pred_non["p_oui"] - threshold)
    pred_non["is_disagree"] = (pred_non["case"] == "disagree").astype(int)

    # Point-biserial correlation
    pb_corr, pb_pval = pointbiserialr(pred_non["is_disagree"], pred_non["confidence"])

    correlation_results.append({
        "model": model_name,
        "pointbiserial_r": pb_corr,
        "pvalue": pb_pval,
    })

corr_df = pd.DataFrame(correlation_results)

print("\n📊 Correlation between confidence and case type:\n")
print("(Negative correlation = less confidence for 'disagree')\n")
print(f"{'Model':20s} | {'Point-biserial r':>15s} | {'p-value':>12s} | {'Significant':>12s}")
print("-" * 70)

for _, row in corr_df.iterrows():
    sig = "✓" if row["pvalue"] < 0.05 else ""
    print(f"{row['model']:20s} | {row['pointbiserial_r']:+15.4f} | {row['pvalue']:.2e} | {sig:>12s}")

# ============================================================
# 7) CALIBRATION
# ============================================================
print("\n" + "=" * 80)
print("5) CALIBRATION ANALYSIS")
print("=" * 80)

print("""
Question: When the model says "80% chance it's 'oui'",
is it actually correct 80% of the time?
""")

def calibration_analysis(pred_df, n_bins=10, min_samples=10):
    """Analyze model calibration

    Args:
        pred_df: DataFrame with p_oui and y_true columns
        n_bins: Number of probability bins
        min_samples: Minimum samples to include a bin (default: 10)
    """
    # Create probability bins
    bins = np.linspace(0, 1, n_bins + 1)
    pred_df = pred_df.copy()
    pred_df["p_bin"] = pd.cut(pred_df["p_oui"], bins=bins, labels=False, include_lowest=True)

    calibration = []
    for b in range(n_bins):
        bin_data = pred_df[pred_df["p_bin"] == b]
        n_samples = len(bin_data)

        # Skip bins with too few samples
        if n_samples >= min_samples:
            mean_pred = bin_data["p_oui"].mean()
            actual_positive = (bin_data["y_true"] == 1).mean()

            calibration.append({
                "bin": b,
                "bin_range": f"{bins[b]:.1f}-{bins[b+1]:.1f}",
                "mean_predicted_p": mean_pred,
                "actual_positive_rate": actual_positive,
                "n_samples": n_samples,
            })

    return pd.DataFrame(calibration)


# Plot calibration for ALL models
selected_models = list(all_predictions.keys())

n_models = len(selected_models)
n_cols = 3
n_rows = (n_models + n_cols - 1) // n_cols

fig, axes = plt.subplots(n_rows, n_cols, figsize=(5*n_cols, 4*n_rows))
axes = axes.flatten()

for i, model_name in enumerate(selected_models):
    ax = axes[i]
    pred_df = all_predictions[model_name]
    cal_df = calibration_analysis(pred_df)

    # Perfect calibration curve
    ax.plot([0, 1], [0, 1], 'k--', label='Perfect calibration', alpha=0.5)

    # Calibration points
    ax.scatter(cal_df["mean_predicted_p"], cal_df["actual_positive_rate"],
               s=cal_df["n_samples"]*2, alpha=0.6, c='#3498db', edgecolors='#2980b9', linewidths=1)
    ax.plot(cal_df["mean_predicted_p"], cal_df["actual_positive_rate"],
            '#3498db', alpha=0.5, linewidth=1.5)

    # Add sample counts next to each point
    for _, row in cal_df.iterrows():
        ax.annotate(f'{int(row["n_samples"])}',
                    (row["mean_predicted_p"], row["actual_positive_rate"]),
                    textcoords="offset points", xytext=(5, 5), fontsize=7,
                    color='#2980b9', alpha=0.9)

    ax.set_xlabel("Predicted probability p(oui)")
    ax.set_ylabel("Actual 'oui' rate")
    ax.set_title(f'{model_name}')
    ax.set_xlim(0, 1)
    ax.set_ylim(0, 1)
    ax.legend(fontsize=8, loc='lower right')
    ax.grid(True, alpha=0.3)

# Hide empty axes
for j in range(i+1, len(axes)):
    axes[j].set_visible(False)

plt.suptitle("Calibration Diagram\n(Point size = sample count | Bins with n < 10 excluded)",
             fontsize=12, y=1.02)
plt.tight_layout()
plot_and_save(fig, os.path.join(ANALYSIS_OUTPUT, "calibration_plot.png"))


# ============================================================
# CALIBRATION SUMMARY TABLE - ALL MODELS
# ============================================================
print("\n" + "=" * 80)
print("📊 CALIBRATION TABLES - ALL MODELS")
print("(Note: Bins with n < 10 samples excluded)")
print("=" * 80)

# Calculate ECE for all models
ece_summary = []

for model_name in selected_models:
    pred_df = all_predictions[model_name]
    cal_df = calibration_analysis(pred_df)

    print(f"\n{'='*65}")
    print(f"Model: {model_name}")
    print(f"{'='*65}")
    print(f"{'Bin':^12s} | {'Mean p_oui':^12s} | {'Actual rate':^12s} | {'n_samples':^10s} | {'Gap':^10s}")
    print("-" * 65)

    for _, row in cal_df.iterrows():
        gap = row["actual_positive_rate"] - row["mean_predicted_p"]
        print(f"{row['bin_range']:^12s} | {row['mean_predicted_p']:^12.3f} | "
              f"{row['actual_positive_rate']:^12.3f} | {row['n_samples']:^10.0f} | {gap:^+10.3f}")

    # Calculate Expected Calibration Error (ECE)
    total_samples = cal_df["n_samples"].sum()
    ece = (cal_df["n_samples"] * np.abs(cal_df["actual_positive_rate"] - cal_df["mean_predicted_p"])).sum() / total_samples
    n_bins_used = len(cal_df)
    print(f"\nECE (Expected Calibration Error): {ece:.4f} (computed on {n_bins_used} bins, {int(total_samples)} samples)")

    ece_summary.append({
        "model": model_name,
        "ece": ece,
        "n_samples": total_samples,
    })

# ============================================================
# ECE SUMMARY TABLE - ALL MODELS
# ============================================================
print("\n" + "=" * 80)
print("📊 ECE SUMMARY - RANKED BY CALIBRATION")
print("=" * 80)
print("(ECE = 0 = perfectly calibrated, lower is better)\n")

ece_df = pd.DataFrame(ece_summary).sort_values("ece")
print(f"{'Rank':<6} {'Model':<25} {'ECE':>10} {'n_samples':>12}")
print("-" * 55)

for rank, (_, row) in enumerate(ece_df.iterrows(), 1):
    print(f"{rank:<6} {row['model']:<25} {row['ece']:>10.4f} {row['n_samples']:>12.0f}")

# Save ECE table
ece_df.to_csv(os.path.join(ANALYSIS_OUTPUT, "calibration_ece_all_models.csv"), index=False)
print(f"\n✓ ECE table saved: {ANALYSIS_OUTPUT}/calibration_ece_all_models.csv")


# ============================================================
# 8) ANALYSIS BY PROBABILITY QUANTILES
# ============================================================
print("\n" + "=" * 80)
print("6) QUANTILE ANALYSIS")
print("=" * 80)

print("""
Question: Is the agree-disagree gap uniform across the distribution?
Or is it concentrated in certain probability zones?
""")

def analyze_by_quantile(pred_df, n_quantiles=5):
    """Analyze disagree rate by p_oui quantile"""
    pred_non = pred_df[pred_df["y_true"] == 0].copy()

    # Create quantiles
    pred_non["quantile"] = pd.qcut(pred_non["p_oui"], n_quantiles, labels=False, duplicates='drop')

    results = []
    for q in sorted(pred_non["quantile"].dropna().unique()):
        q_data = pred_non[pred_non["quantile"] == q]

        n_agree = len(q_data[q_data["case"] == "agree"])
        n_disagree = len(q_data[q_data["case"] == "disagree"])
        n_total = n_agree + n_disagree

        pct_disagree = 100 * n_disagree / n_total if n_total > 0 else 0

        # FP rate in this quantile
        fp_rate = 100 * q_data["is_FP"].mean()

        results.append({
            "quantile": int(q),
            "p_oui_range": f"{q_data['p_oui'].min():.2f}-{q_data['p_oui'].max():.2f}",
            "n_total": n_total,
            "n_agree": n_agree,
            "n_disagree": n_disagree,
            "pct_disagree": pct_disagree,
            "fp_rate": fp_rate,
        })

    return pd.DataFrame(results)


# Analyze for representative models
selected_models = ["TF-IDF", "SAUL"]
if "BEST_ENSEMBLE" in all_predictions:
    selected_models.append("BEST_ENSEMBLE")

selected_models = [m for m in selected_models if m in all_predictions]

fig, axes = plt.subplots(2, len(selected_models), figsize=(5*len(selected_models), 8))

for col, model_name in enumerate(selected_models):
    pred_df = all_predictions[model_name]
    quantile_df = analyze_by_quantile(pred_df)

    x = quantile_df["quantile"].values

    # Subplot 1: % disagree by quantile
    ax = axes[0, col]
    ax.bar(x, quantile_df["pct_disagree"], color='#e74c3c', alpha=0.7)
    ax.axhline(y=quantile_df["pct_disagree"].mean(), color='black', linestyle='--', alpha=0.5)
    ax.set_xlabel("p(oui) Quantile")
    ax.set_ylabel("% Disagreement")
    ax.set_title(f'{model_name}\n% Disagreement by Quantile')
    ax.set_xticks(x)
    ax.set_xticklabels([f'Q{i+1}\n{r["p_oui_range"]}' for i, r in quantile_df.iterrows()], fontsize=8)

    # Subplot 2: FP rate by quantile
    ax = axes[1, col]
    ax.bar(x, quantile_df["fp_rate"], color='#9b59b6', alpha=0.7)
    ax.set_xlabel("p(oui) Quantile")
    ax.set_ylabel("FP Rate (%)")
    ax.set_title(f'{model_name}\nFP Rate by Quantile')
    ax.set_xticks(x)
    ax.set_xticklabels([f'Q{i+1}' for i in range(len(quantile_df))], fontsize=9)

plt.tight_layout()
plot_and_save(fig, os.path.join(ANALYSIS_OUTPUT, "analysis_by_quantile.png"))


# ============================================================
# 9) SUMMARY AND EXPORT
# ============================================================
print("\n" + "=" * 80)
print("SUMMARY AND EXPORT")
print("=" * 80)

# Final stats
print("\n📊 Key Statistics:\n")

# Average error gap
error_pivot = error_df.pivot(index="model", columns="case", values="error_rate")
error_pivot["delta"] = error_pivot["disagree"] - error_pivot["agree"]
print(f"   Δ FP rate mean (disagree - agree): {error_pivot['delta'].mean():+.1f}%")

# Average separation
print(f"   Average distribution separation:   {overlap_df['separation'].mean():.4f}")

# Average correlation
print(f"   Confidence-disagreement corr:      {corr_df['pointbiserial_r'].mean():.4f}")

# Create Excel with multiple sheets
output_excel = os.path.join(ANALYSIS_OUTPUT, "prediction_analysis_complete.xlsx")

with pd.ExcelWriter(output_excel, engine='openpyxl') as writer:
    error_df.to_excel(writer, sheet_name='FP_error_rate', index=False)
    overlap_df.to_excel(writer, sheet_name='Distribution_separation', index=False)
    corr_df.to_excel(writer, sheet_name='Correlation', index=False)

print(f"\n📁 Results exported: {output_excel}")

print("\n📈 Figures generated:")
for f in os.listdir(ANALYSIS_OUTPUT):
    if f.endswith('.png'):
        print(f"   • {f}")

print("\n" + "=" * 80)
print("ANALYSIS COMPLETE")
print("=" * 80)

# calibration ag/disag

In [ ]:
# ============================================================
# 7) CALIBRATION — WITH AGREE/DISAGREE DISTINCTION
# ============================================================
print("\n" + "=" * 80)
print("5) CALIBRATION ANALYSIS")
print("=" * 80)

print("""
Question: When the model says "80% chance it's 'oui'",
is it actually correct 80% of the time?

We compare calibration on 'agree' vs 'disagree' cases.
""")

# ============================================================
# BEST ENSEMBLE CONFIG (Nested CV - Stacking_LR, MCC=0.5258)
# CamemBERTav2+JuriBERT+LLaMA+SAUL
# ============================================================
BEST_ENSEMBLE_MODELS = ["CamemBERTav2", "JuriBERT", "LLaMA", "SAUL"]
BEST_ENSEMBLE_METHOD = "Stacking_LR"
BEST_ENSEMBLE_THRESHOLD = 0.610120
BEST_ENSEMBLE_MCC = 0.5258

# Best thresholds per model (exact values from latest search - 2000 points grid)
BEST_THRESHOLDS = {
    "CamemBERT": 0.482666,
    "CamemBERTav2": 0.587369,
    "JuriBERT": 0.739345,
    "LLaMA": 0.614332,
    "SAUL": 0.793622,
    "ST-MiniLM": 0.539045,
    "ST-MPNet": 0.631491,
    "TF-IDF": 0.471811,
}

# Minimum samples to include a bin in calibration analysis
MIN_SAMPLES_PER_BIN = 10


def calibration_analysis(pred_df, n_bins=10, min_samples=MIN_SAMPLES_PER_BIN):
    """Analyze model calibration

    Args:
        pred_df: DataFrame with p_oui and y_true columns
        n_bins: Number of probability bins
        min_samples: Minimum samples to include a bin (default: 10)
    """
    bins = np.linspace(0, 1, n_bins + 1)
    pred_df = pred_df.copy()
    pred_df["p_bin"] = pd.cut(pred_df["p_oui"], bins=bins, labels=False, include_lowest=True)

    calibration = []
    for b in range(n_bins):
        bin_data = pred_df[pred_df["p_bin"] == b]
        n_samples = len(bin_data)

        # Skip bins with too few samples
        if n_samples >= min_samples:
            mean_pred = bin_data["p_oui"].mean()
            actual_positive = (bin_data["y_true"] == 1).mean()

            calibration.append({
                "bin": b,
                "bin_range": f"{bins[b]:.1f}-{bins[b+1]:.1f}",
                "mean_predicted_p": mean_pred,
                "actual_positive_rate": actual_positive,
                "n_samples": n_samples,
            })

    return pd.DataFrame(calibration)


def calibration_analysis_by_case(pred_df, n_bins=10, min_samples=MIN_SAMPLES_PER_BIN):
    """Analyze calibration separately for agree and disagree cases

    Args:
        pred_df: DataFrame with p_oui, y_true, and case columns
        n_bins: Number of probability bins
        min_samples: Minimum samples to include a bin (default: 10)
    """
    bins = np.linspace(0, 1, n_bins + 1)
    pred_df = pred_df.copy()
    pred_df["p_bin"] = pd.cut(pred_df["p_oui"], bins=bins, labels=False, include_lowest=True)

    results = {"agree": [], "disagree": []}

    for case in ["agree", "disagree"]:
        case_data = pred_df[pred_df["case"] == case]

        for b in range(n_bins):
            bin_data = case_data[case_data["p_bin"] == b]
            n_samples = len(bin_data)

            # Skip bins with too few samples
            if n_samples >= min_samples:
                mean_pred = bin_data["p_oui"].mean()
                actual_positive = (bin_data["y_true"] == 1).mean()

                results[case].append({
                    "bin": b,
                    "mean_predicted_p": mean_pred,
                    "actual_positive_rate": actual_positive,
                    "n_samples": n_samples,
                })

    return {k: pd.DataFrame(v) for k, v in results.items()}


# DEBUG: Display available models
print(f"\nAvailable models: {list(all_predictions.keys())}")
print(f"Total count: {len(all_predictions)}\n")

# Plot calibration for ALL models with agree/disagree distinction
selected_models = list(all_predictions.keys())

n_models = len(selected_models)
n_cols = 3
n_rows = (n_models + n_cols - 1) // n_cols

print(f"Creating a {n_rows}x{n_cols} grid for {n_models} models")

fig, axes = plt.subplots(n_rows, n_cols, figsize=(5*n_cols, 4*n_rows))

# Ensure axes is always a 1D array
if n_models == 1:
    axes = [axes]
elif n_rows == 1:
    axes = list(axes) if hasattr(axes, '__iter__') else [axes]
else:
    axes = axes.flatten()

for i, model_name in enumerate(selected_models):
    ax = axes[i]
    pred_df = all_predictions[model_name]

    # Calibration by case
    cal_by_case = calibration_analysis_by_case(pred_df)

    # Perfect calibration line
    ax.plot([0, 1], [0, 1], 'k--', label='Perfect', alpha=0.5, linewidth=1.5)

    # Global calibration (for reference, in gray)
    cal_global = calibration_analysis(pred_df)
    if len(cal_global) > 0:
        ax.plot(cal_global["mean_predicted_p"], cal_global["actual_positive_rate"],
                'gray', alpha=0.3, linewidth=2, label='Global')

    # AGREE calibration
    if len(cal_by_case["agree"]) > 0:
        cal_agree = cal_by_case["agree"]
        ax.scatter(cal_agree["mean_predicted_p"], cal_agree["actual_positive_rate"],
                   s=cal_agree["n_samples"]*1.5, alpha=0.7, c='#3498db',
                   edgecolors='#2980b9', linewidths=1, label=f'Agreement (n={cal_agree["n_samples"].sum()})')
        ax.plot(cal_agree["mean_predicted_p"], cal_agree["actual_positive_rate"],
                '#3498db', alpha=0.5, linewidth=1.5)
        # Add sample count next to each point
        for _, row in cal_agree.iterrows():
            ax.annotate(f'{int(row["n_samples"])}',
                        (row["mean_predicted_p"], row["actual_positive_rate"]),
                        textcoords="offset points", xytext=(5, 5), fontsize=6,
                        color='#2980b9', alpha=0.8)

    # DISAGREE calibration
    if len(cal_by_case["disagree"]) > 0:
        cal_disagree = cal_by_case["disagree"]
        ax.scatter(cal_disagree["mean_predicted_p"], cal_disagree["actual_positive_rate"],
                   s=cal_disagree["n_samples"]*1.5, alpha=0.7, c='#e74c3c',
                   edgecolors='#c0392b', linewidths=1, marker='s', label=f'Disagreement (n={cal_disagree["n_samples"].sum()})')
        ax.plot(cal_disagree["mean_predicted_p"], cal_disagree["actual_positive_rate"],
                '#e74c3c', alpha=0.5, linewidth=1.5)
        # Add sample count next to each point
        for _, row in cal_disagree.iterrows():
            ax.annotate(f'{int(row["n_samples"])}',
                        (row["mean_predicted_p"], row["actual_positive_rate"]),
                        textcoords="offset points", xytext=(5, -8), fontsize=6,
                        color='#c0392b', alpha=0.8)

    ax.set_xlabel("Predicted probability p(oui)")
    ax.set_ylabel("Actual 'oui' rate")
    ax.set_title(f'{model_name}')
    ax.set_xlim(0, 1)
    ax.set_ylim(0, 1)
    ax.legend(fontsize=7, loc='lower right')
    ax.grid(True, alpha=0.3)

# Hide empty axes
for j in range(len(selected_models), len(axes)):
    axes[j].set_visible(False)

plt.suptitle("Calibration Diagram — Agreement (●) vs Disagreement (■)\n"
             "(Point size = sample count | Bins with n < 10 excluded)",
             fontsize=12, y=1.02)
plt.tight_layout()
plot_and_save(fig, os.path.join(ANALYSIS_OUTPUT, "calibration_plot_by_case.png"))


# ============================================================
# CALIBRATION SUMMARY TABLE - ALL MODELS
# ============================================================
print("\n" + "=" * 80)
print("📊 CALIBRATION TABLES - ALL MODELS")
print("(Note: Bins with n < 10 samples excluded)")
print("=" * 80)

# Calculate ECE for all models
ece_summary = []

for model_name in selected_models:
    pred_df = all_predictions[model_name]
    cal_df = calibration_analysis(pred_df)

    print(f"\n{'='*65}")
    print(f"Model: {model_name}")
    print(f"{'='*65}")
    print(f"{'Bin':^12s} | {'Mean p_oui':^12s} | {'Actual rate':^12s} | {'n_samples':^10s} | {'Gap':^10s}")
    print("-" * 65)

    for _, row in cal_df.iterrows():
        gap = row["actual_positive_rate"] - row["mean_predicted_p"]
        print(f"{row['bin_range']:^12s} | {row['mean_predicted_p']:^12.3f} | "
              f"{row['actual_positive_rate']:^12.3f} | {row['n_samples']:^10.0f} | {gap:^+10.3f}")

    # Calculate Expected Calibration Error (ECE)
    total_samples = cal_df["n_samples"].sum()
    ece = (cal_df["n_samples"] * np.abs(cal_df["actual_positive_rate"] - cal_df["mean_predicted_p"])).sum() / total_samples
    n_bins_used = len(cal_df)
    print(f"\nECE (Expected Calibration Error): {ece:.4f} (computed on {n_bins_used} bins, {int(total_samples)} samples)")

    ece_summary.append({
        "model": model_name,
        "ece": ece,
        "n_samples": total_samples,
    })

# ============================================================
# ECE SUMMARY TABLE - ALL MODELS
# ============================================================
print("\n" + "=" * 80)
print("📊 ECE SUMMARY - RANKED BY CALIBRATION")
print("=" * 80)
print("(ECE = 0 = perfectly calibrated, lower is better)\n")

ece_df = pd.DataFrame(ece_summary).sort_values("ece")
print(f"{'Rank':<6} {'Model':<25} {'ECE':>10} {'n_samples':>12}")
print("-" * 55)

for rank, (_, row) in enumerate(ece_df.iterrows(), 1):
    print(f"{rank:<6} {row['model']:<25} {row['ece']:>10.4f} {row['n_samples']:>12.0f}")

# Save ECE table
ece_df.to_csv(os.path.join(ANALYSIS_OUTPUT, "calibration_ece_all_models.csv"), index=False)
print(f"\n✓ ECE table saved: {ANALYSIS_OUTPUT}/calibration_ece_all_models.csv")

# ============================================================
# LATEX TABLE - ECE ALL MODELS
# ============================================================
print("\n" + "=" * 80)
print("LATEX TABLE - ECE")
print("=" * 80)
print("""
\\begin{table}[h]
\\centering
\\begin{tabular}{lcc}
\\toprule
\\textbf{Model} & \\textbf{ECE} & \\textbf{Rank} \\\\
\\midrule""")

for rank, (_, row) in enumerate(ece_df.iterrows(), 1):
    print(f"{row['model']} & {row['ece']:.4f} & {rank} \\\\")

print("""\\bottomrule
\\end{tabular}
\\caption{Expected Calibration Error (ECE) per model — lower = better calibrated}
\\label{tab:calibration_ece}
\\end{table}
""")

# Calib with LAWMA

In [ ]:
# ============================================================
# CALIBRATION VALUES BY CASE (AGREE/DISAGREE) FOR LATEX
# WITH LAWMA
# ============================================================

import os, re, warnings
import numpy as np
import pandas as pd

warnings.filterwarnings("ignore")

# ============================================================
# CONFIG (from A1 script)
# ============================================================
BASE_PATH = "artifacts"  # not shipped — see DATA.md
EXCEL_PATH = "DATA/outputs/benchmark.csv"
OUTPUT_PATH = os.path.join(BASE_PATH, "oof_proba_final")
BEST_OUTPUT_PATH = os.path.join(OUTPUT_PATH, "best_models")
ANALYSIS_OUTPUT = os.path.join(OUTPUT_PATH, "A1_analysis_nested_cv_with_lawma")
os.makedirs(ANALYSIS_OUTPUT, exist_ok=True)

SEED = 42
N_SPLITS = 5
MIN_SAMPLES_PER_BIN = 10

# ============================================================
# MODELS AND FILES
# ============================================================
MODELS = {
    "CamemBERT": "proba_oof_CamemBERT_BEST_cfg3_mean_avg_last_k_4_MLP2.npy",
    "CamemBERTav2": "proba_oof_CamemBERTav2_BEST_cfg1_mean_avg_last_k_4_MLP1.npy",
    "JuriBERT": "proba_oof_JuriBERT-base_BEST_cfg1_cls_last_LR.npy",
    "LLaMA": "proba_oof_LLaMA_BEST_cfg2_mean_layer_-2_MLP1.npy",
    "SAUL": "proba_oof_SAUL_BEST_cfg3_mean_last_k_4_LR.npy",
    "ST-MiniLM": "proba_oof_ST-MiniLM_BEST_cfg3_mean_layer_-2_MLP2.npy",
    "ST-MPNet": "proba_oof_ST-MPNet_BEST_cfg1_mean_avg_last_k_4_MLP2.npy",
}
TFIDF_CSV = "tfidf_oof_probas.csv"

# Lawma: best config = MLP1, cfg1, mean, last (MCC=0.4576)
LAWMA_PROBA_PATH = os.path.join(
    BASE_PATH, "outputs", "outputs_lawma_mlp1_mlp2_3cfg",
    "proba_oof_Lawma-8B__cfg1__mean__last__mlp1.npy"
)

# Best ensemble (Stacking-LR, nested CV)
BEST_ENSEMBLE_PATH = os.path.join(BEST_OUTPUT_PATH, "proba_oof_BEST_ENSEMBLE_nested_cv.npy")
BEST_ENSEMBLE_MODELS = ["CamemBERTav2", "JuriBERT", "LLaMA", "SAUL"]
BEST_ENSEMBLE_METHOD = "Stacking_LR"
BEST_ENSEMBLE_THRESHOLD = 0.610120

# ============================================================
# LOAD DATA (identical to A1)
# ============================================================
print("=" * 80)
print("LOADING DATA")
print("=" * 80)

df0 = pd.read_excel(EXCEL_PATH)
cols = ["decision_id", "chunk_id", "eval_A1", "eval_A2", "eval_A3"]
df0 = df0[cols].copy()

def extract(x):
    if pd.isna(x): return np.nan
    s = str(x).lower()
    if re.search(r"\boui\b", s) and not re.search(r"\bnon\b", s): return "oui"
    if re.search(r"\bnon\b", s) and not re.search(r"\boui\b", s): return "non"
    return np.nan

df0["a"] = df0["eval_A1"].apply(extract)
df0["t"] = df0["eval_A2"].apply(extract)
df0["s"] = df0["eval_A3"].apply(extract)

def resolve(row):
    a, t, s = row["a"], row["t"], row["s"]
    if pd.notna(a) and pd.notna(t) and a == t:
        return a, "agree"
    if pd.notna(a) and pd.notna(t) and a != t and pd.notna(s):
        return s, "disagree"
    return np.nan, "other"

res = df0.apply(resolve, axis=1)
df0["label_str"] = res.apply(lambda x: x[0])
df0["case"] = res.apply(lambda x: x[1])

df0 = df0[df0["label_str"].isin(["oui", "non"])].copy()
df0["label"] = df0["label_str"].map({"oui": 1, "non": 0}).astype(int)
df0 = df0.reset_index(drop=True)

print(f"Total: {len(df0)} samples")
print(f"  - agree: {(df0['case'] == 'agree').sum()}")
print(f"  - disagree: {(df0['case'] == 'disagree').sum()}")

# ============================================================
# LOAD PROBAS
# ============================================================
print("\nLoading probabilities...")

probas = {}

# Individual models
for name, fname in MODELS.items():
    path = os.path.join(OUTPUT_PATH, fname)
    if os.path.exists(path):
        probas[name] = np.load(path)
        print(f"  ✓ {name}")

# TF-IDF
tfidf_path = os.path.join(OUTPUT_PATH, TFIDF_CSV)
if os.path.exists(tfidf_path):
    tfidf_df = pd.read_csv(tfidf_path)
    probas["TF-IDF"] = tfidf_df["proba_oui"].values
    print(f"  ✓ TF-IDF")

# Lawma
lawma_loaded = False
if os.path.exists(LAWMA_PROBA_PATH):
    probas["Lawma"] = np.load(LAWMA_PROBA_PATH)
    print(f"  ✓ Lawma (from {LAWMA_PROBA_PATH})")
    lawma_loaded = True
else:
    # Fallback: search for it
    for root, dirs, files in os.walk(os.path.join(BASE_PATH, "outputs")):
        for f in files:
            if "Lawma" in f and "cfg1" in f and "last" in f and "mlp1" in f and f.endswith(".npy"):
                probas["Lawma"] = np.load(os.path.join(root, f))
                print(f"  ✓ Lawma (found via search: {os.path.join(root, f)})")
                lawma_loaded = True
                break
        if lawma_loaded:
            break

if not lawma_loaded:
    print("  ✗ Lawma NOT FOUND — will be excluded from analysis")

# Best ensemble
if os.path.exists(BEST_ENSEMBLE_PATH):
    probas["BEST_ENSEMBLE"] = np.load(BEST_ENSEMBLE_PATH)
    print(f"  ✓ BEST_ENSEMBLE (Stacking-LR)")
else:
    print(f"  ✗ BEST_ENSEMBLE not found")

print(f"\n{len(probas)} models loaded")

# ============================================================
# SAVE LAWMA PROBAS IN SAME FORMAT AS OTHER MODELS
# ============================================================
if "Lawma" in probas:
    lawma_save_path = os.path.join(OUTPUT_PATH, "proba_oof_Lawma_BEST_cfg1_mean_last_MLP1.npy")
    np.save(lawma_save_path, probas["Lawma"])
    print(f"\n✓ Saved Lawma copy: {lawma_save_path}")

# ============================================================
# BUILD PREDICTION DATAFRAMES
# ============================================================
all_predictions = {}

for model_name, p_oui in probas.items():
    pred_df = pd.DataFrame({
        "decision_id": df0["decision_id"].values,
        "case": df0["case"].values,
        "y_true": df0["label"].values,
        "p_oui": p_oui,
    })
    all_predictions[model_name] = pred_df

# ============================================================
# CALIBRATION FUNCTION
# ============================================================
def get_calibration_by_case_detailed(pred_df, n_bins=10, min_samples=MIN_SAMPLES_PER_BIN):
    """Get detailed calibration data for agree and disagree cases"""
    bins = np.linspace(0, 1, n_bins + 1)
    pred_df = pred_df.copy()
    pred_df["p_bin"] = pd.cut(pred_df["p_oui"], bins=bins, labels=False, include_lowest=True)

    results = []

    for case in ["agree", "disagree"]:
        case_data = pred_df[pred_df["case"] == case]

        for b in range(n_bins):
            bin_data = case_data[case_data["p_bin"] == b]
            n_samples = len(bin_data)

            bin_range = f"{bins[b]:.1f}-{bins[b+1]:.1f}"

            if n_samples >= min_samples:
                mean_pred = bin_data["p_oui"].mean()
                actual_rate = (bin_data["y_true"] == 1).mean()
                gap = actual_rate - mean_pred

                results.append({
                    "case": case,
                    "bin": b,
                    "bin_range": bin_range,
                    "mean_p": mean_pred,
                    "actual_rate": actual_rate,
                    "gap": gap,
                    "n_samples": n_samples,
                })
            else:
                results.append({
                    "case": case,
                    "bin": b,
                    "bin_range": bin_range,
                    "mean_p": np.nan,
                    "actual_rate": np.nan,
                    "gap": np.nan,
                    "n_samples": n_samples,
                })

    return pd.DataFrame(results)


# ============================================================
# GENERATE DATA FOR ALL MODELS
# ============================================================
print("\n" + "=" * 80)
print("CALIBRATION BY CASE - DATA FOR LATEX TABLES")
print("=" * 80)

all_calibration_data = []

for model_name in all_predictions.keys():
    pred_df = all_predictions[model_name]
    cal_data = get_calibration_by_case_detailed(pred_df)
    cal_data["model"] = model_name
    all_calibration_data.append(cal_data)

cal_df_all = pd.concat(all_calibration_data, ignore_index=True)

# ============================================================
# DISPLAY PER MODEL
# ============================================================
for model_name in all_predictions.keys():
    model_data = cal_df_all[cal_df_all["model"] == model_name]

    print(f"\n{'='*80}")
    print(f"MODEL: {model_name}")
    print(f"{'='*80}")

    agree_data = model_data[model_data["case"] == "agree"].set_index("bin_range")
    disagree_data = model_data[model_data["case"] == "disagree"].set_index("bin_range")

    print(f"\n{'Bin':<12} | {'AGREEMENT':^35} | {'DISAGREEMENT':^35}")
    print(f"{'':12} | {'p_pred':>8} {'actual':>8} {'gap':>8} {'n':>6} | {'p_pred':>8} {'actual':>8} {'gap':>8} {'n':>6}")
    print("-" * 90)

    for bin_range in agree_data.index:
        a = agree_data.loc[bin_range]
        d = disagree_data.loc[bin_range] if bin_range in disagree_data.index else None

        if pd.notna(a["mean_p"]):
            a_str = f"{a['mean_p']:8.3f} {a['actual_rate']:8.3f} {a['gap']:+8.3f} {int(a['n_samples']):6d}"
        else:
            a_str = f"{'--':>8} {'--':>8} {'--':>8} {int(a['n_samples']):6d}"

        if d is not None and pd.notna(d["mean_p"]):
            d_str = f"{d['mean_p']:8.3f} {d['actual_rate']:8.3f} {d['gap']:+8.3f} {int(d['n_samples']):6d}"
        elif d is not None:
            d_str = f"{'--':>8} {'--':>8} {'--':>8} {int(d['n_samples']):6d}"
        else:
            d_str = f"{'--':>8} {'--':>8} {'--':>8} {'--':>6}"

        print(f"{bin_range:<12} | {a_str} | {d_str}")

    agree_valid = agree_data[agree_data["mean_p"].notna()]
    disagree_valid = disagree_data[disagree_data["mean_p"].notna()]

    if len(agree_valid) > 0:
        ece_agree = (agree_valid["n_samples"] * np.abs(agree_valid["gap"])).sum() / agree_valid["n_samples"].sum()
    else:
        ece_agree = np.nan

    if len(disagree_valid) > 0:
        ece_disagree = (disagree_valid["n_samples"] * np.abs(disagree_valid["gap"])).sum() / disagree_valid["n_samples"].sum()
    else:
        ece_disagree = np.nan

    print(f"\nECE Agreement:    {ece_agree:.4f}" if pd.notna(ece_agree) else "\nECE Agreement:    N/A")
    print(f"ECE Disagreement: {ece_disagree:.4f}" if pd.notna(ece_disagree) else "ECE Disagreement: N/A")


# ============================================================
# LATEX TABLE - SELECTED MODELS (SIDE BY SIDE)
# ============================================================
print("\n" + "=" * 80)
print("LATEX TABLE - CALIBRATION BY CASE")
print("=" * 80)

latex_models = ["SAUL", "TF-IDF", "BEST_ENSEMBLE", "Lawma"]
latex_models = [m for m in latex_models if m in all_predictions]

for model_name in latex_models:
    model_data = cal_df_all[cal_df_all["model"] == model_name]
    agree_data = model_data[model_data["case"] == "agree"].set_index("bin_range")
    disagree_data = model_data[model_data["case"] == "disagree"].set_index("bin_range")

    print(f"""
% ============================================================
% {model_name}
% ============================================================
\\begin{{table}}[h]
\\centering
\\small
\\begin{{tabular}}{{l|cccc|cccc}}
\\toprule
 & \\multicolumn{{4}}{{c|}}{{\\textbf{{Agreement}}}} & \\multicolumn{{4}}{{c}}{{\\textbf{{Disagreement}}}} \\\\
\\textbf{{Bin}} & $\\bar{{p}}$ & Actual & Gap & n & $\\bar{{p}}$ & Actual & Gap & n \\\\
\\midrule""")

    for bin_range in agree_data.index:
        a = agree_data.loc[bin_range]
        d = disagree_data.loc[bin_range] if bin_range in disagree_data.index else None

        if pd.notna(a["mean_p"]):
            a_vals = f"{a['mean_p']:.2f} & {a['actual_rate']:.2f} & {a['gap']:+.2f} & {int(a['n_samples'])}"
        else:
            a_vals = f"-- & -- & -- & {int(a['n_samples'])}"

        if d is not None and pd.notna(d["mean_p"]):
            d_vals = f"{d['mean_p']:.2f} & {d['actual_rate']:.2f} & {d['gap']:+.2f} & {int(d['n_samples'])}"
        elif d is not None:
            d_vals = f"-- & -- & -- & {int(d['n_samples'])}"
        else:
            d_vals = "-- & -- & -- & --"

        print(f"{bin_range} & {a_vals} & {d_vals} \\\\")

    agree_valid = agree_data[agree_data["mean_p"].notna()]
    disagree_valid = disagree_data[disagree_data["mean_p"].notna()]
    ece_agree = (agree_valid["n_samples"] * np.abs(agree_valid["gap"])).sum() / agree_valid["n_samples"].sum() if len(agree_valid) > 0 else np.nan
    ece_disagree = (disagree_valid["n_samples"] * np.abs(disagree_valid["gap"])).sum() / disagree_valid["n_samples"].sum() if len(disagree_valid) > 0 else np.nan

    print(f"""\\midrule
\\textbf{{ECE}} & \\multicolumn{{4}}{{c|}}{{{ece_agree:.4f}}} & \\multicolumn{{4}}{{c}}{{{ece_disagree:.4f}}} \\\\
\\bottomrule
\\end{{tabular}}
\\caption{{Calibration analysis for {model_name}: Agreement vs Disagreement cases. Bins with $n < 10$ excluded.}}
\\label{{tab:calibration_{model_name.lower().replace('_', '').replace('-', '')}}}
\\end{{table}}
""")


# ============================================================
# SUMMARY TABLE - ECE BY CASE FOR ALL MODELS
# ============================================================
print("\n" + "=" * 80)
print("LATEX TABLE - ECE SUMMARY BY CASE")
print("=" * 80)

ece_by_case = []
for model_name in all_predictions.keys():
    model_data = cal_df_all[cal_df_all["model"] == model_name]
    agree_data = model_data[(model_data["case"] == "agree") & (model_data["mean_p"].notna())]
    disagree_data = model_data[(model_data["case"] == "disagree") & (model_data["mean_p"].notna())]

    ece_agree = (agree_data["n_samples"] * np.abs(agree_data["gap"])).sum() / agree_data["n_samples"].sum() if len(agree_data) > 0 else np.nan
    ece_disagree = (disagree_data["n_samples"] * np.abs(disagree_data["gap"])).sum() / disagree_data["n_samples"].sum() if len(disagree_data) > 0 else np.nan

    ece_by_case.append({
        "model": model_name,
        "ece_agree": ece_agree,
        "ece_disagree": ece_disagree,
        "ece_diff": ece_disagree - ece_agree if pd.notna(ece_agree) and pd.notna(ece_disagree) else np.nan,
        "n_agree": int(agree_data["n_samples"].sum()) if len(agree_data) > 0 else 0,
        "n_disagree": int(disagree_data["n_samples"].sum()) if len(disagree_data) > 0 else 0,
    })

ece_case_df = pd.DataFrame(ece_by_case).sort_values("ece_diff", ascending=False)

print("""
\\begin{table}[h]
\\centering
\\begin{tabular}{lccccc}
\\toprule
\\textbf{Model} & \\textbf{ECE (Agree)} & \\textbf{ECE (Disagree)} & \\textbf{$\\Delta$ ECE} & $n_{agree}$ & $n_{disagree}$ \\\\
\\midrule""")

for _, row in ece_case_df.iterrows():
    ece_a = f"{row['ece_agree']:.4f}" if pd.notna(row['ece_agree']) else "--"
    ece_d = f"{row['ece_disagree']:.4f}" if pd.notna(row['ece_disagree']) else "--"
    ece_diff = f"{row['ece_diff']:+.4f}" if pd.notna(row['ece_diff']) else "--"
    print(f"{row['model']} & {ece_a} & {ece_d} & {ece_diff} & {row['n_agree']} & {row['n_disagree']} \\\\")

print("""\\bottomrule
\\end{tabular}
\\caption{Expected Calibration Error (ECE) by annotator agreement status. $\\Delta$ ECE = ECE(Disagree) - ECE(Agree). Positive values indicate worse calibration on disagreement cases.}
\\label{tab:ece_by_case}
\\end{table}
""")

# ============================================================
# SAVE CSV FOR FURTHER ANALYSIS
# ============================================================
cal_df_all.to_csv(os.path.join(ANALYSIS_OUTPUT, "calibration_by_case_all_models.csv"), index=False)
ece_case_df.to_csv(os.path.join(ANALYSIS_OUTPUT, "ece_by_case_summary.csv"), index=False)

print(f"\n✓ Saved: calibration_by_case_all_models.csv")
print(f"✓ Saved: ece_by_case_summary.csv")